# 聚类模型——逐行解读代码流程

## data/data_utils.py 数据处理相关的函数

In [ ]:
# data/data_utils.py 
# 用来存放数据处理相关的函数（跟图片读取、图片整理、标签整理有关的工具）

import numpy as np          # 用于数值计算和数组操作
import cv2                  # OpenCV库，用于图像处理
import torch                # PyTorch深度学习框架
from torch.utils.data import Dataset  # PyTorch数据集基类
from PIL import Image       # Python图像处理库，用于图像读取、保存和转换
import glob                 # 用于文件路径模式匹配
import os                   # 操作系统接口，用于文件路径操作

# ============================================================
# 细胞对齐函数
# ============================================================
def align_cell_by_major_axis(image):
    """
    根据细胞的主轴对齐细胞图像，使细胞的长轴水平放置
    
    工作原理：
    1. 将图像转为二值图像，分离细胞和背景
    2. 找到细胞的轮廓
    3. 计算最小外接矩形，获取中心点、宽高和角度
    4. 旋转图像使细胞长轴水平
    
    参数:
    image: 输入图像 (OpenCV格式，BGR或灰度)
    
    返回:
    aligned_image: 对齐后的图像
    """
    # 转换为灰度图 - 如果是彩色图像则转换，否则直接使用
    if len(image.shape) == 3:  # 判断是否为彩色图像(3通道)
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    else:
        gray = image.copy()
    
    # 阈值分割，将细胞与背景分离
    # 二值化：将图像变成"只有黑和白"的图，方便把细胞从背景里抠出来
    # 背景通常很白(接近255)，细胞通常较暗，所以用240作为阈值分离
    # THRESH_BINARY_INV表示反转二值化结果，即背景变黑(0)，细胞变白(255)
    _, binary = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
    
    # 寻找图像中的轮廓
    # RETR_EXTERNAL只检测最外层轮廓，CHAIN_APPROX_SIMPLE压缩水平、垂直和对角线方向的轮廓点
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 如果没有找到轮廓，返回原图
    if not contours:
        return image
    
    # 找到最大的轮廓（假设是细胞）
    # 使用contourArea计算每个轮廓的面积，选择面积最大的轮廓
    largest_contour = max(contours, key=cv2.contourArea)
    
    # 计算最小包围矩形
    # minAreaRect返回矩形的中心点、宽高和旋转角度
    rect = cv2.minAreaRect(largest_contour)
    center, (width, height), angle = rect
    
    # 确保宽度大于高度，使主轴始终水平
    # 如果高度大于宽度，旋转90度使长边水平
    if height > width:
        angle += 90
    
    # 获取旋转矩阵
    # getRotationMatrix2D创建一个2x3的旋转矩阵，用于后续的warpAffine变换
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    
    # 计算旋转后的图像大小，确保不会裁剪
    # 使用三角函数计算旋转后图像的新尺寸
    h, w = image.shape[:2]
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])
    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))
    
    # 调整旋转矩阵以考虑平移
    # 确保旋转后的图像居中显示
    M[0, 2] += (new_w / 2) - center[0]
    M[1, 2] += (new_h / 2) - center[1]
    
    # 应用旋转变换
    # warpAffine执行仿射变换，将图像旋转到指定角度
    # borderMode和borderValue指定边界填充方式，这里用白色填充
    aligned_image = cv2.warpAffine(image, M, (new_w, new_h), 
                                  flags=cv2.INTER_LINEAR,  # 线性插值
                                  borderMode=cv2.BORDER_CONSTANT,  # 常数边界
                                  borderValue=(255, 255, 255))  # 白色填充背景
    
    return aligned_image

# ============================================================
# 细胞图像数据集类
# ============================================================
class CellImageDataset(Dataset):
    """
    细胞图像数据集类，继承自PyTorch的Dataset
    
    功能：
    1. 加载细胞图像
    2. 可选地对图像进行主轴对齐
    3. 应用图像变换（如标准化、调整大小等）
    4. 处理加载错误
    5. 缓存已对齐的图像以提高效率
    """
    def __init__(self, image_paths, transform=None, source_labels=None, align_cells=False):
        """
        初始化数据集
        
        参数:
        image_paths: 图像文件路径列表
        transform: PyTorch变换函数，用于预处理图像
        source_labels: 每个图像对应的标签列表，若不提供则默认全为0
        align_cells: 是否对细胞图像进行主轴对齐
        """
        self.image_paths = image_paths  # 存储所有图像路径
        self.transform = transform  # 图像变换函数
        # 如果没有提供标签，则默认所有图像标签为0
        self.source_labels = source_labels if source_labels is not None else [0] * len(image_paths)
        self.align_cells = align_cells  # 是否进行细胞对齐
        
        # 缓存已对齐的图像，避免重复计算提高效率
        self.aligned_images_cache = {}
    
    def __len__(self):
        """返回数据集中的样本数量"""
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        """
        获取指定索引的样本
        
        参数:
        idx: 样本索引
        
        返回:
        img: 处理后的图像
        source_label: 对应的标签
        idx: 样本索引（用于后续追踪）
        """
        img_path = self.image_paths[idx]  # 获取图像路径
        source_label = self.source_labels[idx]  # 获取对应标签
        
        try: 
            # 处理图像对齐
            if self.align_cells:  # 如果需要对齐
                if img_path not in self.aligned_images_cache:  # 检查缓存中是否已有对齐后的图像
                    # 使用OpenCV读取图像
                    img = cv2.imread(img_path)
                    if img is None:
                        raise ValueError(f"无法读取图像: {img_path}")
                    
                    # 转换为RGB（因为OpenCV默认是BGR）
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    
                    # 进行主轴对齐
                    aligned_img = align_cell_by_major_axis(img)
                    
                    # 将NumPy数组转换为PIL图像
                    img = Image.fromarray(aligned_img)
                    
                    # 缓存对齐后的图像，避免重复计算
                    self.aligned_images_cache[img_path] = img
                else:
                    # 直接从缓存中获取已对齐的图像
                    img = self.aligned_images_cache[img_path]
            else:
                # 不需要对齐，直接读取图像
                # PIL的Image.open()函数按照图像原始格式打开，convert('RGB')确保转换为RGB模式
                img = Image.open(img_path).convert('RGB')
            
            # 应用图像变换（如标准化、调整大小等）
            if self.transform:
                img = self.transform(img)  # 将PIL图像转换为PyTorch模型可处理的格式
            
            return img, source_label, idx
            
        except Exception as e:
            # 错误处理：如果图像加载失败，返回空白图像
            print(f"加载图像 {img_path} 时出错: {e}")
            
            # 根据是否需要变换，返回不同格式的空白图像，导入的是自定义的变换函数
            if self.transform:
                # 返回PyTorch张量格式的空白图像
                dummy_img = torch.zeros((3, 224, 224))
            else:
                # 返回NumPy数组格式的空白图像
                dummy_img = np.zeros((224, 224, 3), dtype=np.uint8)
                
            return dummy_img, source_label, idx

# ============================================================
# 图像路径收集函数
# ============================================================
def collect_image_paths(directory, extensions=['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff']):
    """
    收集指定目录下所有支持格式的图片路径
    
    参数:
    directory: 图像所在目录
    extensions: 支持的图像文件扩展名列表
    
    返回:
    paths: 所有图像的完整路径列表
    """
    paths = []
    if os.path.exists(directory):  # 确保目录存在
        for ext in extensions:  # 遍历所有支持的扩展名
            # 使用glob.glob查找匹配指定模式的所有文件路径
            paths.extend(glob.glob(os.path.join(directory, ext)))
    return paths

# ============================================================
# 参考数据收集函数
# ============================================================
def collect_reference_data(malignant_dir, benign_dir):
    """
    收集参考域数据，包括恶性和良性细胞图像
    
    功能：
    1. 收集恶性细胞图像，支持两种组织方式：
       - 按亚型分类（LUAD, LUSC, SCLC）
       - 所有恶性图像放在一起
    2. 收集良性细胞图像
    
    参数:
    malignant_dir: 恶性细胞图像目录
    benign_dir: 良性细胞图像目录
    
    返回:
    image_paths: 所有图像路径列表
    labels: 对应的标签列表
    """
    image_paths = []  # 存储所有图像路径
    labels = []  # 存储对应标签
    
    # 定义恶性亚型的映射关系
    categories = {
        "LUAD": 1,  # 肺腺癌
        "LUSC": 2,  # 肺鳞癌
        "SCLC": 3   # 小细胞肺癌
    }
    
    # 检查恶性目录下是否有子文件夹    
    subdirs = [d for d in os.listdir(malignant_dir) 
               if os.path.isdir(os.path.join(malignant_dir, d))]
    
    # 处理恶性图像
    if subdirs and any(subdir in categories for subdir in subdirs):
        # 模式1：如果有子文件夹且子文件夹名在categories中，按亚型分类处理
        for cat_name, label in categories.items():
            subdir_path = os.path.join(malignant_dir, cat_name)
            if os.path.exists(subdir_path):
                # 收集该亚型的所有图像路径
                paths = collect_image_paths(subdir_path)
                image_paths.extend(paths)
                # 为该亚型的所有图像分配相同的标签
                labels.extend([label] * len(paths))
    else:
        # 模式2：如果没有子文件夹或子文件夹名不在categories中，将整个恶性目录作为一个整体
        malignant_paths = collect_image_paths(malignant_dir)
        image_paths.extend(malignant_paths)
        # 所有恶性图统一打标签5，5代表malignant
        labels.extend([5] * len(malignant_paths))
        
    # 收集良性图像
    benign_paths = collect_image_paths(benign_dir)
    image_paths.extend(benign_paths)
    # 所有良性图统一打标签4
    labels.extend([4] * len(benign_paths))
    
    return image_paths, labels

    # 标签说明：
    # 模式1：细分恶性亚型
    # LUAD (肺腺癌) = 1
    # LUSC (肺鳞癌) = 2
    # SCLC (小细胞肺癌) = 3
    # 模式2：不分亚型，整个恶性统一
    # malignant (恶性) = 5
    # 两种模式共有：
    # benign (良性) = 4

# ============================================================
# 辅助函数
# ============================================================
def extract_patient_id(patient_folder):
    """
    从路径中提取病理ID
    
    参数:
    patient_folder: 病人文件夹路径
    
    返回:
    patient_id: 提取的病理ID（文件夹名称）
    """
    # os.path.basename返回路径的最后一部分（文件夹名或文件名）
    return os.path.basename(patient_folder)

## models/models.py 模型定义的类和获取模型及转换函数的工厂函数

### 19个模型的特征提取器

In [ ]:
# =============================================================================
# models/models.py
# 职责：定义所有 CNN / Transformer 骨干网络的特征提取器类，
#       并提供统一的工厂函数 get_model_and_transform() 供外部调用。
#
# ★ 核心设计思路（必读）★
#
#   本包的目标是"特征提取"而非"分类"。
#   但模型是在分类任务上训练的（如良性/恶性二分类），
#   训练好的权重文件里包含了分类头（最后一个 Linear 层）的参数。
#
#   因此，每个特征提取器类都遵循以下统一的四步模式：
#
#     Step 1: 初始化标准 torchvision 模型（pretrained=False，不下载 ImageNet 权重）
#     Step 2: 修改分类头结构，使其与训练时完全一致（如 num_classes=2）
#             → 目的：让 load_state_dict 能正确匹配权重的形状，不报 size mismatch
#     Step 3: 调用 load_weights() 从本地 .pth 文件加载训练好的权重
#             → 必须在 Step 2 之后、Step 4 之前执行！
#     Step 4: 将分类头替换为 nn.Identity()（直通层，输入原样输出）
#             → 这样 forward() 就输出倒数第二层的特征向量，而不是分类 logits
#
# ★ 为什么用 nn.Identity() 而不是直接截断网络？★
#
#   方法A（截断）: nn.Sequential(*list(model.children())[:-1])
#     → 危险！会丢失 forward() 中的函数式调用，如：
#       DenseNet 的 F.relu / F.adaptive_avg_pool2d
#       ShuffleNet 的 x.mean([2, 3])
#       这些操作不在 children() 里，截断后就消失了，特征会出错。
#
#   方法B（Identity）: model.fc = nn.Identity()
#     → 安全！复用官方的 forward() 逻辑，只是让最后一步变成"什么都不做"，
#       从而安全地输出倒数第二层特征。
#
#   本文件混合使用两种方法：
#     - ResNet/ResNeXt：结构简单，forward 无隐式调用，用截断法
#     - DenseNet/ShuffleNet/GoogLeNet 等：有隐式调用，用 Identity 法
# =============================================================================

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import cv2
from torch.utils.data import Dataset
from PIL import Image
import glob
import os


# =============================================================================
# ① AlexNet 特征提取器
# =============================================================================
# AlexNet 网络结构：
#   features（5个卷积层组）→ avgpool → flatten → classifier（全连接层组）
#
# classifier 完整结构（索引 0~6）：
#   [0] Dropout
#   [1] Linear(9216 → 4096)
#   [2] ReLU
#   [3] Dropout
#   [4] Linear(4096 → 4096)
#   [5] ReLU
#   [6] Linear(4096 → num_classes)  ← 分类头，不需要
#
# 策略：取 classifier[:6]（索引 0~5），输出 4096 维特征向量。
# 使用截断法（因为 AlexNet forward 无隐式函数调用）。
# =============================================================================
class CustomAlexNetFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        # 调用父类 nn.Module 的初始化（PyTorch 所有自定义模型的规范写法）
        super(CustomAlexNetFeatureExtractor, self).__init__()

        # Step 1: 初始化标准 AlexNet，不加载 ImageNet 预训练权重
        self.alexnet = models.alexnet(pretrained=False)

        # Step 2: 修改分类头输出维度，与训练时保持一致
        # classifier[6] 是最后的 Linear 层
        # in_features 获取该层的输入维度（= 4096），只改输出维度
        in_features = self.alexnet.classifier[6].in_features  # = 4096
        self.alexnet.classifier[6] = nn.Linear(in_features, num_classes)
        # 现在 classifier[6] = Linear(4096 → 2)，与训练时一致，权重可正确加载

        # Step 3: 定义特征提取部分（取 classifier 的前 6 层，索引 0~5）
        # 包含：Dropout → Linear(4096) → ReLU → Dropout → Linear(4096) → ReLU
        # 输出：4096 维 embedding（倒数第二层特征）
        # 注意：这是引用而非复制，后续 alexnet.classifier 修改不影响此引用
        self.feature_extractor_part = self.alexnet.classifier[:6]

        # Step 4: 加载训练好的权重（必须在 Step 2 修改结构后执行）
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        """
        前向传播
        Args:
            x: 输入图像张量 [B, 3, 224, 224]
            layer: 'penultimate' → 返回 4096 维特征向量（默认）
                   其他值       → 返回完整分类 logits [B, num_classes]
        """
        # 通过 5 个卷积层组提取空间特征
        # [B, 3, 224, 224] → [B, 256, 6, 6]
        x = self.alexnet.features(x)

        # 自适应平均池化（AlexNet 默认输出已是 6×6，此步不改变尺寸）
        x = self.alexnet.avgpool(x)

        # 展平：从第 1 维开始，将所有维度合并
        # [B, 256, 6, 6] → [B, 9216]
        x = torch.flatten(x, 1)

        if layer == 'penultimate':
            # 通过前 6 层全连接层，得到 4096 维特征
            # [B, 9216] → [B, 4096]
            x = self.feature_extractor_part(x)
            return x
        else:
            # 通过完整 classifier（含第 7 层分类头）
            # [B, 9216] → [B, num_classes]
            x = self.alexnet.classifier(x)
            return x

    def load_weights(self, model_path):
        """
        加载模型权重，自动处理键名前缀不匹配问题。

        背景：训练时若模型被包裹在另一个类中：
            class MyModel:
                self.alexnet = AlexNet()
        保存的 state_dict 键名会带 'alexnet.' 前缀：
            'alexnet.features.0.weight'
        但 torchvision AlexNet 期望的键名是：
            'features.0.weight'
        需要去掉前缀才能正确加载。
        """
        try:
            # 从磁盘加载权重字典（OrderedDict: 键=层名, 值=权重张量）
            state_dict = torch.load(model_path)

            # 检查所有键是否都以 'alexnet.' 开头
            if all(k.startswith('alexnet.') for k in state_dict.keys()):
                # 字典推导式：去掉每个键的前 8 个字符（'alexnet.' 的长度）
                new_state_dict = {k[8:]: v for k, v in state_dict.items()}
                # strict=False：允许部分键不匹配（如修改过的分类头）
                self.alexnet.load_state_dict(new_state_dict, strict=False)
                print("AlexNet模型权重加载成功（已处理'alexnet.'前缀）！")
            else:
                # 键名无前缀，直接加载
                self.alexnet.load_state_dict(state_dict, strict=False)
                print("AlexNet模型权重加载成功！")
        except Exception as e:
            print(f"加载AlexNet模型权重时出错: {e}")
            raise RuntimeError(f"无法加载AlexNet模型权重: {e}")


# =============================================================================
# ② ResNeXt-50 特征提取器
# =============================================================================
# ResNeXt 网络结构：
#   conv1 → bn1 → relu → maxpool →
#   layer1 → layer2 → layer3 → layer4 →
#   avgpool（全局平均池化）→ flatten → fc（全连接分类头）
#
# 策略：用 children() 截断，去掉最后的 fc 层。
# 输出 avgpool 后的 [B, 2048, 1, 1]，再手动 flatten 得到 [B, 2048]。
#
# 为什么这里用截断法而非 Identity？
# ResNeXt 的 forward() 逻辑非常直接，没有隐式函数调用，截断安全。
# 但注意：必须先 load_weights，再做截断（因为截断后 fc 不在 penultimate_layer 里）。
# =============================================================================
class CustomResNeXtFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomResNeXtFeatureExtractor, self).__init__()

        # Step 1: 初始化 ResNeXt-50（32x4d：32组卷积，每组宽度4）
        self.resnext = models.resnext50_32x4d(pretrained=False)

        # Step 2: 修改 fc 层输出维度以匹配训练结构
        in_features = self.resnext.fc.in_features  # = 2048
        self.resnext.fc = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在截断网络之前！）
        # 原因：截断后 fc 层不在 penultimate_layer 里，
        # 但权重文件里有 fc 层的参数，需要先加载到 self.resnext
        self.load_weights(model_path)

        # Step 4: 构建特征提取子网络（去掉最后的 fc 层）
        # list(self.resnext.children())：返回所有直接子模块列表
        # [:-1]：切片去掉最后一个（fc 层）
        # nn.Sequential(*...)：将剩余模块重新组合为顺序网络
        self.penultimate_layer = nn.Sequential(*list(self.resnext.children())[:-1])
        # 包含：conv1→bn1→relu→maxpool→layer1→layer2→layer3→layer4→avgpool
        # 输出形状：[B, 2048, 1, 1]

    def forward(self, x, layer='penultimate'):
        """
        Args:
            x: [B, 3, 224, 224]
        Returns:
            [B, 2048]
        """
        if layer == 'penultimate':
            x = self.penultimate_layer(x)  # [B, 2048, 1, 1]
            # ★ 用 flatten 而非 squeeze！
            # squeeze 在 batch_size=1 时会把 batch 维也压掉：
            #   [1, 2048, 1, 1] → squeeze → [2048]（错误！）
            #   [1, 2048, 1, 1] → flatten(1) → [1, 2048]（正确）
            return torch.flatten(x, 1)  # [B, 2048]
        else:
            raise ValueError(f"不支持的层: {layer}")

    def load_weights(self, model_path):
        try:
            # map_location='cpu'：无论训练时用什么设备，先加载到 CPU
            # 避免在无 GPU 机器上报 CUDA 相关错误
            state_dict = torch.load(model_path, map_location='cpu')
            new_state_dict = {}
            for k, v in state_dict.items():
                if k.startswith('resnext.'):
                    new_state_dict[k[8:]] = v  # 去掉 'resnext.' 前缀（长度8）
                else:
                    new_state_dict[k] = v
            self.resnext.load_state_dict(new_state_dict, strict=False)
            print("ResNeXt模型权重加载成功！")
        except Exception as e:
            print(f"加载ResNeXt模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ResNeXt模型权重: {e}")


# =============================================================================
# ③ ConvNeXt-Large 特征提取器
# =============================================================================
# ConvNeXt 网络结构：
#   features（分层卷积块）→ avgpool → classifier
#
# classifier 结构（索引 0~2）：
#   [0] LayerNorm2d（层归一化）
#   [1] Flatten（展平）
#   [2] Linear(1536 → num_classes)  ← 分类头
#
# 策略：将 classifier[2] 替换为 nn.Identity()
# 这样 forward() 输出经过 LayerNorm + Flatten 后的 1536 维特征。
# =============================================================================
class CustomConvNextLargeFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomConvNextLargeFeatureExtractor, self).__init__()

        # Step 1
        self.convnext = models.convnext_large(pretrained=False)

        # Step 2: 修改 classifier[2] 的输出维度
        in_features = self.convnext.classifier[2].in_features  # = 1536
        self.convnext.classifier[2] = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（在替换 Identity 之前！）
        self.load_weights(model_path)

        # Step 4: 保存分类头，将其替换为 Identity
        self.classifier_head = self.convnext.classifier[2]   # Linear(1536 → num_classes)
        self.convnext.classifier[2] = nn.Identity()
        # 现在 convnext(x) 输出流程：
        # features → avgpool → LayerNorm2d → Flatten → Identity
        # 输出：[B, 1536]

    def forward(self, x, layer='penultimate'):
        # 直接调用整个 convnext 的 forward，最后一层已是 Identity
        features = self.convnext(x)
        if layer == 'penultimate':
            return features               # [B, 1536]
        else:
            return self.classifier_head(features)  # [B, num_classes]

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('convnext.') for k in state_dict.keys()):
                new_state_dict = {k[9:]: v for k, v in state_dict.items()}  # 去掉前缀（长度9）
                self.convnext.load_state_dict(new_state_dict, strict=False)
            else:
                self.convnext.load_state_dict(state_dict, strict=False)
            print("ConvNext模型权重加载成功！")
        except Exception as e:
            print(f"加载ConvNext模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ConvNext模型权重: {e}")


# =============================================================================
# ④ GoogLeNet (Inception v1) 特征提取器
# =============================================================================
# GoogLeNet 网络结构（简化）：
#   Inception 模块组 → avgpool → Dropout → flatten → fc（1024 → num_classes）
#
# 策略：将 fc 替换为 Identity，输出 1024 维特征。
#
# ★ 两个重要注意点 ★
# 1. aux_logits=False：禁用辅助分类器
#    GoogLeNet 训练时有两个辅助分类头用于梯度传播，推理时不需要。
#    若不禁用，训练模式下 forward 返回 (main_output, aux1, aux2) 三元组，
#    会导致后续代码出错。
#
# 2. 必须调用 model.eval() 再提取特征！
#    GoogLeNet 的 forward 包含 Dropout，训练模式下随机丢弃特征，
#    导致每次提取的特征不一致，无法用于聚类。
#    eval() 模式会禁用 Dropout，保证特征的确定性。
# =============================================================================
class CustomGoogLeNetFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomGoogLeNetFeatureExtractor, self).__init__()

        # Step 1: 初始化 GoogLeNet，禁用辅助分类器
        self.googlenet = models.googlenet(pretrained=False, aux_logits=False)

        # Step 2: 修改 fc 层输出维度
        in_features = self.googlenet.fc.in_features  # = 1024
        self.googlenet.fc = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在 Step 4 替换 Identity 之前！）
        self.load_weights(model_path)

        # Step 4: 分离并替换分类头
        self.classifier_head = self.googlenet.fc   # 保存 Linear(1024 → num_classes)
        self.googlenet.fc = nn.Identity()           # 替换为直通层
        # 现在 googlenet(x) 输出：Inception块→avgpool→Dropout→flatten→Identity
        # 输出：[B, 1024]

    def forward(self, x, layer='penultimate'):
        """
        ★ 调用前请确保已执行 model.eval() ★
        否则 Dropout 会随机丢弃特征，导致每次结果不同。
        """
        features = self.googlenet(x)
        if layer == 'penultimate':
            return features                         # [B, 1024]
        else:
            return self.classifier_head(features)  # [B, num_classes]

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('googlenet.') for k in state_dict.keys()):
                new_state_dict = {k[10:]: v for k, v in state_dict.items()}  # 去掉前缀（长度10）
                self.googlenet.load_state_dict(new_state_dict, strict=False)
                print("GoogLeNet模型权重加载成功（已处理'googlenet.'前缀）！")
            else:
                self.googlenet.load_state_dict(state_dict, strict=False)
                print("GoogLeNet模型权重加载成功！")
        except Exception as e:
            print(f"加载GoogLeNet模型权重时出错: {e}")
            raise RuntimeError(f"无法加载GoogLeNet模型权重: {e}")


# =============================================================================
# ⑤ ResNet-50 特征提取器
# =============================================================================
# ResNet-50 网络结构：
#   conv1→bn1→relu→maxpool→layer1→layer2→layer3→layer4→avgpool→flatten→fc
#
# 策略：children() 截断，去掉 fc，保留到 avgpool。
# 输出 [B, 2048, 1, 1]，手动 flatten 得到 [B, 2048]。
#
# ★ 注意顺序 ★：这里先截断再加载权重（与 ResNeXt 相反）。
# 因为 self.features 是对 resnet 子模块的引用，
# 加载权重到 self.resnet 后，self.features 里的参数也会同步更新。
# =============================================================================
class CustomResNet50FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomResNet50FeatureExtractor, self).__init__()
        self.resnet = models.resnet50(pretrained=False)
        in_features = self.resnet.fc.in_features  # = 2048
        self.resnet.fc = nn.Linear(in_features, num_classes)

        # 截断：去掉最后的 fc 层
        # 包含：conv1→bn1→relu→maxpool→layer1→layer2→layer3→layer4→avgpool
        self.features = nn.Sequential(*list(self.resnet.children())[:-1])

        # 加载权重（self.features 是引用，权重会同步到 self.resnet 的子模块）
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        x = self.features(x)       # [B, 2048, 1, 1]
        x = torch.flatten(x, 1)    # [B, 2048]
        return x

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('resnet.') for k in state_dict.keys()):
                new_state_dict = {k[7:]: v for k, v in state_dict.items()}  # 去掉前缀（长度7）
                self.resnet.load_state_dict(new_state_dict, strict=False)
                print("ResNet50模型权重加载成功（已处理'resnet.'前缀）！")
            else:
                self.resnet.load_state_dict(state_dict, strict=False)
                print("ResNet50模型权重加载成功！")
        except Exception as e:
            print(f"加载ResNet50模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ResNet50模型权重: {e}")


# =============================================================================
# ⑥ ResNet-101 特征提取器
# =============================================================================
# 与 ResNet-50 完全相同的设计模式，骨干网络更深（101层 vs 50层）。
# 输出特征维度同为 2048。
# =============================================================================
class CustomResNet101FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomResNet101FeatureExtractor, self).__init__()
        self.resnet = models.resnet101(pretrained=False)
        in_features = self.resnet.fc.in_features  # = 2048
        self.resnet.fc = nn.Linear(in_features, num_classes)
        self.features = nn.Sequential(*list(self.resnet.children())[:-1])
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        x = self.features(x)      # [B, 2048, 1, 1]
        x = torch.flatten(x, 1)   # [B, 2048]
        return x

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('resnet.') for k in state_dict.keys()):
                new_state_dict = {k[7:]: v for k, v in state_dict.items()}
                self.resnet.load_state_dict(new_state_dict, strict=False)
                print("ResNet101模型权重加载成功（已处理'resnet.'前缀）！")
            else:
                self.resnet.load_state_dict(state_dict, strict=False)
                print("ResNet101模型权重加载成功！")
        except Exception as e:
            print(f"加载ResNet101模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ResNet101模型权重: {e}")


# =============================================================================
# ⑦ DenseNet-121 特征提取器
# =============================================================================
# DenseNet 网络结构：
#   features（密集连接卷积块）→ F.relu → F.adaptive_avg_pool2d → flatten → classifier
#
# ★ 必须用 Identity 法！★
# DenseNet 的 forward() 中，features 之后有两步函数式调用：
#   out = F.relu(out, inplace=True)
#   out = F.adaptive_avg_pool2d(out, (1, 1))
# 这两步不在 children() 里，用截断法会丢失它们，特征会出错。
# 必须保留官方 forward()，用 Identity 替换 classifier。
#
# classifier 是单个 Linear 层（不是 Sequential），直接替换即可。
# 输出特征维度：DenseNet-121 = 1024
# =============================================================================
class CustomDenseNet121FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomDenseNet121FeatureExtractor, self).__init__()

        # Step 1
        self.densenet = models.densenet121(pretrained=False)

        # Step 2: DenseNet 的 classifier 是单个 Linear（不是 Sequential）
        in_features = self.densenet.classifier.in_features  # = 1024
        self.densenet.classifier = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在替换 Identity 之前！）
        self.load_weights(model_path)

        # Step 4: 分离并替换分类头
        self.classifier_head = self.densenet.classifier  # Linear(1024 → num_classes)
        self.densenet.classifier = nn.Identity()
        # 官方 forward 流程变为：
        # features → F.relu → F.adaptive_avg_pool2d → flatten → Identity
        # 输出：[B, 1024]

    def forward(self, x, layer='penultimate'):
        features = self.densenet(x)
        if layer == 'penultimate':
            return features                         # [B, 1024]
        else:
            return self.classifier_head(features)  # [B, num_classes]

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('densenet.') for k in state_dict.keys()):
                new_state_dict = {k[9:]: v for k, v in state_dict.items()}  # 去掉前缀（长度9）
                self.densenet.load_state_dict(new_state_dict, strict=False)
            else:
                self.densenet.load_state_dict(state_dict, strict=False)
            print("DenseNet121模型权重加载成功！")
        except Exception as e:
            print(f"加载DenseNet121模型权重时出错: {e}")
            raise RuntimeError(f"无法加载DenseNet121模型权重: {e}")


# =============================================================================
# ⑧ DenseNet-161 特征提取器
# =============================================================================
# 与 DenseNet-121 完全相同的设计模式，骨干网络更宽（增长率 48 vs 32）。
# 输出特征维度：DenseNet-161 = 2208
# =============================================================================
class CustomDenseNet161FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomDenseNet161FeatureExtractor, self).__init__()
        self.densenet = models.densenet161(pretrained=False)
        in_features = self.densenet.classifier.in_features  # = 2208
        self.densenet.classifier = nn.Linear(in_features, num_classes)
        self.load_weights(model_path)
        self.classifier_head = self.densenet.classifier
        self.densenet.classifier = nn.Identity()

    def forward(self, x, layer='penultimate'):
        features = self.densenet(x)
        if layer == 'penultimate':
            return features   # [B, 2208]
        else:
            return self.classifier_head(features)

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('densenet.') for k in state_dict.keys()):
                new_state_dict = {k[9:]: v for k, v in state_dict.items()}
                self.densenet.load_state_dict(new_state_dict, strict=False)
            else:
                self.densenet.load_state_dict(state_dict, strict=False)
            print("DenseNet161模型权重加载成功！")
        except Exception as e:
            print(f"加载DenseNet161模型权重时出错: {e}")
            raise RuntimeError(f"无法加载DenseNet161模型权重: {e}")


# =============================================================================
# ⑨ VGG-16 特征提取器
# =============================================================================
# VGG-16 网络结构：
#   features（13个卷积层）→ avgpool → classifier（全连接层组）
#
# classifier 完整结构（索引 0~6）：
#   [0] Linear(25088 → 4096)
#   [1] ReLU
#   [2] Dropout
#   [3] Linear(4096 → 4096)
#   [4] ReLU
#   [5] Dropout
#   [6] Linear(4096 → num_classes)  ← 分类头
#
# 策略：先加载权重，再将 classifier 截断为前 6 层（索引 0~5），输出 4096 维特征。
# VGG 的 forward() 无隐式函数调用，截断法安全。
# =============================================================================
class CustomVGG16FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomVGG16FeatureExtractor, self).__init__()
        self.vgg = models.vgg16(pretrained=False)

        # Step 2: 修改分类头输出维度
        in_features = self.vgg.classifier[6].in_features  # = 4096
        self.vgg.classifier[6] = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在截断 classifier 之前！）
        self.load_weights(model_path)

        # Step 4: 截断 classifier，只保留前 6 层
        self.classifier_head = self.vgg.classifier[6]  # 保存 Linear(4096 → num_classes)
        # 将 vgg 的 classifier 替换为截断后的版本（只含前 6 层，索引 0~5）
        self.vgg.classifier = nn.Sequential(*list(self.vgg.classifier.children())[:6])
        # 现在 vgg(x) 输出：features → avgpool → flatten → classifier(6层) → [B, 4096]

    def forward(self, x, layer='penultimate'):
        features = self.vgg(x)
        if layer == 'penultimate':
            return features                         # [B, 4096]
        else:
            return self.classifier_head(features)  # [B, num_classes]

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('vgg.') for k in state_dict.keys()):
                new_state_dict = {k[4:]: v for k, v in state_dict.items()}  # 去掉前缀（长度4）
                self.vgg.load_state_dict(new_state_dict, strict=False)
            else:
                self.vgg.load_state_dict(state_dict, strict=False)
            print("VGG16模型权重加载成功！")
        except Exception as e:
            print(f"加载VGG16模型权重时出错: {e}")
            raise RuntimeError(f"无法加载VGG16模型权重: {e}")


# =============================================================================
# ⑩ Inception V3 特征提取器
# =============================================================================
# Inception V3 网络结构（简化）：
#   多个 Inception 模块 → avgpool → Dropout → flatten → fc（2048 → num_classes）
#
# ★ 特别注意 ★
# 1. 标准输入尺寸是 299×299（不是 224×224）
#    → SUPPORTED_MODELS 中配置了专用的 pad_to_square_299_transform
# 2. aux_logits=False：禁用辅助分类器（同 GoogLeNet）
# 3. 策略：将 fc 替换为 Identity，输出 2048 维特征
# =============================================================================
class CustomInceptionV3FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomInceptionV3FeatureExtractor, self).__init__()

        # aux_logits=False：禁用辅助分类器，推理时只有一个输出
        self.inception = models.inception_v3(pretrained=False, aux_logits=False)

        # Step 2
        in_features = self.inception.fc.in_features  # = 2048
        self.inception.fc = nn.Linear(in_features, num_classes)

        # Step 3
        self.load_weights(model_path)

        # Step 4
        self.classifier_head = self.inception.fc
        self.inception.fc = nn.Identity()

    def forward(self, x, layer='penultimate'):
        """
        输入 x 应为 [B, 3, 299, 299]，输入 224×224 会导致特征质量下降。
        当 aux_logits=False 时，inception(x) 直接返回 Tensor（非 InceptionOutputs 命名元组）。
        """
        features = self.inception(x)
        if layer == 'penultimate':
            return features                         # [B, 2048]
        else:
            return self.classifier_head(features)

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('inception.') for k in state_dict.keys()):
                new_state_dict = {k[10:]: v for k, v in state_dict.items()}  # 去掉前缀（长度10）
                self.inception.load_state_dict(new_state_dict, strict=False)
            else:
                self.inception.load_state_dict(state_dict, strict=False)
            print("InceptionV3模型权重加载成功！")
        except Exception as e:
            print(f"加载InceptionV3模型权重时出错: {e}")
            raise RuntimeError(f"无法加载InceptionV3模型权重: {e}")


# =============================================================================
# ⑪ EfficientNet-B0 特征提取器
# =============================================================================
# EfficientNet 网络结构：
#   features（MBConv 模块组）→ avgpool → flatten → classifier
#
# classifier 结构（索引 0~1）：
#   [0] Dropout
#   [1] Linear(1280 → num_classes)
#
# 策略：直接引用 features 和 avgpool 子模块，手动构建 forward，
# 绕过 classifier，输出 1280 维特征。
# 这里用"手动引用子模块"而非 Identity，是另一种安全的特征提取方式。
# =============================================================================
class CustomEfficientNetB0FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomEfficientNetB0FeatureExtractor, self).__init__()
        self.efficientnet = models.efficientnet_b0(pretrained=False)

        # Step 2
        in_features = self.efficientnet.classifier[1].in_features  # = 1280
        self.efficientnet.classifier[1] = nn.Linear(in_features, num_classes)

        # 引用子模块（不是复制，是引用，权重共享）
        # self.features 指向 efficientnet 内部的 MBConv 模块组
        self.features = self.efficientnet.features
        # self.avgpool 指向 efficientnet 内部的自适应平均池化层
        self.avgpool = self.efficientnet.avgpool

        # Step 3
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        """
        手动 forward，绕过 classifier：
        features → avgpool → flatten → [B, 1280]
        """
        x = self.features(x)     # MBConv 模块组，[B, 1280, 7, 7]（224输入时）
        x = self.avgpool(x)      # 自适应平均池化，[B, 1280, 1, 1]
        x = torch.flatten(x, 1)  # 展平，[B, 1280]
        return x

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('efficientnet.') for k in state_dict.keys()):
                new_state_dict = {k[13:]: v for k, v in state_dict.items()}  # 去掉前缀（长度13）
                self.efficientnet.load_state_dict(new_state_dict, strict=False)
                print("EfficientNetB0模型权重加载成功（已处理'efficientnet.'前缀）！")
            else:
                self.efficientnet.load_state_dict(state_dict, strict=False)
                print("EfficientNetB0模型权重加载成功！")
        except Exception as e:
            print(f"加载EfficientNetB0模型权重时出错: {e}")
            raise RuntimeError(f"无法加载EfficientNetB0模型权重: {e}")


# =============================================================================
# ⑫ EfficientNet-V2-Large 特征提取器
# =============================================================================
# 与 EfficientNet-B0 完全相同的设计模式，使用更大的 V2-Large 变体。
# 输出特征维度同为 1280。
# =============================================================================
class CustomEfficientNetV2LargeFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomEfficientNetV2LargeFeatureExtractor, self).__init__()
        self.efficientnet = models.efficientnet_v2_l(pretrained=False)
        in_features = self.efficientnet.classifier[1].in_features  # = 1280
        self.efficientnet.classifier[1] = nn.Linear(in_features, num_classes)
        self.features = self.efficientnet.features   # 引用 MBConv 模块组
        self.avgpool = self.efficientnet.avgpool     # 引用池化层
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # [B, 1280]
        return x

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('efficientnet.') for k in state_dict.keys()):
                new_state_dict = {k[13:]: v for k, v in state_dict.items()}
                self.efficientnet.load_state_dict(new_state_dict, strict=False)
                print("EfficientNetV2L模型权重加载成功（已处理'efficientnet.'前缀）！")
            else:
                self.efficientnet.load_state_dict(state_dict, strict=False)
                print("EfficientNetV2L模型权重加载成功！")
        except Exception as e:
            print(f"加载EfficientNetV2L模型权重时出错: {e}")
            raise RuntimeError(f"无法加载EfficientNetV2L模型权重: {e}")


# =============================================================================
# ⑬ MobileNet-V2 特征提取器
# =============================================================================
# MobileNetV2 网络结构：
#   features（倒残差模块组）→ adaptive_avg_pool2d → flatten → classifier
#
# ★ 注意 ★：adaptive_avg_pool2d 和 flatten 是官方 forward() 中的函数式调用，
# 不在 children() 里，必须用 Identity 替换整个 classifier，保留官方 forward。
#
# classifier 结构（索引 0~1）：
#   [0] Dropout
#   [1] Linear(1280 → num_classes)
# 这里将整个 classifier（含 Dropout）替换为 Identity。
# 输出特征维度：1280
# =============================================================================
class CustomMobileNetV2FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomMobileNetV2FeatureExtractor, self).__init__()
        self.mobilenet = models.mobilenet_v2(pretrained=False)

        # Step 2
        in_features = self.mobilenet.classifier[1].in_features  # = 1280
        self.mobilenet.classifier[1] = nn.Linear(in_features, num_classes)

        # Step 3
        self.load_weights(model_path)

        # Step 4: 将整个 classifier（Dropout + Linear）替换为 Identity
        # 官方 forward 中 features 之后的 adaptive_avg_pool2d 和 flatten
        # 是函数式调用，不受影响，会正常执行
        self.classifier_head = self.mobilenet.classifier  # 保存完整分类头
        self.mobilenet.classifier = nn.Identity()
        # 官方 forward 流程变为：
        # features → adaptive_avg_pool2d((1,1)) → flatten → Identity
        # 输出：[B, 1280]

    def forward(self, x, layer='penultimate'):
        features = self.mobilenet(x)
        if layer == 'penultimate':
            return features                         # [B, 1280]
        else:
            return self.classifier_head(features)

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('mobilenet.') for k in state_dict.keys()):
                new_state_dict = {k[10:]: v for k, v in state_dict.items()}  # 去掉前缀（长度10）
                self.mobilenet.load_state_dict(new_state_dict, strict=False)
            else:
                self.mobilenet.load_state_dict(state_dict, strict=False)
            print("MobileNetV2模型权重加载成功！")
        except Exception as e:
            print(f"加载MobileNetV2模型权重时出错: {e}")
            raise RuntimeError(f"无法加载MobileNetV2模型权重: {e}")


# =============================================================================
# ⑭ MobileNet-V3-Large 特征提取器
# =============================================================================
# MobileNetV3-Large 网络结构：
#   features（含 SE 注意力的模块组）→ avgpool → flatten → classifier
#
# classifier 结构（索引 0~3）：
#   [0] Linear(960 → 1280)
#   [1] Hardswish
#   [2] Dropout
#   [3] Linear(1280 → num_classes)
#
# 策略：直接引用 features 子模块，手动构建 forward，
# 用 F.adaptive_avg_pool2d 进行全局池化，输出 960 维特征。
# （输出的是 features 之后、classifier 之前的特征，即 960 维，而非 1280 维）
#
# ★ 分类头修改逻辑较复杂 ★
# 需要处理 Sequential 结构，只替换最后一个 Linear 的输出维度，
# 保留前面的 Linear(960→1280)、Hardswish、Dropout 层。
# =============================================================================
class CustomMobileNetV3FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomMobileNetV3FeatureExtractor, self).__init__()
        self.mobilenet = models.mobilenet_v3_large(pretrained=False)

        # Step 2: 修改分类头（需要处理 Sequential 结构）
        original_classifier = self.mobilenet.classifier
        if isinstance(original_classifier, nn.Sequential):
            layers = list(original_classifier.children())
            if isinstance(layers[-1], nn.Linear):
                in_features = layers[-1].in_features
                # 保留前面所有层，只替换最后一个 Linear 的输出维度
                new_layers = layers[:-1] + [nn.Linear(in_features, num_classes)]
                self.mobilenet.classifier = nn.Sequential(*new_layers)
            else:
                # 兜底方案：重建标准 MobileNetV3-Large 分类器
                self.mobilenet.classifier = nn.Sequential(
                    nn.Linear(1280, 1024),
                    nn.Hardswish(inplace=True),
                    nn.Dropout(p=0.2, inplace=True),
                    nn.Linear(1024, num_classes)
                )
        else:
            # 兜底方案
            self.mobilenet.classifier = nn.Sequential(
                nn.Linear(1280, 1024),
                nn.Hardswish(inplace=True),
                nn.Dropout(p=0.2, inplace=True),
                nn.Linear(1024, num_classes)
            )

        # 引用 features 子模块（SE模块组，输出 960 通道）
        self.features = self.mobilenet.features

        # Step 3
        self.load_weights(model_path)

    def forward(self, x, layer='penultimate'):
        """
        手动 forward，绕过 classifier：
        features → adaptive_avg_pool2d → flatten → [B, 960]
        """
        x = self.features(x)
        # 全局平均池化：将任意空间尺寸压缩为 1×1
        x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))
        x = torch.flatten(x, 1)  # [B, 960]
        return x

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('mobilenet.') for k in state_dict.keys()):
                new_state_dict = {k[10:]: v for k, v in state_dict.items()}
                self.mobilenet.load_state_dict(new_state_dict, strict=False)
                print("MobileNetV3模型权重加载成功（已处理'mobilenet.'前缀）！")
            else:
                self.mobilenet.load_state_dict(state_dict, strict=False)
                print("MobileNetV3模型权重加载成功！")
        except Exception as e:
            print(f"加载MobileNetV3模型权重时出错: {e}")
            raise RuntimeError(f"无法加载MobileNetV3模型权重: {e}")


# =============================================================================
# ⑮ RegNet-Y-400MF 特征提取器
# =============================================================================
# RegNet 网络结构：
#   stem → trunk_output（4个stage）→ avgpool → flatten → fc
#
# 策略：将 fc 替换为 Identity，输出 440 维特征。
# （RegNet-Y-400MF 的最终通道数为 440）
# =============================================================================
class CustomRegNetFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomRegNetFeatureExtractor, self).__init__()
        self.regnet = models.regnet_y_400mf(pretrained=False)

        # Step 2
        in_features = self.regnet.fc.in_features  # = 440
        self.regnet.fc = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在替换 Identity 之前！）
        self.load_weights(model_path)

        # Step 4
        self.classifier_head = self.regnet.fc
        self.regnet.fc = nn.Identity()
        # 官方 forward：stem → trunk → avgpool → flatten → Identity
        # 输出：[B, 440]

    def forward(self, x, layer='penultimate'):
        features = self.regnet(x)
        if layer == 'penultimate':
            return features                         # [B, 440]
        else:
            return self.classifier_head(features)

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('regnet.') for k in state_dict.keys()):
                new_state_dict = {k[7:]: v for k, v in state_dict.items()}  # 去掉前缀（长度7）
                self.regnet.load_state_dict(new_state_dict, strict=False)
            else:
                self.regnet.load_state_dict(state_dict, strict=False)
            print("RegNet模型权重加载成功！")
        except Exception as e:
            print(f"加载RegNet模型权重时出错: {e}")
            raise RuntimeError(f"无法加载RegNet模型权重: {e}")


# =============================================================================
# ⑯ ShuffleNet-V2-x1.0 特征提取器
# =============================================================================
# ShuffleNetV2 网络结构：
#   conv1 → maxpool → stage2 → stage3 → stage4 → conv5 → mean([2,3]) → fc
#
# ★ 必须用 Identity 法！★
# 官方 forward() 中有：x = x.mean([2, 3])（全局平均池化的函数式调用）
# 这步不在 children() 里，用截断法会丢失，导致维度错误。
# 输出特征维度：1024
# =============================================================================
class CustomShuffleNetV2FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomShuffleNetV2FeatureExtractor, self).__init__()
        self.shufflenet = models.shufflenet_v2_x1_0(pretrained=False)

        # Step 2
        in_features = self.shufflenet.fc.in_features  # = 1024
        self.shufflenet.fc = nn.Linear(in_features, num_classes)

        # Step 3
        self.load_weights(model_path)

        # Step 4: 分离并替换分类头
        # 官方 forward 中 x.mean([2, 3]) 是函数式调用，不能截断，
        # 必须保留官方 forward，用 Identity 替换 fc
        self.classifier_head = self.shufflenet.fc
        self.shufflenet.fc = nn.Identity()
        # 官方 forward：conv1→maxpool→stages→conv5→mean([2,3])→Identity
        # 输出：[B, 1024]

    def forward(self, x, layer='penultimate'):
        features = self.shufflenet(x)
        if layer == 'penultimate':
            return features                         # [B, 1024]
        else:
            return self.classifier_head(features)

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('shufflenet.') for k in state_dict.keys()):
                new_state_dict = {k[11:]: v for k, v in state_dict.items()}  # 去掉前缀（长度11）
                self.shufflenet.load_state_dict(new_state_dict, strict=False)
            else:
                self.shufflenet.load_state_dict(state_dict, strict=False)
            print("ShuffleNetV2模型权重加载成功！")
        except Exception as e:
            print(f"加载ShuffleNetV2模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ShuffleNetV2模型权重: {e}")


# =============================================================================
# ⑰ SqueezeNet 特征提取器（支持 1.0 和 1.1 版本自动检测）
# =============================================================================
# SqueezeNet 网络结构（特殊！）：
#   features（Fire模块组）→ classifier（Conv2d！不是Linear）→ avgpool → flatten
#
# ★ 与其他模型最大的不同 ★
# SqueezeNet 的分类器是 Conv2d(512, num_classes, kernel_size=1)，
# 而非 Linear 层。这是它的独特设计：用 1×1 卷积代替全连接层，
# 大幅减少参数量。
#
# 因此特征提取方式也不同：
#   - 不能用 Identity 替换 classifier（因为 classifier 是 Conv2d，维度不对）
#   - 直接使用 features 部分，手动做全局平均池化，输出 512 维特征
#
# ★ 版本自动检测机制 ★
# SqueezeNet 有 1.0 和 1.1 两个版本，网络结构略有不同：
#   v1.0: features[0] 卷积核为 7×7，输出 96 通道，参数更多
#   v1.1: features[0] 卷积核为 3×3，输出 64 通道，更轻量
# 如果版本与权重不匹配，加载时会报形状不匹配错误。
# 解决方案：在初始化模型之前，先读取权重文件，
# 通过 features.0.weight 的形状自动判断版本，再初始化对应版本。
# =============================================================================
class CustomSqueezeNetFeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomSqueezeNetFeatureExtractor, self).__init__()

        # ── Step 1+2 合并：先检测版本，再初始化对应模型 ──────────────────────
        # 这里的顺序与其他模型不同！
        # 其他模型：先初始化模型 → 再加载权重
        # SqueezeNet：先读权重检测版本 → 再初始化对应版本模型 → 再正式加载权重
        # 原因：必须先知道是 1.0 还是 1.1，才能初始化正确的模型结构
        state_dict, version = self._detect_version_and_load_dict(model_path)

        if version == '1.1':
            print("检测到权重版本为 SqueezeNet 1.1，正在初始化...")
            self.squeezenet = models.squeezenet1_1(weights=None)
            # weights=None：不加载 torchvision 的预训练权重，我们有自己的权重
        else:
            print("检测到权重版本为 SqueezeNet 1.0，正在初始化...")
            self.squeezenet = models.squeezenet1_0(weights=None)

        # ── Step 2：修改分类器结构以匹配训练时的结构 ─────────────────────────
        # SqueezeNet 的 classifier 是一个 Sequential，包含：
        #   [0] Dropout
        #   [1] Conv2d(512, num_classes_原始=1000, kernel_size=1)  ← 需要修改
        #   [2] ReLU
        #   [3] AdaptiveAvgPool2d
        # 我们只修改 [1] 的输出通道数（从 1000 改为 num_classes）
        self.squeezenet.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)

        # ── Step 3：加载权重 ──────────────────────────────────────────────────
        self.load_weights(model_path)

        # ── Step 4：保存子模块引用，供 forward 使用 ──────────────────────────
        # features_block：Fire模块组，输出形状 [B, 512, H, W]
        # 这是我们真正需要的特征提取部分
        self.features_block = self.squeezenet.features

        # classifier_head：保存完整分类器，供 'final' 模式使用
        # 注意：SqueezeNet 的 classifier 包含 AvgPool，
        # 所以 'final' 模式的调用方式比较特殊（见 forward 中的注释）
        self.classifier_head = self.squeezenet.classifier

    def _detect_version_and_load_dict(self, model_path):
        """
        内部辅助函数：通过检查权重文件中第一层卷积的形状，自动判断 SqueezeNet 版本。

        判断依据（features.0.weight 的形状）：
            v1.0: [96, 3, 7, 7]  → 第2个维度索引[2] = 7（卷积核高度为7）
            v1.1: [64, 3, 3, 3]  → 第2个维度索引[2] = 3（卷积核高度为3）

        shape 的四个维度含义：
            [输出通道数, 输入通道数, 卷积核高度, 卷积核宽度]
            所以 shape[2] 就是卷积核的高度（等于宽度，因为是方形卷积核）

        Returns:
            (state_dict, version)
            state_dict: 已去除前缀的权重字典
            version: '1.0' 或 '1.1'
        """
        try:
            # 加载权重文件到 CPU（此时只是为了检测版本，不是正式加载）
            state_dict = torch.load(model_path, map_location='cpu')

            # 处理可能存在的 'squeezenet.' 前缀
            # 如果所有键都以 'squeezenet.' 开头，说明训练时模型被包裹在另一个类里
            if all(k.startswith('squeezenet.') for k in state_dict.keys()):
                # 去掉 'squeezenet.' 前缀（长度为 11）
                state_dict = {k[11:]: v for k, v in state_dict.items()}

            # 核心判断：读取第一层卷积权重的形状
            # 'features.0.weight' 是 SqueezeNet 第一个卷积层的权重键名
            first_layer_shape = state_dict['features.0.weight'].shape

            # shape[2] 是卷积核高度
            if first_layer_shape[2] == 3:   # 3×3 卷积核 → v1.1
                version = '1.1'
            else:                            # 7×7 卷积核 → v1.0
                version = '1.0'

            return state_dict, version

        except Exception as e:
            raise RuntimeError(f"解析权重文件失败，请检查路径或文件格式: {e}")

    def forward(self, x, layer='penultimate'):
        """
        前向传播，手动构建特征提取流程，绕过 classifier。

        SqueezeNet 的特征提取流程：
            features（Fire模块组）→ adaptive_avg_pool2d → flatten → [B, 512]

        Args:
            x: 输入图像张量 [B, 3, 224, 224]
            layer: 'penultimate' → 返回 512 维特征向量（默认）
                   其他值       → 返回分类 logits（特殊处理，见下方注释）
        """
        # Step 1: 通过 Fire 模块组提取卷积特征
        # Fire 模块 = squeeze层（1×1卷积降维）+ expand层（1×1和3×3卷积升维）
        # 输出形状：[B, 512, H, W]（H、W 取决于输入尺寸，224输入时约为 13×13）
        x = self.features_block(x)

        # Step 2: 全局平均池化，将空间维度压缩为 1×1
        # [B, 512, H, W] → [B, 512, 1, 1]
        x = torch.nn.functional.adaptive_avg_pool2d(x, (1, 1))

        # Step 3: 展平
        # [B, 512, 1, 1] → [B, 512]
        x = torch.flatten(x, 1)

        if layer == 'penultimate':
            return x  # [B, 512]
        else:
            # ★ 特殊处理 ★
            # SqueezeNet 的 classifier 期望输入是 4D 张量（带空间维度的特征图），
            # 而不是 2D 的展平向量。
            # 因此需要先用 unsqueeze 恢复空间维度：
            #   [B, 512] → unsqueeze(2) → [B, 512, 1] → unsqueeze(3) → [B, 512, 1, 1]
            # 然后通过 classifier（Conv2d → ReLU → AvgPool），
            # 最后 flatten 得到 logits。
            # 注意：这种方式不如直接调用 self.squeezenet(original_input) 自然，
            # 如果需要 logits，建议保存原始输入并直接调用完整模型。
            return self.classifier_head(x.unsqueeze(2).unsqueeze(3)).flatten(1)

    def load_weights(self, model_path):
        """
        正式加载模型权重（_detect_version_and_load_dict 只是预读，这里才是正式加载）。
        """
        try:
            state_dict = torch.load(model_path, map_location='cpu')

            # 处理 'squeezenet.' 前缀
            if all(k.startswith('squeezenet.') for k in state_dict.keys()):
                new_state_dict = {k[11:]: v for k, v in state_dict.items()}
            else:
                new_state_dict = state_dict

            # strict=False：允许部分键不匹配（如修改过的 classifier[1]）
            self.squeezenet.load_state_dict(new_state_dict, strict=False)
            print("SqueezeNet模型权重加载成功！")

        except Exception as e:
            print(f"加载SqueezeNet模型权重时出错: {e}")
            raise RuntimeError(f"无法加载SqueezeNet模型权重: {e}")


# =============================================================================
# ⑱ ViT-B/16 特征提取器（Vision Transformer Base，Patch Size 16）
# =============================================================================
# ViT 与 CNN 的根本区别：
#   CNN：通过卷积核在空间上滑动提取局部特征
#   ViT：将图像切分为固定大小的 patch（16×16），
#        每个 patch 展平后作为一个 token，
#        通过 Transformer 的自注意力机制捕捉全局依赖关系
#
# ViT-B/16 网络结构（torchvision 实现）：
#   conv_proj（patch embedding）→ encoder（12层 Transformer）→
#   select class token（取第0个token）→ heads（分类头）
#
# heads 的结构：
#   heads.head = Linear(768 → num_classes)
#
# ★ class token 的作用 ★
# 输入序列的第 0 个位置是一个可学习的 [CLS] token（class token），
# 它不对应任何图像 patch，而是在 Transformer 中聚合所有 patch 的信息。
# 最终用这个 token 的输出向量做分类，维度为 768（ViT-B 的隐藏层维度）。
#
# 策略：将整个 heads 模块替换为 Identity，
# 官方 forward 会自动选取 class token 并输出 768 维特征。
# =============================================================================
class CustomViT_B16FeatureExtractor(nn.Module):
    def __init__(self, model_path, num_classes=2):
        super(CustomViT_B16FeatureExtractor, self).__init__()

        # Step 1: 初始化 ViT-B/16（不加载预训练权重）
        self.vit = models.vit_b_16(pretrained=False)

        # Step 2: 修改分类头输出维度
        # vit.heads 是一个 Sequential，vit.heads.head 是其中的 Linear 层
        in_features = self.vit.heads.head.in_features  # = 768（ViT-B 的隐藏维度）
        self.vit.heads.head = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重（必须在替换 Identity 之前！）
        self.load_weights(model_path)

        # Step 4: 将整个 heads 模块替换为 Identity
        # torchvision ViT 的官方 forward 流程：
        #   x → conv_proj → encoder → x[:, 0]（取class token）→ heads
        # 替换 heads 后，输出就是 class token 的 768 维特征向量
        self.classifier_head = self.vit.heads   # 保存完整分类头
        self.vit.heads = nn.Identity()

    def forward(self, x, layer='penultimate'):
        """
        官方 forward：
        conv_proj → encoder（12层Transformer）→ 取class token → Identity
        输出 [B, 768]
        """
        features = self.vit(x)
        if layer == 'penultimate':
            return features                         # [B, 768]
        else:
            return self.classifier_head(features)  # [B, num_classes]

    def load_weights(self, model_path):
        try:
            state_dict = torch.load(model_path, map_location='cpu')
            if all(k.startswith('vit.') for k in state_dict.keys()):
                # 去掉 'vit.' 前缀（长度为 4）
                new_state_dict = {k[4:]: v for k, v in state_dict.items()}
                self.vit.load_state_dict(new_state_dict, strict=False)
            else:
                self.vit.load_state_dict(state_dict, strict=False)
            print("ViT-B16模型权重加载成功！")
        except Exception as e:
            print(f"加载ViT-B16模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ViT-B16模型权重: {e}")


# =============================================================================
# ⑲ ViT-L/16 特征提取器（Vision Transformer Large，Patch Size 16）
# =============================================================================
# 与 ViT-B/16 的区别：
#   ViT-B/16：12层 Transformer，隐藏维度 768，输出特征 768 维
#   ViT-L/16：24层 Transformer，隐藏维度 1024，输出特征 1024 维
#
# ★ 与 ViT-B/16 的实现差异 ★
# ViT-B/16 用 Identity 替换 heads，利用官方 forward 直接输出 class token。
# ViT-L/16 用了另一种方法：Forward Hook（前向钩子）。
#
# ★ 什么是 Forward Hook？★
# PyTorch 允许给任意模块注册一个"钩子函数"，
# 每次该模块执行 forward() 时，钩子函数会被自动调用，
# 可以在钩子里捕获该模块的输出。
#
# ViT-L/16 的 forward 流程：
#   conv_proj → encoder（24层Transformer）→ x[:, 0]（取class token）→ heads
#
# 这里在 encoder 的最后一层（encoder.layers[-1]）注册钩子，
# 捕获该层输出的 class token（output[:, 0]），
# 这样得到的是经过 24 层 Transformer 处理但还未经过 heads 的原始特征。
#
# ★ 为什么 ViT-L/16 不用 Identity 而用 Hook？★
# 注释掉的代码（上方的 # 注释块）显示，原本也打算用 Identity 方法，
# 但当前激活的实现改用了 Hook 方法。
# Hook 方法的优点：不修改模型结构，更灵活；
# 缺点：每次 forward 都要注册和移除钩子，有轻微性能开销。
# =============================================================================
class CustomViT_L16FeatureExtractor(nn.Module):
    # ── 注释掉的代码块说明 ────────────────────────────────────────────────────
    # 下方注释掉的代码是最初的实现方案（与 ViT-B/16 相同，用 Identity 替换 heads）：
    #   self.classifier_head = self.vit.heads
    #   self.vit.heads = nn.Identity()
    # 该方案被放弃，改用 Hook 方法（当前激活的实现）。
    # 两种方案的特征提取结果应该相同，但 Hook 方法不修改模型结构。
    # ─────────────────────────────────────────────────────────────────────────

    def __init__(self, model_path, num_classes=2):
        super(CustomViT_L16FeatureExtractor, self).__init__()

        # Step 1: 初始化 ViT-L/16
        self.vit = models.vit_l_16(pretrained=False)

        # Step 2: 修改分类头输出维度
        in_features = self.vit.heads.head.in_features  # = 1024（ViT-L 的隐藏维度）
        self.vit.heads.head = nn.Linear(in_features, num_classes)

        # Step 3: 加载权重
        # 注意：这里没有 Step 4（不替换 Identity），
        # 而是在 forward 中用 Hook 动态捕获特征
        self.load_weights(model_path)

    def forward(self, x, layer=None):
        """
        使用 Forward Hook 提取 encoder 最后一层的 class token 特征。

        ★ Hook 工作原理 ★
        1. 定义钩子函数 hook_fn：当目标模块执行 forward 时被调用，
           捕获输出的第 0 个 token（class token）
        2. 将钩子注册到 encoder 最后一层
        3. 执行完整的 vit forward（包括 heads 分类头）
        4. 移除钩子（避免内存泄漏和重复捕获）
        5. 返回钩子捕获的 class token 特征

        Args:
            x: 输入图像张量 [B, 3, 224, 224]
            layer: 保留参数（为与其他提取器接口一致），实际不使用
        Returns:
            class_token_feature: [B, 1024]
        """
        # 用于存储钩子捕获的特征（使用列表是为了在闭包中修改外部变量）
        class_token_feature = None

        def hook_fn(module, input, output):
            """
            钩子函数，在目标模块 forward 完成后自动调用。
            参数：
                module: 触发钩子的模块（即 encoder.layers[-1]）
                input:  该模块的输入（元组形式）
                output: 该模块的输出张量，形状为 [B, num_tokens, hidden_dim]
                        其中 num_tokens = 1(class token) + num_patches
                        ViT-L/16 输入 224×224 时：num_patches = (224/16)² = 196
                        所以 output 形状为 [B, 197, 1024]
            """
            nonlocal class_token_feature
            # output[:, 0] 取所有 batch 的第 0 个 token（class token）
            # [B, 197, 1024] → [B, 1024]
            class_token_feature = output[:, 0]

        # 找到 encoder 的最后一层 Transformer block
        # self.vit.encoder.layers 是一个 ModuleList，包含 24 个 EncoderBlock
        # [-1] 取最后一个（第 24 层）
        last_layer = self.vit.encoder.layers[-1]

        # 注册前向钩子到最后一层
        # register_forward_hook 返回一个 hook handle，用于后续移除钩子
        hook = last_layer.register_forward_hook(hook_fn)

        # 执行完整的 forward（钩子会在 last_layer 执行时自动触发）
        # 返回值是分类 logits，这里用 _ 忽略，我们只需要钩子捕获的特征
        _ = self.vit(x)

        # ★ 必须移除钩子！★
        # 如果不移除，下次调用 forward 时钩子会再次触发，
        # 导致 class_token_feature 被覆盖（虽然结果相同，但会有内存泄漏风险）
        hook.remove()

        # 返回钩子捕获的 class token 特征
        return class_token_feature  # [B, 1024]

    def load_weights(self, model_path):
        """
        注意：这里使用 strict=True（默认），不允许键名不匹配。
        与其他模型使用 strict=False 不同，
        这意味着权重文件必须与模型结构完全匹配，否则会报错。
        """
        try:
            state_dict = torch.load(model_path)
            if all(k.startswith('vit.') for k in state_dict.keys()):
                new_state_dict = {k[4:]: v for k, v in state_dict.items()}
                # 注意：这里没有 strict=False，使用默认的 strict=True
                # 要求权重文件与模型结构完全一致
                self.vit.load_state_dict(new_state_dict)
                print("ViT模型权重加载成功（已处理'vit.'前缀）！")
            else:
                self.vit.load_state_dict(state_dict)
                print("ViT模型权重加载成功！")
        except Exception as e:
            print(f"加载ViT模型权重时出错: {e}")
            raise RuntimeError(f"无法加载ViT模型权重: {e}")



### 图像预处理辅助函数（pad_to_square_transform和pad_to_square_299_transform）

In [ ]:

# =============================================================================
# 图像预处理辅助函数
# =============================================================================
# 背景：细胞图像的尺寸各不相同，且通常不是正方形。
# 大多数 CNN 要求固定尺寸的正方形输入（如 224×224）。
# 直接 resize 会导致图像变形（细胞形态失真），影响特征质量。
# 解决方案：先将图像填充为正方形（保持宽高比），再 resize 到目标尺寸。
# 填充用白色背景（255,255,255），与病理切片的白色背景一致。
# =============================================================================

def pad_to_square_transform(img):
    """
    将任意尺寸的 PIL 图像填充为正方形，然后 resize 到 224×224。
    用于除 InceptionV3 之外的所有模型。

    处理流程：
        原始图像（任意尺寸）
        → 计算最长边 max_side
        → 创建 max_side × max_side 的白色背景图
        → 将原图居中粘贴到白色背景上（短边两侧各填充一半白色）
        → resize 到 224×224

    Args:
        img: PIL.Image 对象（任意尺寸）
    Returns:
        PIL.Image 对象，尺寸为 224×224
    """
    try:
        # 获取原始图像的宽和高（PIL 中 size 返回 (width, height)）
        width, height = img.size

        if width != height:
            # 图像不是正方形，需要填充
            # 取较长的边作为正方形的边长
            max_side = max(width, height)

            # 创建白色背景正方形图像
            # Image.new(mode, size, color)
            # 'RGB'：三通道彩色图像
            # (max_side, max_side)：目标尺寸（宽, 高）
            # (255, 255, 255)：白色背景（R=255, G=255, B=255）
            new_image = Image.new('RGB', (max_side, max_side), (255, 255, 255))

            # 将原图居中粘贴
            # paste(im, box)：box 是左上角坐标 (x, y)
            # 水平居中：(max_side - width) // 2
            # 垂直居中：(max_side - height) // 2
            new_image.paste(img, ((max_side - width) // 2, (max_side - height) // 2))

            # resize 到 224×224
            # Image.LANCZOS：高质量的下采样滤波器（原名 ANTIALIAS），
            # 比 BILINEAR 和 NEAREST 质量更好，适合缩小图像
            return new_image.resize((224, 224), Image.LANCZOS)
        else:
            # 已经是正方形，直接 resize（不需要填充）
            return img.resize((224, 224), Image.LANCZOS)

    except Exception as e:
        print(f"Error in pad_to_square_transform: {e}")
        # 出错时返回纯白色的默认图像，避免整个流程崩溃
        return Image.new('RGB', (224, 224), (255, 255, 255))


def pad_to_square_299_transform(img):
    """
    将任意尺寸的 PIL 图像填充为正方形，然后 resize 到 299×299。
    专用于 InceptionV3（其标准输入尺寸为 299×299，而非 224×224）。

    与 pad_to_square_transform 的唯一区别：目标尺寸是 299×299。

    ★ 为什么 InceptionV3 需要 299×299？★
    InceptionV3 的设计中，第一层卷积步长为 2，后续还有多次步长为 2 的操作，
    经过多次下采样后，特征图尺寸为 8×8。
    如果输入 224×224，最终特征图尺寸会不匹配，导致全局平均池化后维度异常。
    使用 299×299 才能保证特征图尺寸正确（8×8 → avgpool → 1×1 → 2048维）。

    Args:
        img: PIL.Image 对象（任意尺寸）
    Returns:
        PIL.Image 对象，尺寸为 299×299
    """
    try:
        width, height = img.size
        max_side = max(width, height)

        # 注意：这里没有 if width != height 的判断，
        # 无论是否是正方形都会创建新图像（与 224 版本略有不同，但结果等价）
        new_image = Image.new('RGB', (max_side, max_side), (255, 255, 255))
        new_image.paste(img, ((max_side - width) // 2, (max_side - height) // 2))

        # resize 到 299×299
        new_image = new_image.resize((299, 299), Image.LANCZOS)
        return new_image

    except Exception as e:
        print(f"Error in pad_to_square_299_transform: {e}")
        return Image.new('RGB', (299, 299), (255, 255, 255))



### 工厂函数：get_model_and_transform

In [ ]:

# =============================================================================
# 模型注册表：SUPPORTED_MODELS
# =============================================================================
# 这是一个字典，将模型名称字符串映射到该模型的所有配置信息。
# 工厂函数 get_model_and_transform() 通过查询这个字典来创建模型。
#
# 每个条目包含三个字段：
#   'class'    : 对应的特征提取器类（未实例化）
#   'input_size': 模型期望的输入图像尺寸（正方形边长）
#   'pad_func' : 图像预处理函数（将图像填充并缩放到 input_size）
#
# 设计优点：
#   - 新增模型只需在这里添加一行，工厂函数无需修改
#   - 模型名称与实现解耦，外部调用只需传字符串
#   - input_size 和 pad_func 绑定在一起，避免配置不一致
# =============================================================================
SUPPORTED_MODELS = {
    # 模型名称字符串          对应类                                输入尺寸  填充函数
    'AlexNet':          {'class': CustomAlexNetFeatureExtractor,          'input_size': 224, 'pad_func': pad_to_square_transform},
    'ResNeXt':          {'class': CustomResNeXtFeatureExtractor,          'input_size': 224, 'pad_func': pad_to_square_transform},
    'ConvNextLarge':    {'class': CustomConvNextLargeFeatureExtractor,    'input_size': 224, 'pad_func': pad_to_square_transform},
    'GoogLeNet':        {'class': CustomGoogLeNetFeatureExtractor,        'input_size': 224, 'pad_func': pad_to_square_transform},
    'ResNet50':         {'class': CustomResNet50FeatureExtractor,         'input_size': 224, 'pad_func': pad_to_square_transform},
    'ResNet101':        {'class': CustomResNet101FeatureExtractor,        'input_size': 224, 'pad_func': pad_to_square_transform},
    'DenseNet121':      {'class': CustomDenseNet121FeatureExtractor,      'input_size': 224, 'pad_func': pad_to_square_transform},
    'DenseNet161':      {'class': CustomDenseNet161FeatureExtractor,      'input_size': 224, 'pad_func': pad_to_square_transform},
    'VGG16':            {'class': CustomVGG16FeatureExtractor,            'input_size': 224, 'pad_func': pad_to_square_transform},
    'InceptionV3':      {'class': CustomInceptionV3FeatureExtractor,      'input_size': 299, 'pad_func': pad_to_square_299_transform},  # ← 唯一使用 299 的模型
    'EfficientNetB0':   {'class': CustomEfficientNetB0FeatureExtractor,   'input_size': 224, 'pad_func': pad_to_square_transform},
    'EfficientNetV2Large': {'class': CustomEfficientNetV2LargeFeatureExtractor, 'input_size': 224, 'pad_func': pad_to_square_transform},
    'MobileNetV2':      {'class': CustomMobileNetV2FeatureExtractor,      'input_size': 224, 'pad_func': pad_to_square_transform},
    'MobileNetV3':      {'class': CustomMobileNetV3FeatureExtractor,      'input_size': 224, 'pad_func': pad_to_square_transform},
    'RegNet':           {'class': CustomRegNetFeatureExtractor,           'input_size': 224, 'pad_func': pad_to_square_transform},
    'ShuffleNetV2':     {'class': CustomShuffleNetV2FeatureExtractor,     'input_size': 224, 'pad_func': pad_to_square_transform},
    'SqueezeNet':       {'class': CustomSqueezeNetFeatureExtractor,       'input_size': 224, 'pad_func': pad_to_square_transform},
    'ViT_B16':          {'class': CustomViT_B16FeatureExtractor,          'input_size': 224, 'pad_func': pad_to_square_transform},
    'ViT_L16':          {'class': CustomViT_L16FeatureExtractor,          'input_size': 224, 'pad_func': pad_to_square_transform},
}


# =============================================================================
# 工厂函数：get_model_and_transform()
# =============================================================================
# 这是整个 models.py 对外暴露的唯一接口。
# 外部代码（如 qrr_pipeline.py）只需调用这一个函数，
# 传入模型名称字符串，就能得到：
#   1. 已加载权重、已移至目标设备的特征提取器模型
#   2. 配套的图像预处理 transform 流程
#   3. 该模型期望的输入图像尺寸
#
# 这是"工厂模式"设计：调用方不需要知道具体类的名称，
# 只需要知道模型类型字符串，工厂函数负责创建正确的对象。
# =============================================================================
def get_model_and_transform(model_type, device, model_path=None, mean=None, std=None):
    """
    工厂函数：根据模型类型字符串，创建并返回特征提取器、图像变换和输入尺寸。

    Args:
        model_type (str): 模型类型，必须是 SUPPORTED_MODELS 中的键之一
                          如 'ResNeXt', 'AlexNet', 'ViT_B16' 等
        device (torch.device): 模型运行设备，如 torch.device('cuda') 或 'cpu'
        model_path (str): 本地 .pth 权重文件路径
                          若为 None，使用 ImageNet 默认归一化参数
        mean (list): 图像归一化均值，如 [0.485, 0.456, 0.406]
                     若为 None，使用 ImageNet 默认值
        std (list):  图像归一化标准差，如 [0.229, 0.224, 0.225]
                     若为 None，使用 ImageNet 默认值

    Returns:
        model (nn.Module):          已加载权重的特征提取器，已移至 device
        transform (Compose):        图像预处理流程（pad → ToTensor → Normalize）
        input_size (int):           模型期望的输入图像边长（224 或 299）
    """

    # ── 1. 确定归一化参数 ────────────────────────────────────────────────────
    # 归一化的作用：将像素值从 [0, 1] 标准化到均值为0、标准差为1的分布，
    # 使模型训练更稳定，收敛更快。
    #
    # 两种情况使用 ImageNet 默认值：
    #   a) model_path 为 None（没有自定义权重，可能用 ImageNet 预训练权重）
    #   b) mean 或 std 任一为 None（用户没有提供自定义归一化参数）
    if model_path is None or (mean is None and std is None):
        # ImageNet 数据集的统计均值和标准差（RGB三通道）
        # 这是深度学习领域最常用的归一化参数
        mean = [0.485, 0.456, 0.406]
        std  = [0.229, 0.224, 0.225]
    else:
        # 使用调用方传入的自定义归一化参数
        # 注释掉的代码是针对特定病理图像数据集计算的统计值，
        # 可以根据实际数据集替换
        # mean = [0.8243, 0.8675, 0.9146]
        # std  = [0.2708, 0.2159, 0.1419]
        mean = mean  # 直接使用传入值（这两行赋值看似多余，但保留了显式性）
        std  = std

    # ── 2. 查询模型注册表，验证模型类型 ─────────────────────────────────────
    if model_type not in SUPPORTED_MODELS:
        # 提供清晰的错误信息，列出所有支持的模型类型
        raise ValueError(
            f"不支持的模型类型: {model_type}. "
            f"支持的模型类型: {list(SUPPORTED_MODELS.keys())}"
        )

    # 从注册表中取出该模型的配置信息
    model_info  = SUPPORTED_MODELS[model_type]
    model_class = model_info['class']       # 特征提取器类（如 CustomResNeXtFeatureExtractor）
    input_size  = model_info['input_size']  # 输入尺寸（224 或 299）
    pad_func    = model_info['pad_func']    # 图像填充函数

    # ── 3. 实例化模型并移至目标设备 ─────────────────────────────────────────
    # model_class(model_path=model_path)：
    #   调用对应类的 __init__，内部会执行四步模式（初始化→修改结构→加载权重→替换Identity）
    model = model_class(model_path=model_path)

    # 将模型参数和缓冲区移至指定设备（CPU 或 GPU）
    # 推理时输入张量也需要在同一设备上
    model = model.to(device)

    # ── 4. 构建图像预处理流程 ────────────────────────────────────────────────
    # transforms.Compose：将多个变换操作串联成一个流水线，
    # 每个变换的输出作为下一个变换的输入
    transform = transforms.Compose([
        # 变换1：填充并缩放图像到目标尺寸
        # transforms.Lambda 将普通函数包装成 transform 对象
        # pad_func 是 pad_to_square_transform 或 pad_to_square_299_transform
        # 输入：PIL.Image（任意尺寸）
        # 输出：PIL.Image（224×224 或 299×299，正方形，白色填充）
        transforms.Lambda(pad_func),

        # 变换2：将 PIL.Image 转换为 PyTorch 张量
        # 同时将像素值从 [0, 255] 归一化到 [0.0, 1.0]
        # 输入：PIL.Image（H×W×C，uint8）
        # 输出：torch.Tensor（C×H×W，float32，值域 [0,1]）
        # 注意通道顺序从 HWC 变为 CHW（PyTorch 的标准格式）
        transforms.ToTensor(),

        # 变换3：标准化（Z-score 归一化）
        # 对每个通道：output = (input - mean) / std
        # 输入：torch.Tensor（C×H×W，值域 [0,1]）
        # 输出：torch.Tensor（C×H×W，值域约 [-2.5, 2.5]）
        # 目的：使输入分布与模型训练时的数据分布一致
        transforms.Normalize(mean=mean, std=std)
    ])

    # 返回三元组：模型、预处理流程、输入尺寸
    return model, transform, input_size

## models/feature_extractor.py 提取模型特征向量的函数

In [ ]:
# =============================================================================
# models/feature_extractor.py
# 职责：调用 models.py 中定义的特征提取器，对一批细胞图像进行推理，
#       并将深度特征与细胞形态特征（大小）融合，输出最终的特征矩阵。
#
# ★ 本文件只有一个核心函数：extract_features_with_cell_size() ★
#
# 【输入】
#   - 一个已加载权重的特征提取模型（来自 models.py）
#   - 一个 DataLoader（来自 data_utils.py 的 CellImageDataset）
#   - 每张图像对应的细胞大小（非背景像素数）
#
# 【输出】
#   - 融合了细胞大小信息的特征矩阵，形状 [N, D+1]
#     N = 图像总数，D = 模型输出的深度特征维度，+1 = 细胞大小特征
#   - 每张图像的 source_label（来自哪个数据集/类别）
#   - 每张图像的文件路径列表
#
# 【为什么要加入细胞大小特征？】
#   深度特征主要捕捉细胞的纹理、形态等视觉信息，
#   但不同类型的肿瘤细胞在大小上也有显著差异。
#   将细胞大小作为一个额外特征维度，可以帮助聚类算法
#   更好地区分不同类型的细胞群体。
# =============================================================================

import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from PIL import Image
import glob
import os
from tqdm import tqdm  # 进度条库，用于显示特征提取的进度


def extract_features_with_cell_size(
    model,
    dataloader,
    device,
    cell_sizes,
    feature_layer='penultimate',
    cell_size_weight=1.0
):
    """
    从模型中提取深度特征，并将细胞大小信息融合到特征向量末尾。

    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    完整处理流程：

      [DataLoader] → 按 batch 送入模型
          ↓
      [模型推理] → 深度特征向量 [N, D]
          ↓
      [细胞大小预处理]
          原始像素数 → log1p变换 → Min-Max归一化 → [N, 1]
          ↓
      [特征融合] → hstack → [N, D+1]
          ↓
      返回 (features_with_size, source_labels, image_paths)
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    Args:
        model:             特征提取模型（来自 models.py，继承自 nn.Module）
        dataloader:        PyTorch DataLoader，每个 batch 返回
                           (images, source_labels, indices) 三元组
        device:            运行设备，torch.device('cuda') 或 torch.device('cpu')
        cell_sizes:        长度为 N 的列表/数组，cell_sizes[i] 是第 i 张图像的
                           非背景像素数（由 data_utils.py 中的
                           count_non_background_pixels() 计算）
        feature_layer:     传给 model.forward() 的 layer 参数，
                           'penultimate' 表示取倒数第二层特征（默认）
        cell_size_weight:  细胞大小特征的缩放权重（默认 1.0）
                           增大此值 → 聚类时更重视细胞大小差异
                           减小此值 → 聚类时更重视深度视觉特征

    Returns:
        features_with_size (np.ndarray): 形状 [N, D+1]
                           每行是一张图像的完整特征向量
                           前 D 维是深度特征，最后 1 维是归一化后的细胞大小
        source_labels (np.ndarray):      形状 [N,]，每个元素是整数标签
                           0=LUAD参考, 1=LUSC参考, 2=SCLC参考,
                           3=正常参考, 4=其他参考, 5=恶性参考,
                           6=候选域图像（具体编码见 data_utils.py）
        image_paths (list of str):       长度为 N 的列表，每个元素是图像文件的绝对路径
    """

    # ── 第一步：设置模型为推理模式 ────────────────────────────────────────────
    # model.eval() 的作用：
    #   1. 禁用 Dropout（Dropout 在训练时随机丢弃神经元，推理时应关闭）
    #   2. 禁用 BatchNorm 的统计量更新（推理时使用训练阶段统计的均值和方差）
    #   3. 对于 GoogLeNet，禁用辅助分类器的随机行为
    # 如果不调用 eval()，同一张图像每次提取的特征可能不同，导致聚类结果不稳定
    model.eval()

    # 用于收集各 batch 结果的临时列表
    features_list = []      # 存储每个 batch 的深度特征矩阵
    source_labels_list = [] # 存储每个 batch 的 source_label 数组
    indices_list = []       # 存储每个 batch 的全局索引数组
    # 注意：使用列表收集后再 vstack/concatenate，
    # 比每次 append 到大数组效率更高（避免频繁内存分配）

    # ── 第二步：遍历 DataLoader，逐 batch 提取深度特征 ───────────────────────
    # torch.no_grad()：上下文管理器，在其内部禁用梯度计算
    # 作用：
    #   1. 节省显存（不需要存储中间激活值用于反向传播）
    #   2. 加速推理（跳过梯度相关计算）
    # 推理时永远应该使用 torch.no_grad()
    with torch.no_grad():

        # tqdm 包装 dataloader，显示进度条
        # desc="提取特征" 是进度条左侧的描述文字
        for batch_images, batch_labels, batch_indices in tqdm(dataloader, desc="提取特征"):

            # ── 2a. 将图像张量移至目标设备 ──────────────────────────────────
            # batch_images 从 DataLoader 出来时在 CPU 上
            # 必须移到与模型相同的设备才能进行计算
            # 形状：[batch_size, 3, H, W]，H=W=224（或299）
            batch_images = batch_images.to(device)

            # ── 2b. 前向传播，提取深度特征 ──────────────────────────────────
            # model(batch_images, layer=feature_layer) 调用对应模型的 forward()
            # feature_layer='penultimate' 时，返回倒数第二层特征
            # 对于大多数模型，输出形状为 [batch_size, D]（2D 张量）
            # 但某些模型可能返回 4D 张量（见下方处理）
            batch_features = model(batch_images, layer=feature_layer)

            # ── 2c. 处理 4D 特征张量（兜底逻辑）───────────────────────────
            # 正常情况下，models.py 中的所有模型都已在 forward() 中完成展平，
            # 输出 2D 张量 [batch_size, D]。
            # 但作为防御性编程，这里检查是否有模型返回了 4D 特征图
            # [batch_size, channels, height, width]。
            # 如果是 4D，用全局平均池化压缩空间维度：
            #   [B, C, H, W] → mean(dim=(2,3)) → [B, C]
            # dim=(2, 3) 表示同时对第 2 维（H）和第 3 维（W）取平均
            if len(batch_features.shape) == 4:
                batch_features = torch.mean(batch_features, dim=(2, 3))
                # 等价于全局平均池化（Global Average Pooling）
                # 每个通道的所有空间位置取平均，得到该通道的代表值

            # ── 2d. 将特征从 GPU 移回 CPU，并转换为 NumPy 数组 ─────────────
            # .cpu()：将张量从 GPU 移回 CPU（如果本来就在 CPU 则无操作）
            # .numpy()：将 PyTorch 张量转换为 NumPy 数组
            # 注意：.numpy() 只能在 CPU 张量上调用，且要求不需要梯度
            # （torch.no_grad() 已保证不需要梯度）
            batch_features = batch_features.cpu().numpy()
            # 此时 batch_features 形状：[batch_size, D]，dtype=float32

            # 收集本 batch 的结果
            features_list.append(batch_features)
            # batch_labels 是 CPU 张量（DataLoader 默认不移到 GPU），直接转 numpy
            source_labels_list.append(batch_labels.numpy())
            # batch_indices 是每张图像在 dataset 中的全局索引，用于后续查找路径和大小
            indices_list.append(batch_indices.numpy())

    # ── 第三步：合并所有 batch 的结果 ────────────────────────────────────────
    # np.vstack：垂直堆叠（沿第 0 轴），用于合并 2D 数组
    # 例：[array([B1, D]), array([B2, D]), ...] → array([N, D])
    features = np.vstack(features_list)
    # features 形状：[N, D]，N = 数据集总图像数

    # np.concatenate：沿第 0 轴拼接 1D 数组
    # 例：[array([B1,]), array([B2,]), ...] → array([N,])
    source_labels = np.concatenate(source_labels_list)
    # source_labels 形状：[N,]

    indices = np.concatenate(indices_list)
    # indices 形状：[N,]，每个元素是该图像在 dataset 中的原始索引

    # ── 第四步：按索引还原图像路径 ───────────────────────────────────────────
    # DataLoader 使用多线程加载数据，batch 的顺序可能与 dataset 的顺序不完全一致
    # （尤其是 shuffle=True 时）。
    # 因此不能直接用 range(N) 作为索引，必须用 DataLoader 返回的 batch_indices
    # 来还原每张图像在 dataset 中的原始位置，从而获取正确的路径。
    #
    # dataloader.dataset 是底层的 CellImageDataset 对象
    # .image_paths 是 dataset 中存储的所有图像路径列表
    # indices[i] 是第 i 张图像在 dataset 中的原始索引
    image_paths = [dataloader.dataset.image_paths[idx] for idx in indices]
    # image_paths 是长度为 N 的字符串列表

    # ── 第五步：按索引提取对应的细胞大小 ────────────────────────────────────
    # cell_sizes 是整个 dataset 的细胞大小列表（长度 = dataset 总大小）
    # 需要用 indices 来取出与当前特征矩阵对应的那些大小
    batch_cell_sizes = np.array([cell_sizes[idx] for idx in indices])
    # batch_cell_sizes 形状：[N,]，每个元素是对应图像的非背景像素数（正整数）

    # ── 第六步：对细胞大小进行对数变换 ──────────────────────────────────────
    # 为什么要做对数变换？
    #
    # 问题：细胞大小（像素数）的分布通常是右偏的（少数细胞非常大），
    #       例如：大多数细胞 500~2000 像素，少数细胞可达 10000+ 像素。
    #       如果直接使用原始像素数，大细胞的特征值会远大于小细胞，
    #       主导聚类结果，使聚类对大小差异过于敏感。
    #
    # 解决方案：np.log1p(x) = log(1 + x)
    #   - 压缩大值的范围（如 10000 → log(10001) ≈ 9.2）
    #   - 保持小值的相对差异（如 100 → log(101) ≈ 4.6，200 → log(201) ≈ 5.3）
    #   - log1p 而非 log 的原因：当 x=0 时，log(0) 是负无穷，
    #     而 log1p(0) = log(1) = 0，避免数值问题
    log_cell_sizes = np.log1p(batch_cell_sizes)
    # log_cell_sizes 形状：[N,]，值域约为 [0, 10]（取决于细胞大小范围）

    # ── 第七步：Min-Max 归一化 ────────────────────────────────────────────────
    # 为什么要归一化？
    #   深度特征向量的各维度通常在 [-1, 1] 或 [-3, 3] 范围内（经过归一化的网络输出）。
    #   如果细胞大小特征的数值范围与深度特征差异太大，
    #   会导致细胞大小特征在欧氏距离计算中占比失衡。
    #   归一化到 [0, 1] 后，cell_size_weight 参数才能有意义地控制其权重。
    #
    # Min-Max 归一化公式：
    #   normalized = (x - min) / (max - min)
    #   将值域从 [min, max] 线性映射到 [0, 1]
    #
    # 特殊情况处理：
    #   如果所有细胞大小完全相同（max == min），分母为 0 会导致除零错误。
    #   此时细胞大小无法提供区分信息，将所有值设为 0。
    if np.max(log_cell_sizes) > np.min(log_cell_sizes):
        # 正常情况：细胞大小有差异，进行归一化
        normalized_log_cell_sizes = (
            (log_cell_sizes - np.min(log_cell_sizes)) /
            (np.max(log_cell_sizes) - np.min(log_cell_sizes))
        )
        # normalized_log_cell_sizes 值域：[0.0, 1.0]
    else:
        # 边界情况：所有细胞大小完全相同（极少发生）
        # np.zeros_like 创建与输入形状相同、全为 0 的数组
        normalized_log_cell_sizes = np.zeros_like(log_cell_sizes)

    # ── 第八步：构造细胞大小特征列，并与深度特征拼接 ─────────────────────────
    # reshape(-1, 1)：将 1D 数组 [N,] 变形为 2D 列向量 [N, 1]
    # 这是为了与 features [N, D] 进行水平拼接（hstack 要求维度兼容）
    #
    # cell_size_weight 的作用：
    #   乘以权重因子后，细胞大小特征的值域变为 [0, cell_size_weight]
    #   当 cell_size_weight=1.0 时，细胞大小特征与深度特征的各维度量级相当
    #   当 cell_size_weight=2.0 时，细胞大小特征的影响力翻倍
    #   当 cell_size_weight=0.5 时，细胞大小特征的影响力减半
    cell_size_features = cell_size_weight * normalized_log_cell_sizes.reshape(-1, 1)
    # cell_size_features 形状：[N, 1]，值域：[0, cell_size_weight]

    # np.hstack：水平拼接（沿第 1 轴），将两个 2D 数组左右合并
    # [N, D] 和 [N, 1] → [N, D+1]
    # 细胞大小特征被追加到深度特征的最后一列
    features_with_size = np.hstack((features, cell_size_features))
    # features_with_size 形状：[N, D+1]
    # 列含义：[深度特征维0, 深度特征维1, ..., 深度特征维D-1, 归一化细胞大小]

    return features_with_size, source_labels, image_paths

## clustering/K_optimizer.py 如何选最优聚类K值

In [ ]:
# =============================================================================
# clustering/K_optimizer.py
# 职责：为候选域（candidate domain）的 K-Means 聚类自动寻找最优的簇数 K。
#
# ★ 核心问题：K-Means 需要预先指定 K，但我们不知道候选细胞有几种亚型 ★
#
# 【解决思路】
#   对 K=2,4,6,...,max_k 逐一尝试，每个 K 都做一次聚类，
#   然后用一个"加权评分"来衡量该 K 下聚类结果的质量，
#   最终选择评分最高的 K 作为最优解。
#
# 【评分的核心思想：分离度 × 一致性】
#   好的聚类应该满足两个条件：
#   1. 每个候选簇与恶性/良性参考簇之间有明显的距离差异（分离度 Ssep）
#      → 说明该簇的细胞有明确的"倾向性"（偏恶性或偏良性）
#   2. 同一候选簇内的细胞，与簇中心"投票"结果一致（一致性 P_C）
#      → 说明该簇内部的细胞确实是同一类型，不是强行混在一起的
#
# 【参考域 vs 候选域】
#   参考域（reference）：已知类型的细胞（LUAD/LUSC/SCLC/正常等），
#                        已经聚类完成，作为"标准答案"
#   候选域（candidate）：待分析的未知细胞，需要找到最优 K 进行聚类
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity  # 计算余弦相似度
import shutil


# =============================================================================
# 辅助函数：绘制 K 值优化曲线
# =============================================================================
def plot_k_optimization_curve(k_values, scores, best_k, patient_id, save_dir):
    """
    绘制不同 K 值对应的加权评分曲线，并标记最优 K 值。

    Args:
        k_values (list): 尝试过的所有 K 值，如 [2, 4, 6, 8, ...]
        scores (list):   对应每个 K 值的加权评分
        best_k (int):    评分最高的 K 值
        patient_id (str): 病理切片 ID，用于图表标题
        save_dir (str):  图片保存目录
    """
    # 创建 10×6 英寸的画布
    plt.figure(figsize=(10, 6))

    # 绘制折线图，'o-' 表示带圆形标记的实线
    plt.plot(k_values, scores, 'o-', linewidth=2, markersize=8)

    # 在最优 K 值处画一条红色虚线，方便直观定位
    plt.axvline(x=best_k, color='r', linestyle='--', label=f'Best k={best_k}')

    # 坐标轴标签和标题
    plt.xlabel('k (Number of Clusters)', fontsize=24, fontweight='bold', labelpad=20)
    plt.ylabel('Weighted Score',          fontsize=24, fontweight='bold', labelpad=20)
    plt.title(f'{patient_id} K-Optimization', fontsize=28, fontweight='bold', pad=20)

    # 背景网格（虚线，透明度 0.7）
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=16)

    # margins(0.1)：在数据范围的基础上，四周各留 10% 的空白，防止数据点贴边
    plt.margins(0.1)

    # 调整子图边距，防止标签被截断
    plt.subplots_adjust(left=0.15, right=0.85, top=0.85, bottom=0.15)

    # 保存为 300 DPI 的高清图片
    save_path = os.path.join(save_dir, 'k_optimization_plot.png')
    plt.savefig(save_path, dpi=300)

    # 关闭画布，释放内存（不关闭会导致内存泄漏，尤其在批量处理多个病人时）
    plt.close()



### optimize_k_for_candidate_clustering  为候选域聚类寻找最优 K 值

In [ ]:

# =============================================================================
# 核心函数：为候选域聚类寻找最优 K 值
# =============================================================================
def optimize_k_for_candidate_clustering(
    patient_candidate_normalized,       # 候选域特征矩阵（已归一化），shape [N_cand, D+1]
    patient_candidate_source_labels,    # 候选域每个细胞的来源标签，shape [N_cand,]
    patient_candidate_image_paths,      # 候选域每个细胞的图像路径，list of str
    reference_results,                  # 参考域聚类结果字典（见下方说明）
    patient_save_dir,                   # 当前病人的结果保存根目录
    patient_id,                         # 病理切片 ID，如 'P001'
    max_k=20                            # 最大尝试的 K 值（默认 20）
):
    """
    通过遍历多个 K 值，计算每个 K 的"加权分离度评分"，选出最优 K。

    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    reference_results 字典的结构（由上游聚类步骤生成）：
      {
        'cluster_centers': np.ndarray [n_ref_clusters, D+1],  # 参考簇中心坐标
        'n_clusters':      int,                               # 参考簇总数
        'labels':          np.ndarray [N_ref,],               # 参考域每个细胞的簇标签
        ...
      }

    返回值：
      best_candidate_results (dict): 最优 K 下的聚类结果字典
      best_k (int):                  最优 K 值
      若所有 K 值都失败且 k=2 也失败，返回 (None, None)
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """
    print(f"  候选细胞数量为 {len(patient_candidate_normalized)}，开始优化k值...")

    # ── 准备工作：创建目录结构 ────────────────────────────────────────────────
    # k_optimization/
    # ├── k2/   ← 每个 K 值的中间结果单独存放
    # ├── k4/
    # ├── k6/
    # ├── k_optimization_results.csv   ← 所有 K 值的评分汇总
    # └── k_optimization_plot.png      ← 评分曲线图
    k_optimization_dir = os.path.join(patient_save_dir, 'k_optimization')
    os.makedirs(k_optimization_dir, exist_ok=True)
    # exist_ok=True：目录已存在时不报错，直接使用

    # 用于收集 (k, 加权评分) 元组的列表
    k_scores = []

    # ── 确定 K 值搜索范围 ────────────────────────────────────────────────────
    # 上限取 max_k 和"样本数//3"中的较小值。
    # 原因：K-Means 要求每个簇至少有几个样本才有意义。
    # 如果 K 过大（接近样本数），每个簇只有 1~2 个细胞，
    # 聚类结果没有统计意义，且容易出现空簇。
    # 用样本数//3 作为上限，保证平均每个簇至少有 3 个样本。
    max_k = min(max_k, len(patient_candidate_normalized) // 3)

    # range(2, max_k+1, 2)：从 2 开始，步长为 2，即只尝试偶数 K
    # 原因：奇数 K 和偶数 K 的评分差异通常不大，步长为 2 可以减少一半计算量
    k_values = range(2, max_k + 1, 2)

    # ── 定义恶性和良性参考簇的索引 ───────────────────────────────────────────
    # 参考域的聚类结果中，簇 0 和簇 5 被定义为"恶性参考簇"：
    #   簇 0 (R0)：来自恶性肿瘤样本（如 LUAD/LUSC/SCLC）的细胞聚成的簇
    #   簇 5 (R5)：来自恶性标注样本的细胞聚成的簇
    # 其余簇（1,2,3,4,...）为"良性参考簇"（正常细胞、良性细胞等）
    malignant_ref_clusters = [0, 5]
    benign_ref_clusters = [
        i for i in range(reference_results['n_clusters'])
        if i not in malignant_ref_clusters
    ]
    # 例如：若 n_clusters=8，则 benign_ref_clusters = [1, 2, 3, 4, 6, 7]

    # ── 预计算：每个候选细胞与所有参考簇中心的余弦相似度 ─────────────────────
    # 余弦相似度衡量两个向量方向的相似程度，值域 [-1, 1]，越接近 1 越相似。
    # 与欧氏距离相比，余弦相似度对特征向量的模长不敏感，
    # 更适合高维特征空间（如 2048 维的深度特征）。
    reference_centers = reference_results['cluster_centers']
    # reference_centers 形状：[n_ref_clusters, D+1]

    # cosine_similarity(A, B)：计算 A 中每行与 B 中每行的余弦相似度
    # 输入：[N_cand, D+1] 和 [n_ref_clusters, D+1]
    # 输出：[N_cand, n_ref_clusters]，第 i 行第 j 列 = 第 i 个候选细胞与第 j 个参考簇中心的相似度
    cell_similarity_matrix = cosine_similarity(patient_candidate_normalized, reference_centers)

    # 将相似度转换为距离（1 - 相似度），然后找每个细胞距离最近的参考簇
    # np.argmin(distance_matrix, axis=1)：对每行取最小值的列索引
    # 即：对每个候选细胞，找出与它最相似（距离最近）的参考簇编号
    cell_nearest_ref_cluster = np.argmin(1 - cell_similarity_matrix, axis=1)
    # cell_nearest_ref_cluster 形状：[N_cand,]
    # 例如：[0, 5, 2, 0, 3, ...]，表示第1个细胞最近参考簇是簇0，第2个是簇5...

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 主循环：遍历每个候选 K 值，计算加权评分
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    for k in k_values:
        print(f"  尝试k值: {k}")

        try:
            # 为当前 K 值创建独立的保存目录
            k_save_dir = os.path.join(k_optimization_dir, f'k{k}')
            os.makedirs(k_save_dir, exist_ok=True)

            # 下面几个部分的计算不涉及到参考域
            # ── Step A：对候选域进行 K-Means 聚类 ────────────────────────────
            # n_clusters=k：将候选细胞分成 k 个簇
            # random_state=42：固定随机种子，保证结果可复现
            # max_iter=300：最多迭代 300 次（默认 300，防止不收敛时无限循环）
            patient_candidate_kmeans = KMeans(n_clusters=k, random_state=42, max_iter=300)

            # fit_predict：先拟合（计算簇中心），再预测（给每个样本分配簇标签）
            # 返回值：每个样本的簇标签，值域 [0, k-1]
            patient_candidate_labels = patient_candidate_kmeans.fit_predict(
                patient_candidate_normalized
            )
            # patient_candidate_labels 形状：[N_cand,]
            # 例如：[0, 1, 0, 2, 1, 0, ...] 表示第1个细胞属于簇0，第2个属于簇1...

            # ── Step B：检查空簇 ──────────────────────────────────────────────
            # np.bincount：统计每个标签出现的次数
            # minlength=k：确保输出长度为 k（即使某些标签没有出现，也显示为 0）
            cluster_counts = np.bincount(patient_candidate_labels, minlength=k)
            # 例如 k=4 时：cluster_counts = [15, 0, 8, 12]，簇1是空簇

            if np.any(cluster_counts == 0):
                # 空簇意味着 K-Means 初始化不好，某些簇中心没有吸引到任何样本
                # 空簇的评分没有意义，跳过此 K 值
                print(f"  警告: k={k} 时出现空簇，跳过此k值")
                continue

            # ── Step C：计算每个候选细胞到其所属簇中心的欧氏距离 ─────────────
            # patient_candidate_kmeans.cluster_centers_：形状 [k, D+1]，所有簇中心
            # patient_candidate_kmeans.cluster_centers_[patient_candidate_labels]：
            #   利用"花式索引"，为每个细胞取出其所属簇的中心坐标
            #   形状：[N_cand, D+1]（每行是该细胞所属簇的中心）
            # patient_candidate_normalized - centers：每个细胞与其簇中心的差向量
            # np.linalg.norm(..., axis=1)：对每行（每个细胞）计算 L2 范数（欧氏距离）
            patient_candidate_distances = np.linalg.norm(
                patient_candidate_normalized
                - patient_candidate_kmeans.cluster_centers_[patient_candidate_labels],
                axis=1
            )
            # patient_candidate_distances 形状：[N_cand,]
            # 每个值表示该细胞距其所属簇中心的距离，越小说明该细胞越"典型"

            # ── Step D：构建当前 K 值的候选域结果字典 ────────────────────────
            # 这个字典格式与 reference_results 保持一致，便于后续统一处理
            current_candidate_results = {
                'features':       patient_candidate_normalized,     # 特征矩阵 [N_cand, D+1]
                'labels':         patient_candidate_labels,         # 簇标签 [N_cand,]
                'distances':      patient_candidate_distances,      # 到簇中心的距离 [N_cand,]
                'source_labels':  patient_candidate_source_labels,  # 来源标签 [N_cand,]
                'image_paths':    patient_candidate_image_paths,    # 图像路径列表
                'kmeans':         patient_candidate_kmeans,         # KMeans 对象（含簇中心等）
                'cluster_centers': patient_candidate_kmeans.cluster_centers_,  # [k, D+1]
                'n_clusters':     k                                 # 当前 K 值
            }

            # 开始将训练出来的候选域结构与参考域关联
            # ── Step E：计算候选簇中心与参考簇中心之间的相似度矩阵 ───────────
            candidate_centers = current_candidate_results['cluster_centers']
            # candidate_centers 形状：[k, D+1]

            # cosine_similarity(A, B)：
            # 输入：[k, D+1] 和 [n_ref_clusters, D+1]
            # 输出：[k, n_ref_clusters]
            # similarity_matrix[i][j] = 候选簇 i 与参考簇 j 的余弦相似度
            similarity_matrix = cosine_similarity(candidate_centers, reference_centers)

            # 距离矩阵 = 1 - 相似度（相似度越高，距离越小）
            # distance_matrix[i][j] = 候选簇 i 与参考簇 j 的余弦距离，值域 [0, 2]
            distance_matrix = 1 - similarity_matrix

            # 对每个候选簇，找出距离最近（最相似）的参考簇编号
            # np.argmin(distance_matrix, axis=1)：对每行取最小值的列索引
            cluster_nearest_ref = np.argmin(distance_matrix, axis=1)
            # cluster_nearest_ref 形状：[k,]
            # 例如：[0, 3, 5, 1] 表示候选簇0最近参考簇是R0，候选簇1最近是R3...

            # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            # Step F：对每个候选簇计算"分离度 Ssep"和"一致性 P_C"
            # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
            cluster_scores = []
            total_cells = len(patient_candidate_normalized)  # 候选细胞总数

            for cand_idx in range(k):
                # 获取该簇的细胞数量（用于后续计算，虽然此处未直接用到）
                cluster_mask = (patient_candidate_labels == cand_idx)
                cluster_size = np.sum(cluster_mask)

                # ── 指标 a：Dm（候选簇到最近恶性参考簇的距离）────────────────
                # 遍历所有恶性参考簇，取距离最小的那个
                malignant_distances = [
                    distance_matrix[cand_idx, ref_idx]
                    for ref_idx in malignant_ref_clusters
                ]
                Dm = min(malignant_distances)
                # Dm 越小 → 候选簇越靠近恶性参考簇 → 该簇细胞可能是恶性的

                # ── 指标 b：Db（候选簇到最近良性参考簇的距离）────────────────
                benign_distances = [
                    distance_matrix[cand_idx, ref_idx]
                    for ref_idx in benign_ref_clusters
                ]
                # 若没有良性参考簇（极端情况），默认距离为 1.0（中等距离）
                Db = min(benign_distances) if benign_distances else 1.0
                # Db 越小 → 候选簇越靠近良性参考簇 → 该簇细胞可能是良性的

                # ── 指标 c：Ssep（归一化分离度）──────────────────────────────
                # 公式：Ssep = |Dm - Db| / (Dm + Db + ε)
                #
                # 直觉理解：
                #   如果 Dm << Db（候选簇远离良性、靠近恶性）：
                #     |Dm - Db| ≈ Db，分子大 → Ssep 接近 1（高分离度，倾向恶性）
                #   如果 Dm ≈ Db（候选簇到恶性和良性距离相当）：
                #     |Dm - Db| ≈ 0，分子小 → Ssep 接近 0（低分离度，无明确倾向）
                #   如果 Db << Dm（候选簇远离恶性、靠近良性）：
                #     |Dm - Db| ≈ Dm，分子大 → Ssep 接近 1（高分离度，倾向良性）
                #
                # 注意：Ssep 只衡量"有没有明确倾向"，不区分倾向恶性还是良性
                # 这是合理的：无论倾向哪边，只要有明确倾向，说明该簇是"纯净"的
                epsilon = 1e-10  # 防止 Dm + Db = 0 时除零（极端情况）
                Ssep = abs((Dm - Db) / (Dm + Db + epsilon))
                # Ssep 值域：[0, 1]，越接近 1 说明该候选簇的倾向越明确

                # ── 指标 d：P_C（簇内细胞与簇中心的"投票一致性"）─────────────
                # 思路：
                #   - 簇中心最近的参考簇（cluster_nearest_ref[cand_idx]）代表该簇的"整体倾向"
                #   - 每个细胞最近的参考簇（cell_nearest_ref_cluster[cluster_mask]）代表该细胞的"个体倾向"
                #   - 如果大多数细胞的个体倾向与簇的整体倾向一致，说明该簇内部纯净
                #
                # 获取该候选簇内所有细胞的"个体最近参考簇"
                cluster_cells_mask = (patient_candidate_labels == cand_idx)
                cluster_cells_nearest_ref = cell_nearest_ref_cluster[cluster_cells_mask]
                # 形状：[cluster_size,]，每个值是该细胞最近的参考簇编号

                # 该候选簇中心最近的参考簇（整体倾向）
                cluster_nearest_ref_cluster = cluster_nearest_ref[cand_idx]

                # 统计该簇内"个体倾向"与"整体倾向"一致的细胞数量
                matching_cells_count = np.sum(
                    cluster_cells_nearest_ref == cluster_nearest_ref_cluster
                )
                # 例如：簇中心最近参考簇是 R0（恶性），
                # 簇内 10 个细胞中有 8 个也最近 R0，则 matching_cells_count=8

                # P_C = 一致细胞数 / 所有候选细胞总数
                # 注意：分母是 total_cells（全部候选细胞），而非 cluster_size（该簇细胞）
                # 这样大簇的 P_C 自然更大，在加权时占更大权重
                # 体现了"大簇的一致性比小簇更重要"的思想
                P_C = matching_cells_count / total_cells

                # 保存该候选簇的三个指标
                cluster_scores.append((Ssep, cluster_size, P_C))
                # cluster_scores[i] = (Ssep_i, size_i, P_C_i)

            # ── Step G：计算该 K 值的总加权评分 ──────────────────────────────
            # 加权平均公式：
            #   weighted_score = Σ(Ssep_i × P_C_i) / Σ(P_C_i)
            #
            # 直觉理解：
            #   - 用 P_C_i 作为权重，对各簇的分离度 Ssep_i 做加权平均
            #   - P_C_i 大的簇（大簇且内部一致）在评分中占更大比重
            #   - 这样可以防止：少数几个小簇恰好有高 Ssep，但大多数细胞所在的簇分离度很低
            #
            # 好的 K 值应该让大多数细胞都落在"倾向明确"的簇里，
            # 而不是只有少数细胞的簇倾向明确
            total_P_C = sum(P_C for _, _, P_C in cluster_scores)

            if total_P_C > 0:
                # 标准加权平均
                weighted_score = sum(
                    Ssep * P_C for Ssep, _, P_C in cluster_scores
                ) / total_P_C
            else:
                # 极端情况：所有 P_C 都为 0（不应发生，但作为防御性处理）
                weighted_score = 0

            print(f"  k={k} 的加权评分: {weighted_score:.4f}")
            k_scores.append((k, weighted_score))

            # 保存当前 K 值的中间结果（.npy 格式，便于后续快速加载）
            np.save(
                os.path.join(k_save_dir, 'candidate_cluster_centers.npy'),
                candidate_centers
            )
            np.save(
                os.path.join(k_save_dir, 'candidate_labels.npy'),
                patient_candidate_labels
            )
            np.save(
                os.path.join(k_save_dir, 'cluster_scores.npy'),
                np.array(cluster_scores)
            )

        except Exception as e:
            # 捕获任何异常（如数值错误、内存不足等），跳过当前 K 值继续尝试
            print(f"  尝试k={k}时出错: {str(e)}，跳过此k值")
            continue

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 主循环结束后：选出最优 K 并返回结果
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    if not k_scores:
        # ── 情况一：所有 K 值都失败（极端情况）────────────────────────────────
        # 可能原因：样本数太少（max_k 被压缩到 < 2），或数据有严重问题
        print("  警告: 所有k值尝试都失败，将使用默认k=2")

        try:
            # 最后的兜底方案：强制使用 k=2
            k = 2
            k_save_dir = os.path.join(k_optimization_dir, f'k{k}')
            os.makedirs(k_save_dir, exist_ok=True)

            patient_candidate_kmeans = KMeans(n_clusters=k, random_state=42, max_iter=300)
            patient_candidate_labels = patient_candidate_kmeans.fit_predict(
                patient_candidate_normalized
            )

            patient_candidate_distances = np.linalg.norm(
                patient_candidate_normalized
                - patient_candidate_kmeans.cluster_centers_[patient_candidate_labels],
                axis=1
            )

            best_candidate_results = {
                'features':        patient_candidate_normalized,
                'labels':          patient_candidate_labels,
                'distances':       patient_candidate_distances,
                'source_labels':   patient_candidate_source_labels,
                'image_paths':     patient_candidate_image_paths,
                'kmeans':          patient_candidate_kmeans,
                'cluster_centers': patient_candidate_kmeans.cluster_centers_,
                'n_clusters':      k
            }

            best_k = k
            print(f"  使用默认k={k}完成聚类")

        except Exception as e:
            # k=2 也失败，说明数据本身有根本性问题，无法处理
            print(f"  使用默认k=2也失败: {str(e)}，跳过此病理片子")
            return None, None  # 返回 None 告知上游跳过此病人

    else:
        # ── 情况二：正常情况，从有效 K 值中选最优 ─────────────────────────────

        # 按加权评分降序排序，第一个就是最优 K
        # key=lambda x: x[1]：按元组的第二个元素（评分）排序
        # reverse=True：降序（评分高的排前面）
        k_scores.sort(key=lambda x: x[1], reverse=True)
        best_k, best_score = k_scores[0]
        print(f"  优化完成，最佳k值: {best_k}，最佳加权评分: {best_score:.4f}")

        # 保存所有 K 值的评分到 CSV 文件，便于后续分析
        k_df = pd.DataFrame(k_scores, columns=['k', 'weighted_score'])
        k_df.to_csv(
            os.path.join(k_optimization_dir, 'k_optimization_results.csv'),
            index=False
        )

        # 提取 K 值列表和对应评分列表，用于绘图
        k_vals = [x[0] for x in k_scores]
        scores  = [x[1] for x in k_scores]

        # 绘制并保存 K 值优化曲线图
        plot_k_optimization_curve(k_vals, scores, best_k, patient_id, k_optimization_dir)

        # ── 用最优 K 重新做一次聚类（最终结果）──────────────────────────────
        # 为什么要重新聚类？
        # 主循环中每个 K 的聚类结果只保存了 .npy 文件，没有保留完整的 KMeans 对象。
        # 这里重新运行一次，得到完整的结果字典，包含 KMeans 对象本身。
        # （也可以从 .npy 文件恢复，但重新运行更简洁，且 random_state=42 保证结果一致）
        patient_candidate_kmeans = KMeans(n_clusters=best_k, random_state=42, max_iter=300)
        patient_candidate_labels = patient_candidate_kmeans.fit_predict(
            patient_candidate_normalized
        )

        patient_candidate_distances = np.linalg.norm(
            patient_candidate_normalized
            - patient_candidate_kmeans.cluster_centers_[patient_candidate_labels],
            axis=1
        )

        best_candidate_results = {
            'features':        patient_candidate_normalized,
            'labels':          patient_candidate_labels,
            'distances':       patient_candidate_distances,
            'source_labels':   patient_candidate_source_labels,
            'image_paths':     patient_candidate_image_paths,
            'kmeans':          patient_candidate_kmeans,
            'cluster_centers': patient_candidate_kmeans.cluster_centers_,
            'n_clusters':      best_k
        }

    return best_candidate_results, best_k

## clustering/matching_rule_application.py（定义了参考域中恶性簇为0和5）

In [ ]:
# =============================================================================
# clustering/matching_rule_application.py
# 职责：定义"候选簇/候选细胞 与参考簇之间的匹配规则"，并将这些规则应用到
#       候选数据上，筛选出符合特定恶性判别条件的簇或单细胞。
#
# ★ 这个文件本质上是在做"规则筛选（rule-based filtering）" ★
#
# 在前面的流程中，我们已经做完了：
#   1. 用深度模型提取细胞特征
#   2. 用参考域样本建立参考簇（reference clusters）
#   3. 对候选域样本聚类，得到候选簇（candidate clusters）
#
# 现在的问题是：
#   在这些候选簇/候选细胞中，哪些更像"恶性细胞"？
#
# 本文件通过"与参考簇中心的余弦相似度"来判断：
#   - 如果一个候选簇与 R5 很像，并且与其他无关参考簇不像 → 满足规则1
#   - 如果一个候选簇与 R0 很像，并且与其他无关参考簇不像 → 满足规则2
#
# 其中：
#   R0 和 R5 通常被定义为"恶性相关参考簇"
#   其他参考簇视为良性/非目标参考簇
#
# 这个文件包含三个核心函数：
#   ① check_matching_rules()      —— 通用规则检查器
#   ② identify_matching_clusters() —— 对候选簇应用规则1/规则2
#   ③ save_matched_cells()         —— 将满足条件的细胞图像复制保存出来
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import shutil


# =============================================================================
# ① 通用规则检查函数：check_matching_rules()
# =============================================================================
# 这个函数是整个文件中最核心的"逻辑判断模块"。
#
# 它不关心当前是在做规则1还是规则2，
# 只负责根据传入的参数检查：
#   - 目标参考簇的相似度是否足够高
#   - 其他参考簇的相似度是否足够低
#
# 因此它是一个"通用模板函数"，规则1和规则2只是调用时参数不同而已。
# =============================================================================
def check_matching_rules(
    similarities,
    ref_idx_target,
    ref_idx_avoid,
    threshold_target,
    threshold_avoid_others=None,
    threshold_ratio=None
):
    """
    通用规则检查函数：判断一个候选对象（候选簇中心 / 单细胞）是否满足某个匹配规则。

    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    参数说明：

    similarities:
        当前对象与所有参考簇的相似度向量（1D数组）
        例如：
            similarities = [0.41, 0.10, 0.08, 0.05, 0.12, 0.72]
        表示：
            与 R0 相似度 0.41
            与 R1 相似度 0.10
            ...
            与 R5 相似度 0.72

    ref_idx_target:
        目标参考簇索引
        即：当前规则希望"靠近"的参考簇
        例如：
            规则1 中 target = 5（希望更像 R5）
            规则2 中 target = 0（希望更像 R0）

    ref_idx_avoid:
        需要"忽略"的另一个特殊参考簇
        它不会参与"其他簇必须低于阈值"的判断
        例如：
            规则1 以 R5 为目标时，忽略 R0
            规则2 以 R0 为目标时，忽略 R5

        为什么要忽略？
        因为 R0 和 R5 都可能属于"恶性相关簇"，
        当我们希望筛选更像 R5 的对象时，不必因为它也有点像 R0 就立即否决它。

    threshold_target:
        目标参考簇的最低相似度阈值
        若 similarities[target] < threshold_target，则直接判定不满足规则

    threshold_avoid_others:
        对"其他参考簇（非参考恶性簇，也就是参考良性簇）*"的绝对阈值限制
        如果某个其他簇的相似度 >= 这个阈值，则不满足规则。也就是这个簇与其他参考簇的相似度要是低于0.2才行，和threshold_ratio的规则类似，一个是比例而已，一个是固定数值
        例如：
            规则2 要求：其他簇相似度都 < 0.2

    threshold_ratio:
        对"其他参考簇"的相对阈值限制
        如果某个其他簇相似度 >= 目标相似度 / threshold_ratio，则不满足规则。也就是这个簇与目标簇的相似度要是其他参考簇的3倍及以上
        例如：
            规则1 要求：其他簇相似度都 < R5相似度 / 3

    Returns:
        True  -> 满足规则
        False -> 不满足规则
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """

    # ── Step 1：先检查目标参考簇相似度是否达标 ──────────────────────────────
    # 这是最基础的门槛条件。
    # 如果当前对象连目标簇都不够像，就没有必要继续检查其他条件。
    #
    # 例如：
    #   规则1 要求 R5 >= 0.5
    #   若 similarities[5] = 0.42 < 0.5
    #   则直接返回 False
    if similarities[ref_idx_target] < threshold_target:
        return False

    # ── Step 2：构造"其他参考簇"索引列表 ───────────────────────────────────
    # 除了目标簇 ref_idx_target 和被忽略簇 ref_idx_avoid 之外，
    # 剩下的所有参考簇都视为"其他簇"。
    #
    # 举例：
    #   若共有 6 个参考簇 [0,1,2,3,4,5]
    #   规则1：target=5, avoid=0
    #   则 other_indices = [1,2,3,4]
    #
    #   规则2：target=0, avoid=5
    #   则 other_indices = [1,2,3,4]
    other_indices = [
        i for i in range(len(similarities))
        if i != ref_idx_target and i != ref_idx_avoid
    ]

    # ── Step 3：若设置了相对阈值 threshold_ratio，则检查"其他簇不能太像" ──────
    # 规则形式：
    #   其他簇相似度 < 目标簇相似度 / threshold_ratio
    #
    # 例如：
    #   目标簇 R5 相似度 = 0.72
    #   threshold_ratio = 3
    #   则其他簇必须都 < 0.72 / 3 = 0.24
    #
    # 这样做的意义：
    #   不是单纯要求"目标簇高"，还要求"目标簇明显高于其他簇"，
    #   从而保证匹配更有"专一性"。
    #
    # 注意：这里用了 if threshold_ratio: 而不是 if threshold_ratio is not None:
    # 这意味着 threshold_ratio=0 或 None 都不会进入该分支。
    # 一般我们传入的是正数（如 3），所以没问题。
    if threshold_ratio:
        # 计算其他簇允许达到的上限
        limit = similarities[ref_idx_target] / threshold_ratio

        # any(...)：只要有任意一个其他簇相似度 >= limit，就判定失败
        if any(similarities[i] >= limit for i in other_indices):
            return False

    # ── Step 4：若设置了绝对阈值 threshold_avoid_others，则进一步限制其他簇 ──
    # 规则形式：
    #   其他簇相似度都必须 < threshold_avoid_others
    #
    # 例如：
    #   threshold_avoid_others = 0.2
    #   则所有其他簇都必须 < 0.2
    #
    # 这是一种比"相对阈值"更直接的限制方式：
    #   不管目标簇有多高，只要其他簇里有一个 ≥ 0.2，就不通过
    if threshold_avoid_others:
        if any(similarities[i] >= threshold_avoid_others for i in other_indices):
            return False

    # ── Step 5：若以上检查全部通过，则满足规则 ─────────────────────────────
    return True


# =============================================================================
# ② 匹配规则应用函数：identify_matching_clusters()
# =============================================================================
# 这个函数的作用是：
#   对"候选簇中心"逐个计算它们与参考簇中心的相似度，
#   然后判断每个候选簇属于：
#       - 满足规则1
#       - 满足规则2
#       - 都不满足
#
# 输出的 rule1_clusters / rule2_clusters 是候选簇编号列表。
# =============================================================================
def identify_matching_clusters(candidate_centers, reference_centers):
    """
    识别满足规则1和规则2的候选簇。

    Args:
        candidate_centers:
            候选簇中心矩阵，shape = [n_candidate_clusters, D]
            每一行是一个候选簇的中心向量

        reference_centers:
            参考簇中心矩阵，shape = [n_reference_clusters, D]
            每一行是一个参考簇的中心向量

    Returns:
        rule1_clusters:
            满足规则1的候选簇索引列表，如 [0, 3, 5]

        rule2_clusters:
            满足规则2的候选簇索引列表，如 [1, 2]

        similarity_matrix:
            候选簇中心与参考簇中心的余弦相似度矩阵
            shape = [n_candidate_clusters, n_reference_clusters]
    """

    # ── Step 1：计算候选簇中心 vs 参考簇中心的余弦相似度矩阵 ─────────────────
    # cosine_similarity(A, B)：
    #   A: [n_candidate_clusters, D]
    #   B: [n_reference_clusters, D]
    # 输出：
    #   [n_candidate_clusters, n_reference_clusters]
    #
    # similarity_matrix[i][j] 表示：
    #   第 i 个候选簇中心 与 第 j 个参考簇中心 的余弦相似度
    similarity_matrix = cosine_similarity(candidate_centers, reference_centers)

    # 用于保存满足不同规则的候选簇编号
    rule1_clusters = []
    rule2_clusters = []

    # ── Step 2：逐个候选簇进行规则判断 ──────────────────────────────────────
    for i, sims in enumerate(similarity_matrix):
        # enumerate(similarity_matrix)：
        #   i    = 当前候选簇编号
        #   sims = 当前候选簇与所有参考簇的相似度向量
        #
        # 例如：
        #   sims = [0.41, 0.10, 0.08, 0.05, 0.12, 0.72]

        # ────────────────────────────────────────────────────────────────────
        # 规则1：
        #   R5 >= 0.5
        #   且其他簇 < R5 / 3
        #   （忽略 R0，不把 R0 纳入"其他簇"比较）
        #
        # 翻译成人话：
        #   如果一个候选簇非常像 R5，
        #   并且比其它非恶性参考簇明显更像 R5，
        #   那么认为它符合规则1
        # ────────────────────────────────────────────────────────────────────
        if check_matching_rules(
            sims,                  # 当前候选簇与所有参考簇的相似度
            5,                     # ref_idx_target = 5，即目标参考簇 R5
            0,                     # ref_idx_avoid = 0，即忽略 R0
            0.5,                   # threshold_target = 0.5，即 R5 至少要达到 0.5
            threshold_ratio=3      # 其他簇必须都 < R5 / 3
        ):
            rule1_clusters.append(i)

        # ────────────────────────────────────────────────────────────────────
        # 规则2：
        #   R0 >= 0.4
        #   且其他簇 < 0.2
        #   （忽略 R5，不把 R5 纳入"其他簇"比较）
        #
        # 注意：这里用了 elif 而不是 if
        # 表示：
        #   只有在"不满足规则1"时，才检查规则2
        #
        # 也就是说：
        #   规则1优先级更高
        #   如果一个候选簇同时看起来像 R5 和 R0，
        #   那么优先归入规则1，而不会再进入规则2
        # ────────────────────────────────────────────────────────────────────
        elif check_matching_rules(
            sims,                         # 当前候选簇与所有参考簇的相似度
            0,                            # ref_idx_target = 0，即目标参考簇 R0
            5,                            # ref_idx_avoid = 5，即忽略 R5
            0.4,                          # threshold_target = 0.4，即 R0 至少要达到 0.4
            threshold_avoid_others=0.2    # 其他簇都必须 < 0.2
        ):
            rule2_clusters.append(i)

    return rule1_clusters, rule2_clusters, similarity_matrix


# =============================================================================
# ③ 保存满足匹配规则的细胞图像：save_matched_cells()
# =============================================================================
# 这个函数是"结果落地"函数。
#
# 前两个函数负责"判断哪些簇/哪些细胞满足规则"，
# 这个函数负责把这些符合条件的细胞图像真正复制到指定文件夹中。
#
# 它既可以用于：
#   - 单细胞过滤模式（single-cell filtering）
#   - 聚类模式（cluster-based filtering）
#
# 两种模式的区别：
#   1. 单细胞模式：
#      labels 全部为 0（因为没有聚类，每个细胞都视作同一组）
#
#   2. 聚类模式：
#      labels 是聚类标签（如 0,1,2,...），用于标记该细胞属于哪个候选簇
#
# 保存时，会把规则类型、簇标签、最相似参考簇编号都编码到文件名里。
# =============================================================================
def save_matched_cells(
    indices,
    image_paths,
    labels,               # 单细胞模式下通常全为0；聚类模式下为真实聚类标签
    similarity_matrix,
    rule_type,            # 'rule1' 或 'rule2'
    save_dirs,            # {'rule_dir': ..., 'total_dir': ...}
    target_ref_indices    # 允许保存的目标参考簇索引，如 [0, 5]
):
    """
    保存满足匹配规则的细胞图像。

    Args:
        indices:
            待处理细胞的索引列表
            注意：这些索引通常是"局部索引"，
            即指向 image_paths / labels / similarity_matrix 中的位置

        image_paths:
            每个细胞图像对应的原始文件路径列表
            例如：
                ['/data/cell_001.png', '/data/cell_002.png', ...]

        labels:
            每个细胞的标签数组
            - 单细胞模式：所有值都可能是 0
            - 聚类模式：值为对应候选簇标签，如 [0,0,1,2,1,...]

        similarity_matrix:
            每个待处理对象（通常是单细胞）与所有参考簇的相似度矩阵
            shape = [n_cells, n_reference_clusters]

        rule_type:
            当前匹配规则类型，字符串：
            - 'rule1'
            - 'rule2'

        save_dirs:
            一个字典，包含两个保存目录：
            {
                'rule_dir':  当前规则专属目录,
                'total_dir': 汇总目录
            }

        target_ref_indices:
            允许被保存的最相似参考簇编号列表
            例如 [0, 5] 表示：
                只有当该细胞最相似的参考簇是 R0 或 R5 时，才真正保存

    Returns:
        count:
            实际成功保存的细胞数量

        processed_indices:
            实际被保存的细胞索引列表
    """
    # 已保存细胞计数器
    count = 0

    # 用于记录哪些索引真正被处理和保存了
    processed_indices = []

    # ── Step 1：逐个遍历待处理细胞索引 ──────────────────────────────────────
    for idx in indices:

        # ── Step 2：找到该细胞最相似的参考簇 ────────────────────────────────
        # similarity_matrix[idx] 是该细胞与所有参考簇的相似度向量
        # np.argmax(...) 返回最大值所在的位置，即"最相似参考簇编号"
        most_similar_ref = np.argmax(similarity_matrix[idx])

        # ── Step 3：只有最相似参考簇是目标簇（R0 或 R5）时才保存 ────────────
        # 这是一个额外过滤层。
        #
        # 即使某些细胞属于某个满足规则的候选簇，
        # 如果单个细胞本身最相似的参考簇不是 R0 或 R5，
        # 这里也不会把它保存下来。
        #
        # 这说明：
        #   保存策略是"簇级筛选 + 单细胞级再次确认"
        if most_similar_ref in target_ref_indices:

            # 原图路径
            src_path = image_paths[idx]

            # 取出文件名，不含目录
            # 例如：
            #   '/data/cell_001.png' → 'cell_001.png'
            filename = os.path.basename(src_path)

            # 当前细胞所属的簇标签
            # 单细胞模式下通常全是 0
            # 聚类模式下可能是 0/1/2/...
            cluster_label = labels[idx]

            # ── Step 4：构造新文件名 ───────────────────────────────────────
            # 文件名前缀格式：
            #   rule1_cluster2_ref5
            # 表示：
            #   - 来自规则1
            #   - 属于候选簇2
            #   - 最相似参考簇是R5
            #
            # 最终新文件名示例：
            #   rule1_cluster2_ref5_cell_001.png
            prefix = f"{rule_type}_cluster{cluster_label}_ref{most_similar_ref}"
            new_name = f"{prefix}_{filename}"

            # ── Step 5：复制文件到两个目录 ─────────────────────────────────
            # 复制到当前规则专属目录
            shutil.copy(src_path, os.path.join(save_dirs['rule_dir'], new_name))

            # 再复制到总汇总目录
            shutil.copy(src_path, os.path.join(save_dirs['total_dir'], new_name))

            # 更新计数和记录
            count += 1
            processed_indices.append(idx)

    return count, processed_indices

## clustering/calculate_consensus_score.py

In [ ]:
# =============================================================================
# clustering/calculate_consensus_score.py
# 职责：对“参考簇 ↔ 候选簇”的匹配结果计算共识得分（consensus score）。
#
# ★ 这个文件回答的问题是：一个匹配对不仅“像不像”，还“稳不稳” ★
#
# 在前面的流程中，通常已经完成了这些步骤：
#   1. 对参考域（reference domain）做聚类，得到参考簇
#   2. 对候选域（candidate domain）做聚类，得到候选簇
#   3. 根据簇中心相似度和匹配规则，得到一些“匹配对”
#      例如：[(0, 2, 0.83), (5, 1, 0.79)]
#      含义分别是：
#        - 参考簇 R0 与候选簇 C2 匹配，相似度 0.83
#        - 参考簇 R5 与候选簇 C1 匹配，相似度 0.79
#
# 但“匹配”还不够，因为：
#   - 可能只是簇中心碰巧相似
#   - 簇内部的单细胞未必真的都支持这个匹配
#   - 反过来，参考域中的恶性细胞也未必真的把这个候选簇视作最接近的对象
#
# 所以这里引入“双向共识计算（bidirectional consensus）”：
#
#   方向1：候选域 → 参考域
#     看某个候选簇里的细胞，是否整体更接近恶性参考簇（R0 或 R5）
#
#   方向2：参考域 → 候选域
#     看参考域中的恶性细胞（R0 和 R5 内的细胞），是否真的有一部分把当前候选簇
#     视作它们最接近的候选簇
#
# 最终：
#   总共识得分 = (候选域共识得分 + 参考域共识得分) / 2
#
# 这比只看“簇中心相似度”更稳健，因为它同时考虑了：
#   - 候选簇内部的一致性
#   - 参考恶性细胞对该候选簇的反向认可程度
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import shutil


# =============================================================================
# 核心函数：计算匹配簇的共识得分（双向共识）
# =============================================================================
def calculate_consensus_score(reference_results, candidate_results, matching_pairs, save_dir=None):
    """
    计算匹配簇的共识得分（双向共识版本）。

    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    参数说明：

    reference_results:
        参考域聚类结果字典，通常至少包含：
            {
                'features':        参考域样本特征矩阵，shape [N_ref, D]
                'labels':          参考域样本簇标签，shape [N_ref,]
                'cluster_centers': 参考簇中心矩阵，shape [K_ref, D]
                ...
            }

    candidate_results:
        候选域聚类结果字典，通常至少包含：
            {
                'features':        候选域样本特征矩阵，shape [N_cand, D]
                'labels':          候选域样本簇标签，shape [N_cand,]
                'cluster_centers': 候选簇中心矩阵，shape [K_cand, D]
                ...
            }

    matching_pairs:
        匹配对列表，每个元素形如：
            (ref_idx, cand_idx, similarity)

        例如：
            (0, 2, 0.83) 表示：
                参考簇 R0 与候选簇 C2 匹配，簇中心相似度为 0.83

    save_dir:
        如果提供，则把结果保存为 CSV 文件到该目录

    返回：
        consensus_scores:
            一个列表，列表中每个元素是一个字典，记录一个匹配对的详细共识得分信息
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """

    # ── Step 1：从结果字典中提取基础数据 ───────────────────────────────────
    # reference_features：参考域所有样本的特征矩阵
    # 形状：[N_ref, D]
    reference_features = reference_results['features']

    # reference_labels：参考域每个样本所属的参考簇编号
    # 形状：[N_ref,]
    reference_labels = reference_results['labels']

    # candidate_features：候选域所有样本的特征矩阵
    # 形状：[N_cand, D]
    candidate_features = candidate_results['features']

    # candidate_labels：候选域每个样本所属的候选簇编号
    # 形状：[N_cand,]
    candidate_labels = candidate_results['labels']

    # ── Step 2：定义恶性参考簇索引 ──────────────────────────────────────────
    # 按当前项目约定，R0 和 R5 被视为恶性相关参考簇。
    # 后续共识得分只对“匹配到 R0 或 R5 的候选簇”进行计算。
    malignant_ref_clusters = [0, 5]

    # ── Step 3：准备参考簇中心 ─────────────────────────────────────────────
    ref_centers = reference_results['cluster_centers']
    # ref_centers 形状：[K_ref, D]

    # 分别取出 R0 和 R5 的簇中心
    # reshape(1, -1) 的作用：
    #   把 1D 向量 [D,] 变成 2D 矩阵 [1, D]
    # 因为 sklearn 的 cosine_similarity 要求输入是二维数组
    R0_center = ref_centers[0].reshape(1, -1)
    R5_center = ref_centers[5].reshape(1, -1)

    # ── Step 4：预先计算“每个候选样本”到 R0 / R5 的余弦相似度 ─────────────
    # candidate_features: [N_cand, D]
    # R0_center:          [1, D]
    # 输出：
    #   cell_to_R0_similarity: [N_cand, 1]
    #   其中第 i 行表示第 i 个候选样本与 R0 中心的相似度
    cell_to_R0_similarity = cosine_similarity(candidate_features, R0_center)

    # 同理，计算每个候选样本与 R5 中心的相似度
    cell_to_R5_similarity = cosine_similarity(candidate_features, R5_center)

    # ── Step 5：准备候选簇中心 ─────────────────────────────────────────────
    cand_centers = candidate_results['cluster_centers']
    # cand_centers 形状：[K_cand, D]

    # ── Step 6：找出参考域中属于 R0 和 R5 的样本索引 ──────────────────────
    # np.where(reference_labels == 0)[0]：
    #   返回所有“参考标签等于0”的样本索引
    # 即：属于参考簇 R0 的全部单细胞位置
    R0_indices = np.where(reference_labels == 0)[0]

    # 返回所有“参考标签等于5”的样本索引
    # 即：属于参考簇 R5 的全部单细胞位置
    R5_indices = np.where(reference_labels == 5)[0]

    # ── Step 7：取出 R0 和 R5 内部的所有样本特征 ───────────────────────────
    # 这些是真正属于恶性参考簇的单细胞特征，
    # 后续会用来做“参考域 → 候选域”的反向共识计算
    R0_features = reference_features[R0_indices]
    R5_features = reference_features[R5_indices]

    # ── Step 8：初始化结果列表 ─────────────────────────────────────────────
    consensus_scores = []

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # 主循环：逐个匹配对计算双向共识得分
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    for ref_idx, cand_idx, similarity in matching_pairs:

        # ── Step 8.1：只处理与恶性参考簇（R0 或 R5）相关的匹配对 ───────────
        # 如果某个匹配对的参考簇不是 R0 / R5，则跳过
        # 因为当前设计只关注恶性相关匹配
        if ref_idx in malignant_ref_clusters:

            # ── Step 8.2：找出当前候选簇 cand_idx 内部的所有样本索引 ───────
            # candidate_labels == cand_idx 得到一个布尔掩码
            # np.where(...)[0] 得到该簇所有样本的位置索引
            cand_cluster_indices = np.where(candidate_labels == cand_idx)[0]

            # 只有当前候选簇非空时才继续计算
            if len(cand_cluster_indices) > 0:

                # ============================================================
                # 第一部分：候选域共识得分（Candidate Consensus Score）
                # ============================================================
                # 核心思想：
                #   看这个候选簇里的细胞，是否整体都更接近恶性参考中心（R0 或 R5）
                #
                # 但请注意：
                #   当前代码写法中，只比较 R0 和 R5 谁更近，
                #   不和其他参考簇比较。
                #
                # 于是会发生一个很重要的现象：
                #   对于该候选簇内的每一个样本，不是更接近 R0，就是更接近 R5，
                #   因此：
                #       closest_to_R0_count + closest_to_R5_count == 该簇总细胞数
                #
                # 所以 candidate_consensus_score 实际上恒等于 1.0
                #
                # 这一点不是你的理解问题，而是代码逻辑本身如此。
                # 我后面会在解释部分专门指出这一点。
                # ============================================================

                # 统计该候选簇中“更接近 R0”的样本数
                closest_to_R0_count = 0

                # 统计该候选簇中“更接近 R5”的样本数
                closest_to_R5_count = 0

                for idx in cand_cluster_indices:
                    # cell_to_R0_similarity[idx] 和 cell_to_R5_similarity[idx]
                    # 都是形状 [1,] 的数组（不是纯标量）
                    #
                    # 这里写成 if array > array 在多数情况下仍可工作，
                    # 因为它们都是单元素数组，但更严谨的写法应是：
                    #   cell_to_R0_similarity[idx, 0] > cell_to_R5_similarity[idx, 0]
                    #
                    # 当前代码的语义是：
                    #   如果当前候选细胞与 R0 的相似度大于与 R5 的相似度，
                    #   则计入“更接近 R0”
                    #   否则计入“更接近 R5”
                    if cell_to_R0_similarity[idx] > cell_to_R5_similarity[idx]:
                        closest_to_R0_count += 1
                    else:
                        closest_to_R5_count += 1

                # 候选域共识得分：
                #   认为“只要该细胞更接近 R0 或 R5 之一，就算支持恶性共识”
                #
                # 由于 for 循环中每个细胞必然被计入 R0 或 R5 其中一个，
                # 因而：
                #   closest_to_R0_count + closest_to_R5_count = len(cand_cluster_indices)
                #
                # 所以这个得分恒为：
                #   1.0
                candidate_consensus_score = (
                    closest_to_R0_count + closest_to_R5_count
                ) / len(cand_cluster_indices)

                # ============================================================
                # 第二部分：参考域共识得分（Reference Consensus Score）
                # ============================================================
                # 核心思想：
                #   反过来看参考域中的恶性细胞（R0 和 R5 内部的单细胞），
                #   它们是否把“当前候选簇”视作自己最接近的候选簇？
                #
                # 如果很多 R0/R5 的单细胞都把 cand_idx 当作最像的候选簇，
                # 说明 cand_idx 不只是“看起来像恶性”，
                # 也真的得到了恶性参考细胞的“反向认可”。
                # ============================================================

                # ── Step 8.3：取出当前候选簇中心 ───────────────────────────
                current_cand_center = cand_centers[cand_idx].reshape(1, -1)

                # ── Step 8.4：计算 R0 内所有单细胞 到 当前候选簇中心 的相似度 ─
                # 输出形状：[len(R0_indices), 1]
                R0_to_current_cand_similarity = cosine_similarity(
                    R0_features,
                    current_cand_center
                )

                # ── Step 8.5：计算 R0 内所有单细胞 到 所有候选簇中心 的相似度 ─
                # 输出形状：[len(R0_indices), K_cand]
                # 其中：
                #   第 i 行表示第 i 个 R0 单细胞与所有候选簇中心的相似度
                R0_to_all_cand_similarity = cosine_similarity(R0_features, cand_centers)

                # 统计：R0 中有多少个单细胞，把“当前候选簇 cand_idx”
                # 视作它最接近的候选簇
                R0closest_to_CurrentCandidate_count = 0

                for i in range(len(R0_indices)):
                    # 找出第 i 个 R0 样本最相似的候选簇编号
                    closest_cand = np.argmax(R0_to_all_cand_similarity[i])

                    # 如果它最相似的恰好是当前候选簇 cand_idx，计数加一
                    if closest_cand == cand_idx:
                        R0closest_to_CurrentCandidate_count += 1

                # ── Step 8.6：对 R5 重复同样过程 ───────────────────────────
                R5_to_current_cand_similarity = cosine_similarity(
                    R5_features,
                    current_cand_center
                )

                R5_to_all_cand_similarity = cosine_similarity(R5_features, cand_centers)

                R5closest_to_CurrentCandidate_count = 0

                for i in range(len(R5_indices)):
                    closest_cand = np.argmax(R5_to_all_cand_similarity[i])
                    if closest_cand == cand_idx:
                        R5closest_to_CurrentCandidate_count += 1

                # ── Step 8.7：计算参考域共识得分 ────────────────────────────
                # 分母 = R0 样本数 + R5 样本数
                # 分子 = 其中把“当前候选簇”当作最接近候选簇的样本数
                #
                # 含义：
                #   所有恶性参考单细胞中，有多少比例认同“当前候选簇是它们最像的候选簇之一”
                if len(R0_indices) + len(R5_indices) > 0:
                    reference_consensus_score = (
                        R0closest_to_CurrentCandidate_count
                        + R5closest_to_CurrentCandidate_count
                    ) / (len(R0_indices) + len(R5_indices))
                else:
                    # 极端情况：R0 和 R5 都没有样本
                    reference_consensus_score = 0

                # ── Step 8.8：计算总共识得分 ───────────────────────────────
                # 直接取候选域共识 和 参考域共识 的平均值
                consensus_score = (
                    candidate_consensus_score + reference_consensus_score
                ) / 2

                # ── Step 8.9：保存当前匹配对的详细结果 ─────────────────────
                consensus_scores.append({
                    # 匹配对的参考簇编号（如 0 或 5）
                    'Reference_Cluster': ref_idx,

                    # 匹配对的候选簇编号
                    'Candidate_Cluster': cand_idx,

                    # 参考簇中心与候选簇中心的原始相似度
                    'Cluster_Similarity': similarity,

                    # 当前候选簇中的样本数量
                    'Cluster_Size': len(cand_cluster_indices),

                    # 候选簇中更接近 R0 的细胞数
                    'Cells_Closest_To_R0': closest_to_R0_count,

                    # 候选簇中更接近 R5 的细胞数
                    'Cells_Closest_To_R5': closest_to_R5_count,

                    # 候选簇中更接近恶性参考簇（R0 或 R5）的细胞总数
                    # 注意：按当前代码逻辑，它恒等于 Cluster_Size
                    'Cells_Closest_To_Malignant': (
                        closest_to_R0_count + closest_to_R5_count
                    ),

                    # 候选域共识得分
                    # 注意：按当前代码逻辑，它恒等于 1.0
                    'Candidate_Consensus_Score': candidate_consensus_score,

                    # R0 中有多少细胞把当前候选簇作为最接近候选簇
                    'R0Cells_Closest_To_CurrentCandidate': R0closest_to_CurrentCandidate_count,

                    # R5 中有多少细胞把当前候选簇作为最接近候选簇
                    'R5Cells_Closest_To_CurrentCandidate': R5closest_to_CurrentCandidate_count,

                    # R0 + R5 中总共有多少细胞把当前候选簇作为最接近候选簇
                    'R0R5Cells_Closest_To_CurrentCandidate': (
                        R0closest_to_CurrentCandidate_count
                        + R5closest_to_CurrentCandidate_count
                    ),

                    # 参考域共识得分
                    'Reference_Consensus_Score': reference_consensus_score,

                    # 最终总共识得分
                    'Consensus_Score': consensus_score,
                })

    # ── Step 9：如指定保存目录，则输出 CSV 文件 ─────────────────────────────
    if save_dir and consensus_scores:
        consensus_df = pd.DataFrame(consensus_scores)

        save_path = os.path.join(save_dir, 'cluster_matching.csv')
        consensus_df.to_csv(save_path, index=False)

        print(f"共识得分已保存到 {save_path}")

    return consensus_scores

### 上面代码有个重要问题。Candidate_Consensus_Score 实际上恒等于 1，这要不然改代码逻辑，要不然移除此部分计算（那共识得分完全由参考域共识得分决定）

In [ ]:
# =============================================================================
# clustering/calculate_consensus_score.py
# 修正版：计算匹配簇的双向共识得分
#
# 修正点：
# 1. 不再把“更接近 R0 或 R5”都算作支持，从而避免 Candidate_Consensus 恒等于 1
# 2. 改为：只统计“是否支持当前匹配到的具体参考簇 ref_idx”
# 3. Reference_Consensus 也只使用当前 ref_idx 对应参考簇内的细胞
#
# 适配当前项目：
# - reference_results / candidate_results 的字典结构保持不变
# - matching_pairs 仍为 [(ref_idx, cand_idx, similarity), ...]
# - 默认重点关注恶性参考簇 R0 和 R5
# =============================================================================

import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity


def calculate_consensus_score(
    reference_results,
    candidate_results,
    matching_pairs,
    save_dir=None,
    malignant_ref_clusters=[0, 5]
):
    """
    计算匹配簇的双向共识得分（修正版）

    参数:
    ----------
    reference_results : dict
        参考域聚类结果字典，至少包含：
        - 'features': 参考域样本特征矩阵, shape [N_ref, D]
        - 'labels': 参考域样本簇标签, shape [N_ref,]
        - 'cluster_centers': 参考簇中心矩阵, shape [K_ref, D]

    candidate_results : dict
        候选域聚类结果字典，至少包含：
        - 'features': 候选域样本特征矩阵, shape [N_cand, D]
        - 'labels': 候选域样本簇标签, shape [N_cand,]
        - 'cluster_centers': 候选簇中心矩阵, shape [K_cand, D]

    matching_pairs : list
        匹配对列表，每个元素为：
        (ref_idx, cand_idx, similarity)

    save_dir : str or None
        若不为 None，则保存结果到 CSV

    malignant_ref_clusters : list
        关注的恶性参考簇索引，默认 [0, 5]

    返回:
    ----------
    consensus_scores : list[dict]
        每个匹配对的详细共识得分结果
    """

    # -------------------------------------------------------------------------
    # Step 1. 提取基础数据
    # -------------------------------------------------------------------------
    reference_features = reference_results['features']          # [N_ref, D]
    reference_labels = reference_results['labels']             # [N_ref,]
    reference_centers = reference_results['cluster_centers']   # [K_ref, D]

    candidate_features = candidate_results['features']         # [N_cand, D]
    candidate_labels = candidate_results['labels']            # [N_cand,]
    candidate_centers = candidate_results['cluster_centers']  # [K_cand, D]

    # -------------------------------------------------------------------------
    # Step 2. 预计算：
    #   2.1 每个候选细胞 → 所有参考簇中心 的相似度
    #   2.2 每个参考细胞 → 所有候选簇中心 的相似度
    #
    # 这样后面每个 matching pair 就不用重复算，提高效率
    # -------------------------------------------------------------------------
    candidate_to_ref_similarity = cosine_similarity(candidate_features, reference_centers)
    # shape: [N_cand, K_ref]

    reference_to_cand_similarity = cosine_similarity(reference_features, candidate_centers)
    # shape: [N_ref, K_cand]

    # -------------------------------------------------------------------------
    # Step 3. 对每个细胞，预先找“最近中心”
    # -------------------------------------------------------------------------
    # 每个候选细胞最近的参考簇编号
    candidate_nearest_ref = np.argmax(candidate_to_ref_similarity, axis=1)
    # shape: [N_cand,]

    # 每个参考细胞最近的候选簇编号
    reference_nearest_cand = np.argmax(reference_to_cand_similarity, axis=1)
    # shape: [N_ref,]

    # -------------------------------------------------------------------------
    # Step 4. 遍历每个匹配对，计算双向共识
    # -------------------------------------------------------------------------
    consensus_scores = []

    for ref_idx, cand_idx, similarity in matching_pairs:

        # 只处理项目中定义的恶性参考簇匹配（通常是 R0 / R5）
        if ref_idx not in malignant_ref_clusters:
            continue

        # ---------------------------------------------------------------------
        # Step 4.1 候选簇内的样本索引
        # ---------------------------------------------------------------------
        cand_cluster_indices = np.where(candidate_labels == cand_idx)[0]

        # 如果候选簇为空，跳过
        if len(cand_cluster_indices) == 0:
            continue

        # ---------------------------------------------------------------------
        # Step 4.2 当前参考簇内的样本索引
        # ---------------------------------------------------------------------
        ref_cluster_indices = np.where(reference_labels == ref_idx)[0]

        # 如果当前参考簇为空，跳过
        if len(ref_cluster_indices) == 0:
            continue

        # =====================================================================
        # 一、Candidate → Reference 共识
        # =====================================================================
        # 当前候选簇中的每个细胞，看它最近的参考簇是不是当前匹配目标 ref_idx
        #
        # 例如匹配对为 (R5, C2)：
        #   那么就统计 C2 中有多少细胞最近参考簇是 R5
        #   而不是“R0 或 R5 任意一个都算”
        # =====================================================================
        cand_cluster_nearest_ref = candidate_nearest_ref[cand_cluster_indices]

        candidate_support_count = np.sum(cand_cluster_nearest_ref == ref_idx)
        candidate_cluster_size = len(cand_cluster_indices)

        candidate_consensus_score = candidate_support_count / candidate_cluster_size

        # ---------------------------------------------------------------------
        # 为了更细致地分析，再额外统计：
        # 当前候选簇内，最近参考簇分别是 R0 / R5 / 其他 的数量
        # ---------------------------------------------------------------------
        cells_closest_to_R0 = np.sum(cand_cluster_nearest_ref == 0)
        cells_closest_to_R5 = np.sum(cand_cluster_nearest_ref == 5)
        cells_closest_to_others = np.sum(
            (cand_cluster_nearest_ref != 0) & (cand_cluster_nearest_ref != 5)
        )

        # =====================================================================
        # 二、Reference → Candidate 共识
        # =====================================================================
        # 当前参考簇中的每个细胞，看它最近的候选簇是不是当前匹配目标 cand_idx
        #
        # 例如匹配对为 (R5, C2)：
        #   那么就统计 R5 中有多少细胞最近候选簇是 C2
        # =====================================================================
        ref_cluster_nearest_cand = reference_nearest_cand[ref_cluster_indices]

        reference_support_count = np.sum(ref_cluster_nearest_cand == cand_idx)
        reference_cluster_size = len(ref_cluster_indices)

        reference_consensus_score = reference_support_count / reference_cluster_size

        # =====================================================================
        # 三、总共识得分
        # =====================================================================
        consensus_score = (candidate_consensus_score + reference_consensus_score) / 2

        # ---------------------------------------------------------------------
        # 保存结果
        # ---------------------------------------------------------------------
        consensus_scores.append({
            # 匹配对信息
            'Reference_Cluster': ref_idx,
            'Candidate_Cluster': cand_idx,
            'Cluster_Similarity': float(similarity),

            # 候选簇大小 / 参考簇大小
            'Candidate_Cluster_Size': int(candidate_cluster_size),
            'Reference_Cluster_Size': int(reference_cluster_size),

            # 候选域方向统计
            'Candidate_Support_Count': int(candidate_support_count),
            'Candidate_Consensus_Score': float(candidate_consensus_score),

            # 候选簇内细胞的细分流向
            'Cells_Closest_To_R0': int(cells_closest_to_R0),
            'Cells_Closest_To_R5': int(cells_closest_to_R5),
            'Cells_Closest_To_Others': int(cells_closest_to_others),

            # 参考域方向统计
            'Reference_Support_Count': int(reference_support_count),
            'Reference_Consensus_Score': float(reference_consensus_score),

            # 最终总分
            'Consensus_Score': float(consensus_score),
        })

    # -------------------------------------------------------------------------
    # Step 5. 保存结果
    # -------------------------------------------------------------------------
    if save_dir and len(consensus_scores) > 0:
        os.makedirs(save_dir, exist_ok=True)
        consensus_df = pd.DataFrame(consensus_scores)
        save_path = os.path.join(save_dir, 'cluster_matching.csv')
        consensus_df.to_csv(save_path, index=False)
        print(f"共识得分已保存到 {save_path}")

    return consensus_scores

## visualization/visualization.py

In [ ]:
# =============================================================================
# visualization/visualization.py
# 职责：存放项目中与“可视化”相关的函数。
#
# 这个文件主要解决以下几类问题：
# 1. 统计每个聚类簇（cluster）里不同来源类别（LUAD/LUSC/SCLC/Benign/...）的组成
# 2. 把高维特征降到 2D（通常用 UMAP / PCA），画出全局分布图
# 3. 把每个簇里的细胞图像直接贴到二维坐标图上，形成“图像版聚类图”
# 4. 可视化 reference domain 和 candidate domain 的匹配关系
# 5. 画相似度热图
# 6. 可视化经过单细胞 filtering 后保留下来的匹配细胞
#
# 整体上，这个文件属于“结果解释层”：
#   上游模块负责提特征、聚类、匹配、打分；
#   本文件负责把结果变成图，让人能看懂。
# =============================================================================


# =============================================================================
# 0. 导入依赖库
# =============================================================================
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from tqdm import tqdm
import umap  # 这里依赖的是 umap-learn 库，不是 TensorFlow/其他 UMAP
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist
import seaborn as sns
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------------------------------------------------------
# 说明：
# 这里存在一些重复导入，例如：
#   - matplotlib.pyplot as plt 导入了两次
#   - PIL.Image 导入了两次
#   - numpy / os / OffsetImage / AnnotationBbox 也有重复导入
#
# 这不会影响程序运行（Python 会缓存模块），
# 但属于代码整洁性问题，后续可以精简。
# -----------------------------------------------------------------------------


# =============================================================================
# 1. 定义类别名称、颜色、marker 形状
# =============================================================================
# 这些全局变量用于统一整个项目的可视化风格。
#
# source_labels 通常是一个整数标签，例如：
#   0 -> Candidate
#   1 -> LUAD
#   2 -> LUSC
#   3 -> SCLC
#   4 -> Benign
#   5 -> Malignant
#
# 在后面的图里：
#   - category_names 控制图例文字
#   - category_colors 控制颜色
#   - category_markers 控制点的形状
# -----------------------------------------------------------------------------
category_names = {
    0: 'Candidate',
    1: 'LUAD',
    2: 'LUSC',
    3: 'SCLC',
    4: 'Benign',
    5: 'Malignant'
}

category_colors = {
    0: (40, 120, 181),   # Candidate: 深蓝色
    1: (255, 136, 132),  # LUAD: 浅粉色
    2: (200, 36, 35),    # LUSC: 深红色
    3: (255, 165, 0),    # SCLC: 橙色
    4: (154, 201, 219),  # Benign: 浅蓝色
    5: (185, 89, 184)    # Malignant: 紫色
}

category_markers = {
    0: 'o',  # Candidate: 圆形
    1: '^',  # LUAD: 三角形
    2: 'D',  # LUSC: 菱形
    3: '*',  # SCLC: 星形
    4: 'P',  # Benign: 十字形
    5: 'p'   # Malignant: 五边形
}


def set_plot_font():
    """
    设置 matplotlib 的全局字体样式。

    作用：
    1. 指定字体族为 sans-serif（无衬线字体）
    2. 优先使用 Arial，其次 SimHei（黑体）
       - Arial 常用于英文显示
       - SimHei 常用于中文显示，避免中文乱码
    3. 允许坐标轴正常显示负号
    4. 全局加粗字体

    注意：
    如果运行环境没有 Arial 或 SimHei，matplotlib 会回退到其他可用字体。
    """
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'SimHei']
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['font.weight'] = 'bold'


# =============================================================================
# 2. 定义可视化功能函数
# =============================================================================


# =============================================================================
# 2-0. 计算细胞大小：统计图像中非背景像素数量
# =============================================================================
def count_non_background_pixels(image_path):
    """
    计算图像中非 (225,225,225) 背景像素的数量，作为细胞大小的估计。

    参数:
    ----------
    image_path : str
        图像文件路径

    返回:
    ----------
    non_background_count : int
        非背景像素数量。若读取失败则返回 0

    原理说明：
    ----------
    对于单细胞图像，背景通常是近白色，例如 (225,225,225)。
    如果一个像素不是这个背景值，则认为它属于“细胞区域”。

    注意：
    ----------
    这里要求一个像素的 R/G/B 三个通道都正好等于 225，才算背景。
    因此：
    - 背景必须非常统一才适合这个写法
    - 如果图像压缩、插值或背景不完全一致，可能会误计
    """
    try:
        # 读取图像，并强制转为 RGB 三通道
        img = Image.open(image_path).convert('RGB')

        # 转成 numpy 数组，形状通常为 [H, W, 3]
        img_array = np.array(img)

        # 创建背景掩码：
        # img_array == [225,225,225] 会逐像素逐通道比较
        # np.all(..., axis=2) 表示只有当一个像素的三个通道都等于 225 时才为 True
        background_mask = np.all(img_array == [225, 225, 225], axis=2)

        # ~background_mask 取反：背景为 False，非背景为 True
        # np.sum(True) 会按 1 计数，因此得到非背景像素数
        non_background_count = np.sum(~background_mask)

        return non_background_count

    except Exception as e:
        print(f"计算图像 {image_path} 的非背景像素数量时出错: {e}")
        return 0


# =============================================================================
# 2-1. 统计每个簇中各类别数量，并绘制堆叠柱状图
# =============================================================================
def analyze_cluster_composition(labels, source_labels, image_paths, method_name, save_dir):
    """
    统计每个簇中各类别的数量，并保存：
    1. 一个 CSV 汇总表
    2. 一个堆叠柱状图

    参数:
    ----------
    labels : array-like
        聚类标签，例如 [0,0,1,2,1,...]
        表示每个样本属于哪个 cluster

    source_labels : array-like
        样本来源标签，例如：
        0: candidate, 1: LUAD, 2: LUSC, 3: SCLC, 4: Benign, 5: Malignant

    image_paths : list
        每个样本的图像路径（这里主要用于结果表保存，不参与统计逻辑）

    method_name : str
        方法名称，如 "Reference_KMeans"、"Candidate_KMeans"

    save_dir : str
        输出目录

    返回:
    ----------
    cluster_composition : dict
        每个簇的组成信息字典
    """

    # -------------------------------------------------------------------------
    # Step 1. 将 RGB 颜色转成 matplotlib 可识别的十六进制格式
    # 例如：
    #   (40,120,181) -> '#2878b5'
    # 这样后面画柱状图时可以直接使用
    # -------------------------------------------------------------------------
    color_map = {
        i: f'#{c[0]:02x}{c[1]:02x}{c[2]:02x}' for i, c in category_colors.items()
    }

    # 获取所有出现过的簇标签，并排序
    unique_clusters = sorted(set(labels))

    # 最终返回的结果字典
    cluster_composition = {}

    # -------------------------------------------------------------------------
    # Step 2. 构建 DataFrame，方便做分组统计
    # 每行对应一个样本
    # -------------------------------------------------------------------------
    results_df = pd.DataFrame({
        'image_path': image_paths,
        'source_label': source_labels,
        'cluster': labels
    })

    # 把整数标签映射成可读字符串
    # 例如 1 -> 'LUAD'
    results_df['category'] = results_df['source_label'].map(category_names)

    # 数据中实际存在的类别
    # 例如如果当前 reference 域只有 LUAD/LUSC/SCLC/Benign/Malignant，
    # 那么 Candidate(0) 可能不存在
    existing_categories = set(results_df['source_label'].unique())

    # -------------------------------------------------------------------------
    # Step 3. 对每个簇统计类别数量
    # -------------------------------------------------------------------------
    for cluster in unique_clusters:
        # 取出该 cluster 的所有样本
        cluster_samples = results_df[results_df['cluster'] == cluster]

        # 统计 source_label 的频次，返回字典
        # 例如 {1: 20, 2: 5, 4: 3}
        category_counts = cluster_samples['source_label'].value_counts().to_dict()

        # 所有理论上可能的类别标签
        all_possible_categories = set(category_names.keys())

        # 保存该簇的统计结果
        cluster_composition[cluster] = {
            'total': len(cluster_samples),
            'categories': {
                category_names[i]: category_counts.get(i, 0)
                for i in all_possible_categories
                if i in existing_categories
            }
        }

    # -------------------------------------------------------------------------
    # Step 4. 构建 summary DataFrame，便于导出 CSV 和画图
    # 每一行对应一个 cluster
    # -------------------------------------------------------------------------
    summary_data = []

    for cluster, data in cluster_composition.items():
        row = {'Cluster': cluster, 'Total': data['total']}
        row.update(data['categories'])
        summary_data.append(row)

    summary_df = pd.DataFrame(summary_data)

    # 保存 CSV
    csv_path = os.path.join(save_dir, f"{method_name}_cluster_composition.csv")
    summary_df.to_csv(csv_path, index=False)
    print(f"聚类组成摘要已保存到: {csv_path}")

    # -------------------------------------------------------------------------
    # Step 5. 绘制堆叠柱状图
    # 每个柱子代表一个 cluster
    # 柱子内部不同颜色代表不同来源类别的样本数
    # -------------------------------------------------------------------------
    set_plot_font()
    plt.figure(figsize=(18, 12))

    clusters = summary_df['Cluster'].tolist()

    # bottom 用来控制堆叠柱状图的“底部起点”
    # 一开始全为 0
    bottom = np.zeros(len(clusters))

    # 按类别顺序逐层往上叠加
    for i in sorted(category_names.keys()):
        category = category_names[i]

        # 只绘制当前数据中真实存在的类别
        if i in existing_categories and category in summary_df.columns:
            values = summary_df[category].values
            plt.bar(clusters, values, bottom=bottom, label=category, color=color_map[i])

            # 更新底部位置，为下一层堆叠做准备
            bottom += values

    # 图形美化
    plt.xlabel('Cluster', fontsize=28, fontweight='bold', labelpad=20)
    plt.ylabel('Number of Samples', fontsize=28, fontweight='bold', labelpad=20)
    plt.title(f'{method_name} - Cluster Composition', fontsize=32, fontweight='bold', pad=20)
    plt.legend(prop={'weight': 'bold', 'size': 18})

    # 如果簇数量太多，让 x 轴标签旋转 90 度，避免重叠
    plt.xticks(rotation=90 if len(clusters) > 10 else 0)

    # 手动调整 x 轴范围，让柱子不要贴边
    plt.xlim(-0.6, len(clusters) - 0.4)
    plt.margins(0.05)

    # 调整边距，避免标签被裁切
    plt.subplots_adjust(left=0.15, right=0.95, top=0.92, bottom=0.1)

    # 坐标轴刻度样式
    plt.tick_params(axis='both', which='major', labelsize=18, pad=10)
    for tick in plt.gca().get_xticklabels():
        tick.set_fontweight('bold')
    for tick in plt.gca().get_yticklabels():
        tick.set_fontweight('bold')

    # 保存图像
    plt.savefig(os.path.join(save_dir, f"{method_name}_cluster_composition.png"), dpi=300)
    plt.close()

    return cluster_composition


# =============================================================================
# 2-2. 全局 UMAP 可视化（不带图像缩略图）
# =============================================================================
def visualize_all_clusters_without_images(features, labels, source_labels, method_name, save_dir, trained_umap=None):
    """
    创建一个全局 UMAP 可视化，显示所有簇的分布，但不包含缩略图。

    表达方式：
    - 不同 cluster 用不同颜色
    - 不同类别（source label）用不同 marker 形状

    参数:
    ----------
    features : np.ndarray
        特征矩阵，shape [N, D]

    labels : array-like
        每个样本的聚类标签

    source_labels : array-like
        每个样本的来源类别标签

    method_name : str
        方法名称，用于标题和输出文件名

    save_dir : str
        保存目录

    trained_umap : umap.UMAP or None
        如果提供，则直接使用该模型做 transform；
        否则重新 fit 一个 UMAP 模型

    返回:
    ----------
    reducer / trained_umap
        返回训练好的 UMAP 模型，方便其他函数复用同一投影空间
    """
    print(f"为{method_name}创建不带缩略图的全局UMAP可视化...")

    # -------------------------------------------------------------------------
    # Step 1. UMAP 降维
    # -------------------------------------------------------------------------
    if trained_umap is None:
        # 没有现成 UMAP 模型时，重新训练一个
        reducer = umap.UMAP(random_state=42)
        embedding = reducer.fit_transform(features)
    else:
        # 已有 UMAP 模型时，直接映射到已有空间
        embedding = trained_umap.transform(features)

    # 获取所有 cluster 编号
    unique_clusters = sorted(set(labels))

    # 为 cluster 分配颜色
    # cm.rainbow 会在彩虹色带上均匀取样
    cluster_colors = cm.rainbow(np.linspace(0, 1, len(unique_clusters)))

    set_plot_font()
    plt.figure(figsize=(14, 12))

    # -------------------------------------------------------------------------
    # Step 2. 按“簇 × 类别”组合绘制散点
    # -------------------------------------------------------------------------
    for cluster_idx, cluster in enumerate(unique_clusters):
        # 当前簇的样本全局索引
        cluster_indices = np.where(labels == cluster)[0]

        # 当前簇在 2D embedding 里的坐标
        cluster_embedding = embedding[cluster_indices]

        # 当前簇的 source label 列表
        cluster_source_labels = [source_labels[i] for i in cluster_indices]

        # 在该簇内部，再按类别拆开绘制不同 marker
        for category in set(source_labels):
            category_indices = [i for i, sl in enumerate(cluster_source_labels) if sl == category]

            if category_indices:
                category_embedding = cluster_embedding[category_indices]

                plt.scatter(
                    category_embedding[:, 0],
                    category_embedding[:, 1],
                    c=[cluster_colors[cluster_idx]],
                    marker=category_markers.get(category, 'o'),
                    label=f'Cluster {cluster} - {category_names.get(category, "Unknown")}',
                    alpha=0.7,
                    s=50,
                    edgecolors='black',
                    linewidths=0.5
                )

    # -------------------------------------------------------------------------
    # Step 3. 构造两个图例
    #   图例1：颜色代表 cluster
    #   图例2：marker 代表类别
    # -------------------------------------------------------------------------
    cluster_legend_elements = [
        plt.Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor=cluster_colors[i],
            markersize=16,
            label=f'Cluster {cluster}'
        )
        for i, cluster in enumerate(unique_clusters)
    ]

    category_legend_elements = [
        plt.Line2D(
            [0], [0],
            marker=marker,
            color='black',
            markersize=16,
            label=category_names[cat]
        )
        for cat, marker in category_markers.items()
        if cat in source_labels
    ]

    # 第一个图例：clusters
    legend1 = plt.legend(
        handles=cluster_legend_elements,
        loc='upper right',
        title="Clusters",
        title_fontsize=14,
        bbox_to_anchor=(1.0, 1.0),
        prop={'weight': 'bold', 'size': 14},
        labelspacing=0.5,
        borderpad=1.0,
        handletextpad=0.5,
        framealpha=0.9
    )
    plt.gca().add_artist(legend1)

    # 第二个图例：categories
    legend2 = plt.legend(
        handles=category_legend_elements,
        loc='lower right',
        title="Categories",
        title_fontsize=14,
        bbox_to_anchor=(1.0, 0.0),
        prop={'weight': 'bold', 'size': 14},
        labelspacing=0.5,
        borderpad=1.0,
        handletextpad=0.5,
        framealpha=0.9
    )
    plt.gca().add_artist(legend2)

    # 标题与坐标轴
    plt.title(f"{method_name} - All Clusters Distribution", fontsize=32, fontweight='bold', pad=20)
    plt.xlabel('UMAP Dimension 1', fontsize=28, fontweight='bold', labelpad=20)
    plt.ylabel('UMAP Dimension 2', fontsize=28, fontweight='bold', labelpad=20)

    plt.tight_layout()
    plt.tick_params(axis='both', which='major', labelsize=18, pad=10)
    for tick in plt.gca().get_xticklabels():
        tick.set_fontweight('bold')
    for tick in plt.gca().get_yticklabels():
        tick.set_fontweight('bold')

    # 保存图像
    os.makedirs(os.path.join(save_dir, 'cluster_images'), exist_ok=True)
    plt.savefig(
        os.path.join(save_dir, 'cluster_images', f"{method_name}_all_clusters_distribution.png"),
        dpi=300,
        bbox_inches='tight'
    )
    plt.close()

    print(f"  完成{method_name}的不带缩略图的全局可视化")

    # 返回 UMAP 模型，便于后续复用
    if trained_umap is None:
        return reducer
    else:
        return trained_umap


# =============================================================================
# 2-3. 每个簇单独画图：在 UMAP/PCA 坐标上直接贴缩略图
# =============================================================================
def improved_visualize_clusters_with_images(
    features,
    labels,
    source_labels,
    image_paths,
    method_name,
    save_dir,
    log_sizes=None,
    thumbnail_size=30,
    max_images=1000
):
    """
    为每个簇创建二维可视化，并把每个细胞图像直接贴到散点图上。

    特点：
    1. 图像周围用边框颜色表示类别
    2. 图像大小反映细胞实际大小（通过 log_sizes 控制）
    3. 保持原始图像长宽比
    4. 每个 cluster 单独出一张图

    参数:
    ----------
    features : np.ndarray
        特征矩阵 [N, D]

    labels : array-like
        聚类标签 [N,]

    source_labels : array-like
        来源类别标签 [N,]

    image_paths : list
        图像路径列表

    method_name : str
        方法名

    save_dir : str
        输出目录

    log_sizes : np.ndarray or None
        如果已预先计算好细胞大小的 log1p 值，可直接传入，避免重复计算

    thumbnail_size : int
        基础缩略图大小

    max_images : int
        每个簇最多展示多少张图，防止图太拥挤
    """
    print(f"为{method_name}的每个簇创建可视化...")

    # -------------------------------------------------------------------------
    # Step 1. 如果没有预先提供 log_sizes，就现算
    # -------------------------------------------------------------------------
    if log_sizes is None:
        print("计算细胞大小...")
        cell_sizes = []
        for img_path in tqdm(image_paths, desc="计算细胞大小"):
            size = count_non_background_pixels(img_path)
            cell_sizes.append(size)

        # log1p(x) = log(1 + x)
        # 作用：压缩右偏分布，避免大细胞和小细胞差距过大
        log_sizes = np.log1p(np.array(cell_sizes))

    # -------------------------------------------------------------------------
    # Step 2. 将 log_sizes 归一化到 [0,1]，再映射到 [0.5, 2.0] 倍
    # 作为图像尺寸缩放因子
    # -------------------------------------------------------------------------
    if np.max(log_sizes) > np.min(log_sizes):
        normalized_log_sizes = (
            (log_sizes - np.min(log_sizes)) /
            (np.max(log_sizes) - np.min(log_sizes))
        )
        size_scale_factors = 0.5 + 1.5 * normalized_log_sizes
    else:
        # 所有细胞大小完全相同时，统一缩放倍数 = 1
        size_scale_factors = np.ones_like(log_sizes)

    # 所有 cluster 编号
    unique_clusters = sorted(set(labels))

    # -------------------------------------------------------------------------
    # Step 3. 对每个 cluster 分别可视化
    # -------------------------------------------------------------------------
    for cluster in unique_clusters:
        cluster_indices = np.where(labels == cluster)[0]

        # 如果一个簇太大，则随机抽样部分样本做展示
        if len(cluster_indices) > max_images:
            np.random.seed(42)
            selected_indices = np.random.choice(cluster_indices, max_images, replace=False)
        else:
            selected_indices = cluster_indices

        # 当前簇选中的样本数据
        cluster_features = features[selected_indices]
        cluster_source_labels = [source_labels[i] for i in selected_indices]
        cluster_image_paths = [image_paths[i] for i in selected_indices]
        cluster_size_factors = [size_scale_factors[i] for i in selected_indices]

        # ---------------------------------------------------------------------
        # Step 3.1 根据样本数量决定降维策略
        #
        # 原因：
        # - UMAP 对极少样本不稳定，甚至无法运行
        # - 1个样本/2个样本时没必要跑降维
        # - 3~5个样本时 PCA 更稳定
        # - >5个样本时 UMAP 才比较有意义
        # ---------------------------------------------------------------------
        if len(cluster_features) == 1:
            cluster_embedding = np.array([[0.0, 0.0]])
            print(f"  聚类 {cluster} 只有1个样本，将其放在原点")

        elif len(cluster_features) == 2:
            cluster_embedding = np.array([[-0.5, 0.0], [0.5, 0.0]])
            print(f"  聚类 {cluster} 只有2个样本，将它们放在x轴上")

        elif len(cluster_features) <= 5:
            try:
                cluster_embedding = PCA(n_components=2).fit_transform(cluster_features)
                print(f"  聚类 {cluster} 有{len(cluster_features)}个样本，使用PCA降维")
            except Exception as e:
                print(f"  聚类 {cluster} PCA降维失败: {e}，使用随机投影")
                cluster_embedding = np.random.randn(len(cluster_features), 2) * 0.5

        else:
            cluster_reducer = umap.UMAP(
                n_neighbors=min(15, len(cluster_features) - 1),
                min_dist=0.1,
                n_components=2,
                metric='euclidean',
                spread=5.0,   # spread 变大可让点更分散
                random_state=42
            )
            cluster_embedding = cluster_reducer.fit_transform(cluster_features)
            print(f"  聚类 {cluster} 有{len(cluster_features)}个样本，使用UMAP降维")

        set_plot_font()
        plt.figure(figsize=(16, 16))

        # ---------------------------------------------------------------------
        # Step 3.2 先画底层散点（方便看整体布局）
        # 每个点颜色按 source label 赋值，但透明度较低
        # ---------------------------------------------------------------------
        plt.scatter(
            cluster_embedding[:, 0],
            cluster_embedding[:, 1],
            c=[
                f'#{category_colors[sl][0]:02x}{category_colors[sl][1]:02x}{category_colors[sl][2]:02x}'
                for sl in cluster_source_labels
            ],
            alpha=0.3,
            s=50
        )

        # ---------------------------------------------------------------------
        # Step 3.3 计算图中点之间的平均距离，用于动态调整缩略图基础大小
        #
        # 直觉：
        #   如果点彼此距离很近，图像就应该小一点，否则会严重重叠
        #   如果点很分散，图像可以稍大一点
        # ---------------------------------------------------------------------
        if len(cluster_embedding) > 1:
            distances = pdist(cluster_embedding)
            avg_distance = np.mean(distances)

            base_size = min(thumbnail_size, avg_distance * 10)
            base_size = max(base_size, 20)
        else:
            base_size = thumbnail_size

        # ---------------------------------------------------------------------
        # Step 3.4 把每张细胞图像贴到对应坐标上
        # ---------------------------------------------------------------------
        for i, (x, y, source_label, img_path, size_factor) in enumerate(zip(
            cluster_embedding[:, 0],
            cluster_embedding[:, 1],
            cluster_source_labels,
            cluster_image_paths,
            cluster_size_factors
        )):
            try:
                # 读取图像
                img = Image.open(img_path).convert('RGB')

                # 原图宽高比
                width, height = img.size
                aspect_ratio = width / height

                # 根据细胞大小缩放图像
                img_size = int(base_size * size_factor)

                # 保持长宽比
                if aspect_ratio >= 1:
                    new_width = img_size
                    new_height = int(img_size / aspect_ratio)
                else:
                    new_height = img_size
                    new_width = int(img_size * aspect_ratio)

                # 重采样缩放
                img = img.resize((new_width, new_height), Image.LANCZOS)

                # PIL -> numpy
                img_array = np.array(img)

                # 获取边框颜色（按来源类别）
                border_color = category_colors.get(source_label, (0, 0, 0))

                # 直接修改像素，为图像加边框
                border_width = 2

                # 上下左右四条边
                img_array[:border_width, :, :] = border_color
                img_array[-border_width:, :, :] = border_color
                img_array[:, :border_width, :] = border_color
                img_array[:, -border_width:, :] = border_color

                # numpy -> PIL
                bordered_img = Image.fromarray(img_array)

                # OffsetImage：把图像包装成 matplotlib 可放置对象
                imagebox = OffsetImage(bordered_img, zoom=1.0)
                imagebox.image.axes = plt.gca()

                # AnnotationBbox：把图像放到坐标 (x, y)
                ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0.0)
                plt.gca().add_artist(ab)

            except Exception as e:
                print(f"无法显示图像 {img_path}: {e}")
                continue

        # ---------------------------------------------------------------------
        # Step 3.5 添加图例
        # 图例的颜色对应不同 source label
        # ---------------------------------------------------------------------
        legend_elements = [
            plt.Line2D(
                [0], [0],
                marker='s',
                color='w',
                markerfacecolor=f'#{c[0]:02x}{c[1]:02x}{c[2]:02x}',
                markersize=18,
                label=category_names[label]
            )
            for label, c in category_colors.items()
            if label in cluster_source_labels
        ]
        plt.legend(
            handles=legend_elements,
            loc='upper right',
            prop={'weight': 'bold', 'size': 18},
            labelspacing=0.5,
            borderpad=1.0,
            handletextpad=0.5,
            framealpha=0.9
        )

        # 标题与坐标轴
        plt.title(f"{method_name} - Cluster {cluster} ({len(cluster_indices)} samples)",
                  fontsize=32, fontweight='bold', pad=20)
        plt.xlabel('UMAP Dimension 1', fontsize=28, fontweight='bold', labelpad=20)
        plt.ylabel('UMAP Dimension 2', fontsize=28, fontweight='bold', labelpad=20)

        # 设为等比例坐标，避免图像被视觉拉伸
        plt.axis('equal')

        # 给坐标轴外围留更多空间，避免贴图被裁切
        plt.margins(0.2)

        plt.tight_layout()
        plt.tick_params(axis='both', which='major', labelsize=18, pad=10)
        for tick in plt.gca().get_xticklabels():
            tick.set_fontweight('bold')
        for tick in plt.gca().get_yticklabels():
            tick.set_fontweight('bold')

        # 保存
        os.makedirs(os.path.join(save_dir, 'cluster_images'), exist_ok=True)
        plt.savefig(
            os.path.join(save_dir, 'cluster_images', f"{method_name}_cluster{cluster}_images_improved.png"),
            dpi=150,   # 这里故意比 300 低，减少文件大小
            bbox_inches='tight'
        )
        plt.close()

    print(f"  完成{method_name}的可视化")


# =============================================================================
# 2-4. 可视化 reference domain 和 candidate domain 的匹配簇
# =============================================================================
def visualize_matching_clusters(
    reference_results,
    candidate_results,
    matching_pairs,
    save_dir,
    trained_umap=None,
    rule1_matched_clusters=None,
    rule2_matched_clusters=None
):
    """
    将 reference domain 和 candidate domain 分别画在左右两个子图中，
    展示各自的聚类分布，并在图下方标记匹配数量信息。

    参数:
    ----------
    reference_results : dict
        参考域结果字典

    candidate_results : dict
        候选域结果字典

    matching_pairs : list
        匹配对列表，每个元素为 (ref_idx, cand_idx, similarity)

    save_dir : str
        保存目录

    trained_umap : umap.UMAP or None
        若提供，则使用该 UMAP 模型投影 reference/candidate

    rule1_matched_clusters : list or None
        规则1匹配到的候选簇编号列表

    rule2_matched_clusters : list or None
        规则2匹配到的候选簇编号列表

    返回:
    ----------
    reducer / trained_umap
        返回使用的 UMAP 模型
    """
    # -------------------------------------------------------------------------
    # Step 1. 读取 reference / candidate 的特征与标签
    # 注意：这里变量名 reference_embedding / candidate_embedding 实际上装的是 features，
    # 还没做 UMAP，所以名字略有误导性。
    # -------------------------------------------------------------------------
    reference_embedding = reference_results['features']
    reference_labels = reference_results['labels']
    reference_source_labels = reference_results['source_labels']

    candidate_embedding = candidate_results['features']
    candidate_labels = candidate_results['labels']
    candidate_source_labels = candidate_results['source_labels']

    # 簇数量
    n_ref_clusters = reference_results['n_clusters']
    n_cand_clusters = candidate_results['n_clusters']

    # 给 reference / candidate 各自分配颜色方案
    ref_colors = plt.cm.tab10(np.linspace(0, 1, n_ref_clusters))
    cand_colors = plt.cm.tab20(np.linspace(0, 1, n_cand_clusters))

    # -------------------------------------------------------------------------
    # Step 2. UMAP 降维
    # reference 先 fit_transform，candidate 再 transform 到同一空间
    # 这样左右两边的分布具有更可比性
    # -------------------------------------------------------------------------
    if trained_umap is None:
        reducer = umap.UMAP(random_state=42)
        reference_umap = reducer.fit_transform(reference_embedding)
        candidate_umap = reducer.transform(candidate_embedding)
    else:
        reference_umap = trained_umap.transform(reference_embedding)
        candidate_umap = trained_umap.transform(candidate_embedding)

    # -------------------------------------------------------------------------
    # Step 3. 创建左右两个子图
    # -------------------------------------------------------------------------
    plt.figure(figsize=(20, 10))

    # =========================
    # 左图：Reference Domain
    # =========================
    plt.subplot(1, 2, 1)

    for cluster in range(n_ref_clusters):
        cluster_indices = np.where(reference_labels == cluster)[0]
        cluster_embedding = reference_umap[cluster_indices]
        cluster_source_labels = [reference_source_labels[i] for i in cluster_indices]

        for category in set(reference_source_labels):
            category_indices = [i for i, sl in enumerate(cluster_source_labels) if sl == category]

            if category_indices:
                category_embedding = cluster_embedding[category_indices]

                plt.scatter(
                    category_embedding[:, 0],
                    category_embedding[:, 1],
                    c=[ref_colors[cluster]],
                    marker=category_markers.get(category, 'o'),
                    label=f'Cluster {cluster} - {category_names.get(category, "Unknown")}',
                    alpha=0.7,
                    s=50,
                    edgecolors='black',
                    linewidths=0.5
                )

    plt.title('Reference Domain Clusters', fontsize=32, fontweight='bold', pad=20)
    plt.xlabel('UMAP Dimension 1', fontsize=24, fontweight='bold', labelpad=20)
    plt.ylabel('UMAP Dimension 2', fontsize=24, fontweight='bold', labelpad=20)

    # =========================
    # 右图：Candidate Domain
    # =========================
    plt.subplot(1, 2, 2)

    for cluster in range(n_cand_clusters):
        cluster_indices = np.where(candidate_labels == cluster)[0]
        cluster_embedding = candidate_umap[cluster_indices]
        cluster_source_labels = [candidate_source_labels[i] for i in cluster_indices]

        for category in set(candidate_source_labels):
            category_indices = [i for i, sl in enumerate(cluster_source_labels) if sl == category]

            if category_indices:
                category_embedding = cluster_embedding[category_indices]

                plt.scatter(
                    category_embedding[:, 0],
                    category_embedding[:, 1],
                    c=cand_colors[cluster],
                    marker=category_markers.get(category, 'o'),
                    label=f'Cluster {cluster} - {category_names.get(category, "Unknown")}',
                    alpha=0.7,
                    s=50,
                    edgecolors='black',
                    linewidths=0.5
                )

    plt.title('Candidate Domain Clusters', fontsize=32, fontweight='bold', pad=20)
    plt.xlabel('UMAP Dimension 1', fontsize=24, fontweight='bold', labelpad=20)
    plt.ylabel('UMAP Dimension 2', fontsize=24, fontweight='bold', labelpad=20)

    # -------------------------------------------------------------------------
    # Step 4. 在图底部加入匹配信息
    # -------------------------------------------------------------------------
    if rule1_matched_clusters is not None and rule2_matched_clusters is not None:
        plt.figtext(
            0.5,
            0.01,
            f'Matched Pairs: {len(matching_pairs)} (Rule1: {len(rule1_matched_clusters)}, Rule2: {len(rule2_matched_clusters)})',
            ha='center',
            fontsize=14
        )
    else:
        plt.figtext(
            0.5,
            0.01,
            f'Matched Pairs: {len(matching_pairs)}',
            ha='center',
            fontsize=14
        )

    # 保存
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, 'matching_visualization.png'), dpi=300, bbox_inches='tight')
    plt.close()

    print(f"匹配可视化已保存到 '{os.path.join(save_dir, 'matching_visualization.png')}'")

    if trained_umap is None:
        return reducer
    else:
        return trained_umap


# =============================================================================
# 2-5. 画相似度矩阵热图
# =============================================================================
def plot_similarity_heatmap(similarity_matrix, x_labels, y_labels, title, save_path):
    """
    绘制相似度矩阵热图。

    参数:
    ----------
    similarity_matrix : np.ndarray
        2D 矩阵，通常是 reference cluster × candidate cluster 的相似度

    x_labels : list
        X 轴标签，通常是 candidate 簇名称

    y_labels : list
        Y 轴标签，通常是 reference 簇名称

    title : str
        图标题

    save_path : str
        保存路径
    """
    plt.figure(figsize=(12, 10))

    # annot=True 表示在每个格子里打印具体数值
    # fmt=".2f" 表示保留两位小数
    sns.heatmap(
        similarity_matrix,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        xticklabels=x_labels,
        yticklabels=y_labels
    )

    plt.title(title, fontsize=28, fontweight='bold', pad=20)
    plt.xlabel("Candidate", fontsize=24, fontweight='bold', labelpad=20)
    plt.ylabel("Reference", fontsize=24, fontweight='bold', labelpad=20)
    plt.tick_params(axis='both', which='major', labelsize=18, pad=10)
    plt.tight_layout()

    plt.savefig(save_path, dpi=300)
    plt.close()
    print(f"  热图已保存至: {save_path}")


# =============================================================================
# 2-6. 可视化单细胞 filtering 后的匹配簇
# =============================================================================
def visualize_filtered_clusters(
    reference_results,
    candidate_results,
    matching_pairs,
    save_dir,
    reference_umap=None,
    r1_clusters=None,
    r2_clusters=None,
    thumbnail_size=30,
    max_images=1000
):
    """
    可视化匹配簇中保留下来的单细胞，并尽量保持与原始簇内部 UMAP 位置一致。

    核心思想：
    1. 先定位哪些 candidate clusters 是匹配簇
    2. 在这些簇内，只保留“最相似参考簇为 R0 或 R5”的细胞
    3. 整个簇仍然先做一次内部降维
    4. 所有细胞用浅灰色点表示
    5. 通过过滤保留的细胞再用彩色 + 贴图高亮显示

    参数:
    ----------
    reference_results : dict
        参考域结果字典

    candidate_results : dict
        候选域结果字典

    matching_pairs : list
        匹配对列表，元素形如 (ref_idx, cand_idx, similarity)

    save_dir : str
        保存目录

    reference_umap : optional
        当前实现中没有实际使用到，保留接口参数

    r1_clusters, r2_clusters : list or None
        若显式传入，则优先使用这些候选簇编号

    thumbnail_size : int
        基础缩略图尺寸

    max_images : int
        单个图最多贴多少张图
    """
    print("为匹配的簇创建可视化（保持原始UMAP位置）...")

    # 候选域基础数据
    features = candidate_results['features']
    labels = candidate_results['labels']
    source_labels = candidate_results['source_labels']
    image_paths = candidate_results['image_paths']

    # -------------------------------------------------------------------------
    # Step 1. 计算细胞级别的“候选细胞 vs 参考簇中心”相似度
    # 输出 shape [N_cand, K_ref]
    # -------------------------------------------------------------------------
    cell_sims = cosine_similarity(features, reference_results['cluster_centers'])

    # -------------------------------------------------------------------------
    # Step 2. 确定需要处理的匹配簇
    # 优先使用显式传入的 r1_clusters / r2_clusters
    # 否则从 matching_pairs 中自动推断：
    #   ref_idx == 5 -> 规则1
    #   ref_idx == 0 -> 规则2
    # -------------------------------------------------------------------------
    matched_clusters = []
    if r1_clusters:
        matched_clusters.extend([(c, 'rule1') for c in r1_clusters])
    if r2_clusters:
        matched_clusters.extend([(c, 'rule2') for c in r2_clusters])

    if not matched_clusters:
        for ref_idx, cand_idx, _ in matching_pairs:
            if ref_idx == 5:
                matched_clusters.append((cand_idx, 'rule1'))
            elif ref_idx == 0:
                matched_clusters.append((cand_idx, 'rule2'))

    # -------------------------------------------------------------------------
    # Step 3. 获取或计算细胞大小
    # 注意：这里写成 hasattr(candidate_results, 'log_sizes')，
    # 但 candidate_results 通常是 dict，不是对象。
    # 对 dict 应该用：
    #   'log_sizes' in candidate_results
    #
    # 也就是说，这里存在一个潜在 bug。
    # 当前我保留原逻辑注释说明，不改写行为。
    # -------------------------------------------------------------------------
    if hasattr(candidate_results, 'log_sizes'):
        log_sizes = candidate_results['log_sizes']
    else:
        print("计算细胞大小...")
        cell_sizes = []
        for img_path in tqdm(image_paths, desc="计算细胞大小"):
            size = count_non_background_pixels(img_path)
            cell_sizes.append(size)

        log_sizes = np.log1p(np.array(cell_sizes))

    # 归一化并映射到 [0.5, 2.0]
    if np.max(log_sizes) > np.min(log_sizes):
        normalized_log_sizes = (
            (log_sizes - np.min(log_sizes)) /
            (np.max(log_sizes) - np.min(log_sizes))
        )
        size_scale_factors = 0.5 + 1.5 * normalized_log_sizes
    else:
        size_scale_factors = np.ones_like(log_sizes)

    # 创建输出目录
    matched_vis_dir = os.path.join(save_dir, 'matched_clusters_visualization')
    os.makedirs(matched_vis_dir, exist_ok=True)

    # -------------------------------------------------------------------------
    # Step 4. 对每个匹配簇单独作图
    # -------------------------------------------------------------------------
    for cluster_idx, rule_type in matched_clusters:
        cluster_indices = np.where(labels == cluster_idx)[0]

        # ---------------------------------------------------------------------
        # Step 4.1 仅保留“最相似参考簇是 R0 或 R5”的细胞
        # ---------------------------------------------------------------------
        target_ref_indices = [0, 5]
        filtered_indices = []

        for idx in cluster_indices:
            most_similar_ref = np.argmax(cell_sims[idx])
            if most_similar_ref in target_ref_indices:
                filtered_indices.append(idx)

        if not filtered_indices:
            print(f"  簇 {cluster_idx} ({rule_type}) 没有与R0或R5匹配的细胞，跳过")
            continue

        # 如果筛选后太多，随机取样
        if len(filtered_indices) > max_images:
            np.random.seed(42)
            selected_indices = np.random.choice(filtered_indices, max_images, replace=False)
        else:
            selected_indices = filtered_indices

        # ---------------------------------------------------------------------
        # Step 4.2 准备该簇的全部样本信息，以及筛选后样本信息
        # 注意：这里 selected_indices 实际上后面没有真正被用来替代 filtered_indices，
        # 因此“max_images 限制”并没有完全生效，这是原代码的一个小问题。
        # ---------------------------------------------------------------------
        cluster_features = features[cluster_indices]
        filtered_features = features[filtered_indices]

        cluster_source_labels = [source_labels[i] for i in cluster_indices]
        cluster_image_paths = [image_paths[i] for i in cluster_indices]
        cluster_size_factors = [size_scale_factors[i] for i in cluster_indices]

        filtered_source_labels = [source_labels[i] for i in filtered_indices]
        filtered_image_paths = [image_paths[i] for i in filtered_indices]
        filtered_size_factors = [size_scale_factors[i] for i in filtered_indices]

        # ---------------------------------------------------------------------
        # Step 4.3 对“整个簇”做降维，而不是只对过滤后细胞做降维
        # 这样可保持过滤后细胞在原簇结构中的相对位置
        # ---------------------------------------------------------------------
        if len(cluster_features) == 1:
            cluster_embedding = np.array([[0.0, 0.0]])

        elif len(cluster_features) == 2:
            cluster_embedding = np.array([[-0.5, 0.0], [0.5, 0.0]])

        elif len(cluster_features) <= 5:
            try:
                cluster_embedding = PCA(n_components=2).fit_transform(cluster_features)
            except Exception as e:
                print(f"  簇 {cluster_idx} PCA降维失败: {e}，使用随机投影")
                cluster_embedding = np.random.randn(len(cluster_features), 2) * 0.5

        else:
            cluster_reducer = umap.UMAP(
                n_neighbors=min(15, len(cluster_features) - 1),
                min_dist=0.1,
                n_components=2,
                metric='euclidean',
                spread=5.0,
                random_state=42
            )
            cluster_embedding = cluster_reducer.fit_transform(cluster_features)

        # ---------------------------------------------------------------------
        # Step 4.4 建立一个布尔掩码 filtered_mask
        # 长度 = 当前 cluster 的样本数
        # 若某个位置对应的细胞被过滤保留，则该位置为 True
        # ---------------------------------------------------------------------
        filtered_mask = np.zeros(len(cluster_indices), dtype=bool)

        for idx in filtered_indices:
            pos = np.where(cluster_indices == idx)[0]
            if len(pos) > 0:
                filtered_mask[pos[0]] = True

        set_plot_font()
        plt.figure(figsize=(16, 16))

        # 先画整个簇所有细胞（浅灰）
        plt.scatter(
            cluster_embedding[:, 0],
            cluster_embedding[:, 1],
            c='lightgray',
            alpha=0.2,
            s=30
        )

        # 再把过滤后的细胞高亮画出来
        plt.scatter(
            cluster_embedding[filtered_mask, 0],
            cluster_embedding[filtered_mask, 1],
            c=[
                f'#{category_colors[sl][0]:02x}{category_colors[sl][1]:02x}{category_colors[sl][2]:02x}'
                for sl in filtered_source_labels
            ],
            alpha=0.8,
            s=80
        )

        # 动态估计基础缩略图大小
        if len(cluster_embedding) > 1:
            distances = pdist(cluster_embedding)
            avg_distance = np.mean(distances)
            base_size = min(thumbnail_size, avg_distance * 10)
            base_size = max(base_size, 20)
        else:
            base_size = thumbnail_size

        # ---------------------------------------------------------------------
        # Step 4.5 只给过滤后的细胞贴图
        # ---------------------------------------------------------------------
        for i, idx in enumerate(np.where(filtered_mask)[0]):
            x, y = cluster_embedding[idx, 0], cluster_embedding[idx, 1]
            source_label = cluster_source_labels[idx]
            img_path = cluster_image_paths[idx]
            size_factor = cluster_size_factors[idx]

            try:
                img = Image.open(img_path).convert('RGB')

                width, height = img.size
                aspect_ratio = width / height

                img_size = int(base_size * size_factor)

                if aspect_ratio >= 1:
                    new_width = img_size
                    new_height = int(img_size / aspect_ratio)
                else:
                    new_height = img_size
                    new_width = int(img_size * aspect_ratio)

                img = img.resize((new_width, new_height), Image.LANCZOS)

                img_array = np.array(img)

                border_color = category_colors.get(source_label, (0, 0, 0))
                border_width = 2

                img_array[:border_width, :, :] = border_color
                img_array[-border_width:, :, :] = border_color
                img_array[:, :border_width, :] = border_color
                img_array[:, -border_width:, :] = border_color

                bordered_img = Image.fromarray(img_array)

                imagebox = OffsetImage(bordered_img, zoom=1.0)
                imagebox.image.axes = plt.gca()

                ab = AnnotationBbox(imagebox, (x, y), frameon=False, pad=0.0)
                plt.gca().add_artist(ab)

            except Exception as e:
                print(f"无法显示图像 {img_path}: {e}")
                continue

        # ---------------------------------------------------------------------
        # Step 4.6 图例
        # 注意：这里图例中追加的灰色点 label='filtered cells'
        # 实际上灰色点表示“未高亮显示的全部簇内细胞背景”，
        # 从语义上更接近“all cluster cells / background cells”
        # 原标签略不准确，但这里保留原写法。
        # ---------------------------------------------------------------------
        legend_elements = [
            plt.Line2D(
                [0], [0],
                marker='s',
                color='w',
                markerfacecolor=f'#{c[0]:02x}{c[1]:02x}{c[2]:02x}',
                markersize=16,
                label=category_names[label]
            )
            for label, c in category_colors.items()
            if label in filtered_source_labels
        ]

        legend_elements.append(
            plt.Line2D(
                [0], [0],
                marker='o',
                color='lightgray',
                markersize=16,
                label='filtered cells'
            )
        )

        plt.legend(
            handles=legend_elements,
            loc='upper right',
            prop={'weight': 'bold', 'size': 16},
            labelspacing=0.5,
            borderpad=1.0,
            handletextpad=0.5,
            framealpha=0.9
        )

        # 找到该 candidate cluster 对应的 reference cluster（如果有）
        matched_ref = None
        for ref_idx, cand_idx, _ in matching_pairs:
            if cand_idx == cluster_idx:
                matched_ref = ref_idx
                break

        # 标题与坐标轴
        plt.title(
            f"{rule_type} - Cluster {cluster_idx} Matched Cells ({len(filtered_indices)}/{len(cluster_indices)} cells)",
            fontsize=32,
            fontweight='bold',
            pad=20
        )
        plt.xlabel('UMAP Dimension 1', fontsize=28, fontweight='bold', labelpad=20)
        plt.ylabel('UMAP Dimension 2', fontsize=28, fontweight='bold', labelpad=20)

        plt.axis('equal')
        plt.margins(0.2)

        plt.tight_layout()
        plt.tick_params(axis='both', which='major', labelsize=18, pad=10)
        for tick in plt.gca().get_xticklabels():
            tick.set_fontweight('bold')
        for tick in plt.gca().get_yticklabels():
            tick.set_fontweight('bold')

        plt.savefig(
            os.path.join(matched_vis_dir, f"{rule_type}_cluster{cluster_idx}_matched_cells.png"),
            dpi=150,
            bbox_inches='tight'
        )
        plt.close()

    print("  匹配簇可视化完成")
    return matched_vis_dir

## application/pipeline.py 整个串联流程

In [ ]:
# =============================================================================
# pipeline.py
#
# 职责：整个项目的主流程入口（pipeline / 流水线）
#
# 这个文件把项目中已经模块化好的功能串联起来，形成一个完整分析流程：
#
#   1. 加载模型和预处理方式
#   2. 构建参考域（reference domain）
#   3. 提取参考域特征并做 PCA + KMeans 聚类
#   4. 对每个病理片（patient / slide）构建候选域（candidate domain）
#   5. 根据候选样本数决定：
#        - 样本足够多：走“聚类模式”
#        - 样本太少：走“单细胞模式”
#   6. 应用匹配规则（Rule1 / Rule2）
#   7. 保存匹配到的细胞图像
#   8. 计算共识得分
#   9. 生成各种可视化
#  10. 汇总所有病人的统计结果
#
# 从架构上看，这个文件不负责实现底层算法细节，
# 而是负责“调度”各个模块，属于 orchestration / glue code。
# =============================================================================


# =============================================================================
# 0. 导入项目内模块
# =============================================================================
# 这里使用的是相对导入（..），说明这个 pipeline.py 位于一个包结构内部。
# 比如可能类似：
#   project/
#     models/
#     data/
#     clustering/
#     visualization/
#     pipeline/
#
# 每个模块各司其职：
#   - models: 模型加载与特征提取
#   - data: 数据路径收集、Dataset 封装等
#   - clustering: K 优化、匹配规则、共识得分等
#   - visualization: 所有绘图函数
# -----------------------------------------------------------------------------
from ..models.models import get_model_and_transform
from ..data.data_utils import collect_reference_data, collect_image_paths, CellImageDataset, extract_patient_id
from ..models.feature_extractor import extract_features_with_cell_size
from ..clustering.K_optimizer import optimize_k_for_candidate_clustering
from ..clustering.calculate_consensus_score import calculate_consensus_score
from ..clustering.matching_rule_application import identify_matching_clusters, save_matched_cells, check_matching_rules
from ..visualization.visualization import (
    visualize_all_clusters_without_images,
    improved_visualize_clusters_with_images,
    visualize_matching_clusters,
    analyze_cluster_composition,
    plot_similarity_heatmap,
    visualize_filtered_clusters
)

# =============================================================================
# 1. 导入第三方依赖
# =============================================================================
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# -----------------------------------------------------------------------------
# 注意：
# 这段代码里后面用到了 count_non_background_pixels()，
# 但当前文件顶部没有显式导入它。
#
# 也就是说，如果这个函数没有在别处被注入到命名空间，
# 代码会在运行到这里时报 NameError。
#
# 理论上应该补充：
# from ..visualization.visualization import count_non_background_pixels
#
# 我这里保留原始逻辑，只在注释中指出这个潜在问题。
# -----------------------------------------------------------------------------


# =============================================================================
# 2. 设置随机种子，确保结果可复现
# =============================================================================
# 在深度学习 / 聚类 / UMAP 这类任务中，很多步骤都带随机性。
# 固定随机种子有助于：
#   - 实验复现
#   - 调试方便
#   - 多次运行结果更稳定可比
# -----------------------------------------------------------------------------
seed = 42
torch.manual_seed(seed)               # CPU 上的 PyTorch 随机种子
torch.cuda.manual_seed_all(seed)      # 所有 GPU 上的随机种子
np.random.seed(seed)                  # NumPy 随机种子
torch.backends.cudnn.deterministic = True  # 强制使用确定性算法
torch.backends.cudnn.benchmark = False     # 关闭 benchmark，避免不同输入下选择不同算法


# =============================================================================
# 3. 主流程函数：run_pipeline()
# =============================================================================
def run_pipeline(
    positive_folder_path,
    negative_folder_path,
    malignant_cells_dir,
    benign_cells_dir,
    model_path,
    base_save_dir,
    feature_layer='penultimate',
    reference_k=10,
    max_candidate_k=20,
    batch_size=16,
    num_workers=0,
    cell_size_weight=1.0,
    model_type='ResNeXt',
    pca_dim=32,
    mean=None,
    std=None
):
    """
    运行跨域簇匹配分析流水线。

    这个函数是整个项目的核心入口，主要用于分析：
    - 参考域（reference domain）：已知恶性 / 良性细胞
    - 候选域（candidate domain）：来自待分析病理片的细胞

    最终目标：
    通过特征提取、聚类、规则匹配、共识评估和可视化，
    找到候选病理片中可能的恶性细胞亚群。

    参数:
    ----------
    positive_folder_path : str
        阳性病例根目录路径

    negative_folder_path : str
        阴性病例根目录路径

    malignant_cells_dir : str
        参考域中的恶性细胞图像目录

    benign_cells_dir : str
        参考域中的良性细胞图像目录

    model_path : str
        预训练模型权重路径

    base_save_dir : str
        所有输出结果的根目录

    feature_layer : str
        从模型的哪一层提取特征，默认 'penultimate'
        常见表示取倒数第二层特征

    reference_k : int
        参考域做 KMeans 时的簇数

    max_candidate_k : int
        候选域优化 K 时允许的最大 K

    batch_size : int
        DataLoader 批大小

    num_workers : int
        DataLoader 子进程数

    cell_size_weight : float
        细胞大小特征的加权系数

    model_type : str
        使用哪个模型架构，例如 'ResNeXt'

    pca_dim : int
        PCA 降维后的目标维度

    mean, std : optional
        图像标准化使用的均值和标准差
    """

    # =========================================================================
    # Step 1. 初始化运行设备与模型
    # =========================================================================
    # 若有 GPU 则使用 CUDA，否则退回 CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # get_model_and_transform() 通常会做三件事：
    #   1. 构造网络模型
    #   2. 加载训练好的权重
    #   3. 返回对应的图像预处理 transform
    #
    # 返回值中第三项这里没有使用，所以用 _ 接住
    model, transform, _ = get_model_and_transform(
        model_type,
        device,
        model_path,
        mean=mean,
        std=std
    )

    # =========================================================================
    # Step 2. 处理参考域（Reference Domain）
    # =========================================================================
    print("======= 处理参考细胞 =======")

    # -------------------------------------------------------------------------
    # 2.1 收集参考域数据
    # collect_reference_data() 的作用通常是：
    #   - 从 malignant_cells_dir / benign_cells_dir 中收集所有图像路径
    #   - 同时生成对应 source label
    #
    # 返回：
    #   ref_paths   : 所有参考细胞图像路径列表
    #   ref_sources : 每张图的来源标签
    # -------------------------------------------------------------------------
    ref_paths, ref_sources = collect_reference_data(malignant_cells_dir, benign_cells_dir)

    # -------------------------------------------------------------------------
    # 2.2 计算每张参考图像的“细胞大小”
    #
    # 思路：
    #   先统计每张图中的非背景像素数，再做 log1p 压缩尺度
    #
    # 这里的 ref_sizes 会在后面特征提取时融合进 feature 中
    # -------------------------------------------------------------------------
    ref_sizes = np.log1p([
        count_non_background_pixels(p) for p in tqdm(ref_paths, desc="Ref Sizes")
    ])

    # -------------------------------------------------------------------------
    # 2.3 构建 DataLoader
    # CellImageDataset 负责：
    #   - 根据图像路径读取图片
    #   - 进行 transform
    #   - 返回图像张量及其 source label
    #
    # 参考域不打乱顺序（shuffle=False），因为后面希望输出顺序与路径一一对应
    # -------------------------------------------------------------------------
    ref_loader = DataLoader(
        CellImageDataset(ref_paths, transform=transform, source_labels=ref_sources),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    # -------------------------------------------------------------------------
    # 2.4 提取参考域特征，并融合细胞大小
    #
    # extract_features_with_cell_size() 通常会：
    #   - 用深度模型提取图像特征
    #   - 把细胞大小信息按权重拼接 / 融合进特征
    #
    # 返回：
    #   ref_feats   : 特征矩阵
    #   ref_sources : 实际对齐后的来源标签
    #   ref_paths   : 实际对齐后的图像路径
    # -------------------------------------------------------------------------
    ref_feats, ref_sources, ref_paths = extract_features_with_cell_size(
        model,
        ref_loader,
        device,
        ref_sizes,
        feature_layer,
        cell_size_weight
    )

    # -------------------------------------------------------------------------
    # 2.5 参考域 PCA 降维 + 归一化
    #
    # 为什么做 PCA：
    #   - 压缩高维特征
    #   - 降噪
    #   - 减少后续聚类难度
    #
    # n_components 取 min(...) 是为了防止：
    #   PCA 维数不能超过样本数，也不能超过原始特征维度
    # -------------------------------------------------------------------------
    pca = PCA(n_components=min(pca_dim, ref_feats.shape[0], ref_feats.shape[1]))

    # fit_transform 在参考域上拟合 PCA，并输出降维结果
    # normalize 再对每个样本做归一化（通常 L2）
    ref_norm = normalize(pca.fit_transform(ref_feats))

    # -------------------------------------------------------------------------
    # 2.6 参考域 KMeans 聚类
    # 使用固定的 reference_k
    # -------------------------------------------------------------------------
    kmeans = KMeans(n_clusters=reference_k, random_state=42).fit(ref_norm)

    # -------------------------------------------------------------------------
    # 2.7 打包参考域结果字典
    # 这个字典后续会被多个模块复用
    # -------------------------------------------------------------------------
    ref_results = {
        'features': ref_norm,                    # PCA + normalize 后的参考特征
        'labels': kmeans.labels_,               # 每个参考细胞所属簇标签
        'source_labels': ref_sources,           # 每个参考细胞的来源类别
        'image_paths': ref_paths,               # 图像路径
        'cluster_centers': kmeans.cluster_centers_,  # 参考簇中心
        'n_clusters': reference_k,              # 参考簇数量
        'pca': pca                              # 已拟合好的 PCA 模型
    }

    # =========================================================================
    # Step 3. 参考域可视化
    # =========================================================================
    ref_vis_dir = os.path.join(base_save_dir, f'reference_visualization_pca{pca_dim}')
    os.makedirs(ref_vis_dir, exist_ok=True)

    # -------------------------------------------------------------------------
    # 3.1 画参考域全局 UMAP（不带图像）
    # 并把训练好的 UMAP 模型保存下来，后续 candidate 也可以映射到同一个空间
    # -------------------------------------------------------------------------
    reference_umap = visualize_all_clusters_without_images(
        ref_results['features'],
        ref_results['labels'],
        ref_results['source_labels'],
        "Reference",
        ref_vis_dir
    )

    # -------------------------------------------------------------------------
    # 3.2 画参考域每个簇的图像版可视化
    # 每个细胞会直接贴到图上
    # -------------------------------------------------------------------------
    improved_visualize_clusters_with_images(
        ref_results['features'],
        ref_results['labels'],
        ref_results['source_labels'],
        ref_results['image_paths'],
        "Reference",
        ref_vis_dir,
        ref_sizes
    )

    # -------------------------------------------------------------------------
    # 3.3 统计参考域每个簇的来源构成
    # 输出柱状图 + CSV
    # -------------------------------------------------------------------------
    analyze_cluster_composition(
        ref_results['labels'],
        ref_results['source_labels'],
        ref_results['image_paths'],
        "Reference",
        ref_vis_dir
    )

    # =========================================================================
    # Step 4. 准备病理片子列表
    # =========================================================================
    # 把阳性和阴性病例目录都收集起来，每项记录：
    #   (folder_path, status)
    # 其中 status ∈ {'positive', 'negative'}
    # -------------------------------------------------------------------------
    patient_folders = []
    patient_folders.extend([
        (entry.path, 'positive')
        for entry in os.scandir(positive_folder_path)
        if entry.is_dir()
    ])
    patient_folders.extend([
        (entry.path, 'negative')
        for entry in os.scandir(negative_folder_path)
        if entry.is_dir()
    ])

    # 最终用于汇总所有病人的统计表
    results_stats = []

    # =========================================================================
    # Step 5. 循环处理每个病理片
    # =========================================================================
    for folder, status in patient_folders:

        # ---------------------------------------------------------------------
        # 5.1 提取病人 / 切片 ID，并创建专属输出目录
        # ---------------------------------------------------------------------
        pid = extract_patient_id(folder)
        save_dir = os.path.join(base_save_dir, pid)
        os.makedirs(save_dir, exist_ok=True)

        # ---------------------------------------------------------------------
        # 5.2 收集该病理片的候选细胞路径
        #
        # 这里写的是：
        #   collect_image_paths(os.path.join(folder, "malignant"))
        #
        # 说明当前项目约定：
        #   待分析病理片中，候选域细胞图片被放在 malignant 子目录里
        #
        # 如果没有候选图像，则直接跳过此病人
        # ---------------------------------------------------------------------
        cand_paths = collect_image_paths(os.path.join(folder, "malignant"))
        if not cand_paths:
            continue

        # ---------------------------------------------------------------------
        # 5.3 候选域 source label 全部设为 0
        # 因为候选域本身是“未知待分析样本”，统一标记为 Candidate
        # ---------------------------------------------------------------------
        cand_sources = [0] * len(cand_paths)

        # ---------------------------------------------------------------------
        # 5.4 计算候选域细胞大小
        # ---------------------------------------------------------------------
        cand_sizes = np.log1p([count_non_background_pixels(p) for p in cand_paths])

        # ---------------------------------------------------------------------
        # 5.5 构建候选域 DataLoader 并提取特征
        # ---------------------------------------------------------------------
        cand_loader = DataLoader(
            CellImageDataset(cand_paths, transform=transform, source_labels=cand_sources),
            batch_size=batch_size,
            shuffle=False
        )

        cand_feats, _, cand_paths = extract_features_with_cell_size(
            model,
            cand_loader,
            device,
            cand_sizes,
            feature_layer,
            cell_size_weight
        )

        # ---------------------------------------------------------------------
        # 5.6 将候选域特征投影到“参考域拟合好的 PCA 空间”中
        # 这是跨域比较的重要步骤：
        #   reference 和 candidate 必须在同一个特征子空间里
        # ---------------------------------------------------------------------
        cand_norm = normalize(pca.transform(cand_feats))

        # ---------------------------------------------------------------------
        # 5.7 初始化 Rule1 / Rule2 命中计数
        # ---------------------------------------------------------------------
        matched_count_r1 = 0
        matched_count_r2 = 0

        # ---------------------------------------------------------------------
        # 5.8 为不同规则创建输出目录
        #   rule1_matched_cells：规则1匹配结果
        #   rule2_matched_cells：规则2匹配结果
        #   matched_malignant_cells：总汇总目录
        # ---------------------------------------------------------------------
        dirs = {
            'rule1': os.path.join(save_dir, 'rule1_matched_cells'),
            'rule2': os.path.join(save_dir, 'rule2_matched_cells'),
            'total': os.path.join(save_dir, 'matched_malignant_cells')
        }
        for d in dirs.values():
            os.makedirs(d, exist_ok=True)

        # =========================================================================
        # Step 6. 根据候选样本数量，决定走哪种分析模式
        # =========================================================================
        # 逻辑：
        #   - 候选细胞数 >= 32：走聚类模式（cluster mode）
        #   - 候选细胞数 <  32：走单细胞模式（single-cell mode）
        #
        # 为什么要这么分？
        #   因为样本太少时聚类不稳定，强行聚类的意义不大；
        #   直接对单细胞应用规则更合适。
        # -------------------------------------------------------------------------
        if len(cand_norm) >= 32:

            # =====================================================================
            # A. 聚类模式
            # =====================================================================

            # ---------------------------------------------------------------------
            # 6A.1 自动优化候选域 K，并做聚类
            # ---------------------------------------------------------------------
            cand_results, best_k = optimize_k_for_candidate_clustering(
                cand_norm,
                cand_sources,
                cand_paths,
                ref_results,
                save_dir,
                pid,
                max_candidate_k
            )

            # 若优化失败，跳过当前病理片
            if not cand_results:
                continue

            # ---------------------------------------------------------------------
            # cand_results 的结构大致为：
            # {
            #   'features':       候选特征矩阵
            #   'labels':         每个候选细胞的聚类标签
            #   'distances':      到各自簇中心的距离
            #   'source_labels':  来源标签（这里一般全是 Candidate）
            #   'image_paths':    图像路径
            #   'kmeans':         KMeans 对象
            #   'cluster_centers':候选簇中心
            #   'n_clusters':     最优 K
            # }
            # ---------------------------------------------------------------------

            # ---------------------------------------------------------------------
            # 6A.2 保存 cluster_detail.csv
            # 用于记录每张候选细胞图片被分到了哪个 cluster
            # ---------------------------------------------------------------------
            csv_data = {
                'labels': cand_results['labels'],
                'image_paths': cand_results['image_paths']
            }

            df_cluster = pd.DataFrame(csv_data)
            csv_save_path = os.path.join(save_dir, 'cluster_detail.csv')

            # index=False：不写行号
            # utf-8-sig：兼容 Excel 打开中文路径时不乱码
            df_cluster.to_csv(csv_save_path, index=False, encoding='utf-8-sig')

            # ---------------------------------------------------------------------
            # 6A.3 基于候选簇中心与参考簇中心的相似度，识别 Rule1 / Rule2 匹配簇
            #
            # 返回：
            #   r1_clusters      : 满足规则1的候选簇编号列表
            #   r2_clusters      : 满足规则2的候选簇编号列表
            #   similarity_matrix: 候选簇中心 vs 参考簇中心 的相似度矩阵
            # ---------------------------------------------------------------------
            r1_clusters, r2_clusters, similarity_matrix = identify_matching_clusters(
                cand_results['cluster_centers'],
                ref_results['cluster_centers']
            )

            # ---------------------------------------------------------------------
            # 6A.4 计算细胞级别相似度
            # 用于后续保存单细胞时做二次判断
            # ---------------------------------------------------------------------
            cell_sims = cosine_similarity(
                cand_results['features'],
                ref_results['cluster_centers']
            )

            # ---------------------------------------------------------------------
            # 6A.5 保存 Rule1 命中的细胞
            # 先找出：标签属于 r1_clusters 的所有细胞索引
            # 再调用 save_matched_cells 复制图像到输出目录
            # ---------------------------------------------------------------------
            indices_r1 = [
                i for i, l in enumerate(cand_results['labels'])
                if l in r1_clusters
            ]

            c1, _ = save_matched_cells(
                indices_r1,
                cand_paths,
                cand_results['labels'],
                cell_sims,
                'rule1',
                {'rule_dir': dirs['rule1'], 'total_dir': dirs['total']},
                [0, 5]
            )

            # ---------------------------------------------------------------------
            # 6A.6 保存 Rule2 命中的细胞
            # ---------------------------------------------------------------------
            indices_r2 = [
                i for i, l in enumerate(cand_results['labels'])
                if l in r2_clusters
            ]

            c2, _ = save_matched_cells(
                indices_r2,
                cand_paths,
                cand_results['labels'],
                cell_sims,
                'rule2',
                {'rule_dir': dirs['rule2'], 'total_dir': dirs['total']},
                [0, 5]
            )

            matched_count_r1, matched_count_r2 = c1, c2

            # ---------------------------------------------------------------------
            # 6A.7 构建 matching_pairs
            #
            # 当前项目约定：
            #   - Rule1 对应匹配到 R5
            #   - Rule2 对应匹配到 R0
            #
            # 所以这里人工构造：
            #   (5, cand_idx, similarity_matrix[cand_idx, 5])
            #   (0, cand_idx, similarity_matrix[cand_idx, 0])
            # ---------------------------------------------------------------------
            matching_pairs = []

            # Rule1 匹配的候选簇视为匹配 R5
            for cand_idx in r1_clusters:
                matching_pairs.append((5, cand_idx, similarity_matrix[cand_idx, 5]))

            # Rule2 匹配的候选簇视为匹配 R0
            for cand_idx in r2_clusters:
                matching_pairs.append((0, cand_idx, similarity_matrix[cand_idx, 0]))

            # ---------------------------------------------------------------------
            # 6A.8 计算双向共识得分
            # ---------------------------------------------------------------------
            calculate_consensus_score(
                ref_results,
                cand_results,
                matching_pairs,
                save_dir
            )

            # ---------------------------------------------------------------------
            # 6A.9 画 reference / candidate 的匹配可视化
            # 这里使用 reference_umap，使 candidate 可以映射到同一 UMAP 空间
            # ---------------------------------------------------------------------
            visualize_matching_clusters(
                ref_results,
                cand_results,
                matching_pairs,
                save_dir,
                reference_umap,
                r1_clusters,
                r2_clusters
            )

            # ---------------------------------------------------------------------
            # 6A.10 画候选域全局分布（不带图像）
            # ---------------------------------------------------------------------
            visualize_all_clusters_without_images(
                cand_results['features'],
                cand_results['labels'],
                cand_results['source_labels'],
                pid,
                save_dir,
                reference_umap
            )

            # ---------------------------------------------------------------------
            # 6A.11 画候选域每个簇的图像版可视化
            # ---------------------------------------------------------------------
            improved_visualize_clusters_with_images(
                cand_results['features'],
                cand_results['labels'],
                cand_results['source_labels'],
                cand_paths,
                pid,
                save_dir,
                cand_sizes
            )

            # ---------------------------------------------------------------------
            # 6A.12 可视化“筛选后仍保留下来的匹配细胞”
            # ---------------------------------------------------------------------
            visualize_filtered_clusters(
                ref_results,
                cand_results,
                matching_pairs,
                save_dir,
                r1_clusters=r1_clusters,
                r2_clusters=r2_clusters
            )

            # ---------------------------------------------------------------------
            # 6A.13 绘制簇级别相似度热图
            #
            # similarity_matrix 原始形状通常是：
            #   [n_candidate_clusters, n_reference_clusters]
            #
            # 这里转置后：
            #   [n_reference_clusters, n_candidate_clusters]
            # 更符合“行=Reference, 列=Candidate”的阅读习惯
            # ---------------------------------------------------------------------
            xlabel = [f"C{i}" for i in range(cand_results['n_clusters'])]
            ylabel = [f"R{i}" for i in range(ref_results['n_clusters'])]

            plot_similarity_heatmap(
                similarity_matrix.T,
                xlabel,
                ylabel,
                f"{pid} Similarity Matrix",
                os.path.join(save_dir, 'similarity_heatmap.png')
            )

        else:
            # =====================================================================
            # B. 单细胞模式
            # =====================================================================
            # 候选细胞太少时，不再先聚类，而是直接按单细胞与参考簇中心的相似度来判断
            # ---------------------------------------------------------------------

            # ---------------------------------------------------------------------
            # 6B.1 计算每个候选细胞 vs 所有参考簇中心的相似度
            # 输出 shape: [N_cells, K_ref]
            # ---------------------------------------------------------------------
            cell_sims = cosine_similarity(cand_norm, ref_results['cluster_centers'])

            # ---------------------------------------------------------------------
            # 6B.2 Rule1：筛选满足规则1的单细胞
            #
            # check_matching_rules() 的参数含义：
            #   target = 5     -> 目标参考簇 R5
            #   avoid  = 0     -> 忽略 R0
            #   threshold_target = 0.5
            #   threshold_ratio = 3
            # ---------------------------------------------------------------------
            indices_r1 = [
                i for i, s in enumerate(cell_sims)
                if check_matching_rules(s, 5, 0, 0.5, threshold_ratio=3)
            ]

            c1, processed_r1 = save_matched_cells(
                indices_r1,
                cand_paths,
                [0] * len(cand_paths),   # 单细胞模式没有真实聚类标签，因此统一用 0 占位
                cell_sims,
                'rule1',
                {'rule_dir': dirs['rule1'], 'total_dir': dirs['total']},
                [0, 5]
            )

            # ---------------------------------------------------------------------
            # 6B.3 Rule2：筛选满足规则2的单细胞
            # 并显式排除已经被 Rule1 处理过的细胞，避免重复保存
            # ---------------------------------------------------------------------
            indices_r2 = [
                i for i, s in enumerate(cell_sims)
                if i not in processed_r1
                and check_matching_rules(s, 0, 5, 0.4, threshold_avoid_others=0.2)
            ]

            c2, _ = save_matched_cells(
                indices_r2,
                cand_paths,
                [0] * len(cand_paths),
                cell_sims,
                'rule2',
                {'rule_dir': dirs['rule2'], 'total_dir': dirs['total']},
                [0, 5]
            )

            matched_count_r1, matched_count_r2 = c1, c2

            # ---------------------------------------------------------------------
            # 6B.4 单细胞相似度热图
            # 行 = 参考簇，列 = 细胞
            # ---------------------------------------------------------------------
            xlabel = [f"Cell{i}" for i in range(len(cell_sims))]
            ylabel = [f"R{i}" for i in range(ref_results['n_clusters'])]

            plot_similarity_heatmap(
                cell_sims.T,
                xlabel,
                ylabel,
                f"Patient {pid} Cell Similarity Matrix",
                os.path.join(save_dir, 'Cell_similarity_heatmap.png')
            )

        # =========================================================================
        # Step 7. 记录当前病理片的统计结果
        # =========================================================================
        results_stats.append({
            'Patient_ID': pid,
            'Status': status,
            'Rule1': matched_count_r1,
            'Rule2': matched_count_r2,
            'Total': matched_count_r1 + matched_count_r2
        })

        print(f"Patient {pid} Done. Matched: {matched_count_r1 + matched_count_r2}")

    # =========================================================================
    # Step 8. 保存所有病理片的汇总统计
    # =========================================================================
    # 最终输出 stats.xlsx，其中每行对应一个病人/病理片
    pd.DataFrame(results_stats).to_excel(
        os.path.join(base_save_dir, 'stats.xlsx')
    )